# NOTEBOOK 03 — XÂY DỰNG TẬP DỮ LIỆU CẤP TỔN THƯƠNG

## 1. Mục tiêu

Notebook này thực hiện xây dựng tập dữ liệu ở cấp tổn thương
từ bộ dữ liệu CBIS-DDSM.

Các công việc chính:

1. Đọc và chuẩn hóa bốn bảng metadata Mass/Calc Train/Test.
2. Kiểm tra và xác định danh tính từng tổn thương.
3. Xây dựng bảng annotation chuẩn hóa.
4. Liên kết tổn thương với các hình ảnh DICOM.
5. Liên kết tổn thương với ROI mask và bounding box.
6. Kiểm định tính toàn vẹn và khả năng truy xuất nguồn gốc.

## 2. Kiến trúc xử lý

- Tầng 1: Dữ liệu metadata nguồn.
- Tầng 2: Bảng tổn thương trung tâm và các bảng liên kết hình ảnh, ROI.
- Tầng 3: Tập dữ liệu cấp tổn thương đã được kiểm định.

## 3. Nguyên tắc thực hiện

- Giữ nguyên dữ liệu gốc.
- Bảo toàn nhãn Train/Test ban đầu.
- Không mặc định một dòng annotation là một tổn thương.
- Không tự động chọn DICOM candidate khi có nhiều lựa chọn.
- Không tự động resize mask hoặc suy diễn tọa độ.
- Lưu lại các trường hợp chưa xác định để kiểm tra.
- Không huấn luyện mô hình trong notebook này.

In [64]:
# Import thư viện và cấu hình

from pathlib import Path
import pandas as pd
import numpy as np

# 1. CẤU HÌNH ĐƯỜNG DẪN DỰ ÁN

PROJECT_ROOT = Path(r"C:\Breast-Cancer-CBIS-DDSM")

DATA_DIR = PROJECT_ROOT / "Data"
METADATA_DIR = DATA_DIR / "metadata"
PROCESSED_DIR = DATA_DIR / "processed"

OUTPUT_DIR = PROCESSED_DIR

# 2. BỐN BẢNG METADATA NGUỒN

ANNOTATION_FILES = {
    "calc_train": METADATA_DIR / "calc_case_description_train_set.csv",
    "calc_test": METADATA_DIR / "calc_case_description_test_set.csv",
    "mass_train": METADATA_DIR / "mass_case_description_train_set.csv",
    "mass_test": METADATA_DIR / "mass_case_description_test_set.csv",
}

# 3. CÁC KẾT QUẢ TỪ NOTEBOOK 02

NB02_FILES = {
    "dicom_index": PROCESSED_DIR / "dicom_index.csv",
    "csv_dicom_linkage": PROCESSED_DIR / "nb02_csv_dicom_linkage.csv",
    "instance_role_final": PROCESSED_DIR / "nb02_instance_role_final.csv",
    "roi_mask_validation": PROCESSED_DIR / "nb02_roi_mask_validation.csv",
    "patient_split_overlap": PROCESSED_DIR / "nb02_patient_split_overlap.csv",
}

# 4. CẤU HÌNH CHUNG

RANDOM_STATE = 42

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 160)

print("Đã khởi tạo cấu hình Notebook 03.")
print(f"Thư mục dự án: {PROJECT_ROOT}")

Đã khởi tạo cấu hình Notebook 03.
Thư mục dự án: C:\Breast-Cancer-CBIS-DDSM


### PHẦN 01. KIỂM TRA DỮ LIỆU ĐẦU VÀO

### Dữ liệu kiểm tra
- Bốn bảng metadata Mass/Calc Train/Test.
- Chỉ mục DICOM đã xây dựng trong Notebook 02.
- Bảng liên kết CSV với DICOM.
- Bảng phân loại vai trò DICOM cuối cùng.
- Bảng kiểm định ROI mask.
- Bảng kiểm tra giao nhau bệnh nhân giữa các tập dữ liệu.

In [65]:
# KIỂM TRA CÁC FILE ĐẦU VÀO

input_files = {
    **ANNOTATION_FILES,
    **NB02_FILES
}

validation_records = []

for file_name, file_path in input_files.items():

    file_exists = file_path.is_file()

    validation_records.append({
        "ten_file": file_name,
        "duong_dan": str(file_path),
        "ton_tai": file_exists,
        "dung_luong_MB": (
            round(file_path.stat().st_size / (1024 ** 2), 3)
            if file_exists else np.nan
        )
    })

input_validation_df = pd.DataFrame(validation_records)

print("=" * 70)
print("KẾT QUẢ KIỂM TRA FILE ĐẦU VÀO")
print("=" * 70)

display(input_validation_df)

missing_inputs = input_validation_df.loc[
    ~input_validation_df["ton_tai"],
    "ten_file"
].tolist()

print("\nTỔNG KẾT")
print(f"Tổng số file cần kiểm tra: {len(input_files)}")
print(f"Số file tồn tại: {input_validation_df['ton_tai'].sum()}")
print(f"Số file còn thiếu: {len(missing_inputs)}")

if missing_inputs:
    print("\nCác file chưa tìm thấy:")

    for file_name in missing_inputs:
        print(f"- {file_name}")

else:
    print("\nTất cả file đầu vào đều tồn tại.")

KẾT QUẢ KIỂM TRA FILE ĐẦU VÀO


,ten_file,duong_dan,ton_tai,dung_luong_MB
0,calc_train,C:\Breast-Cancer-CBIS-DDSM\Data\metadata\calc_case_description_train_set.csv,True,0.882
1,calc_test,C:\Breast-Cancer-CBIS-DDSM\Data\metadata\calc_case_description_test_set.csv,True,0.182
2,mass_train,C:\Breast-Cancer-CBIS-DDSM\Data\metadata\mass_case_description_train_set.csv,True,0.737
3,mass_test,C:\Breast-Cancer-CBIS-DDSM\Data\metadata\mass_case_description_test_set.csv,True,0.207
4,dicom_index,C:\Breast-Cancer-CBIS-DDSM\Data\processed\dicom_index.csv,True,5.084
5,csv_dicom_linkage,C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb02_csv_dicom_linkage.csv,True,7.798
6,instance_role_final,C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb02_instance_role_final.csv,True,5.695
7,roi_mask_validation,C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb02_roi_mask_validation.csv,True,2.250
8,patient_split_overlap,C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb02_patient_split_overlap.csv,True,0.001



TỔNG KẾT
Tổng số file cần kiểm tra: 9
Số file tồn tại: 9
Số file còn thiếu: 0

Tất cả file đầu vào đều tồn tại.


In [66]:
# KIỂM TRA CẤU TRÚC THƯ MỤC DỰ ÁN

project_directories = {
    "Thư mục dự án": PROJECT_ROOT,
    "Thư mục dữ liệu": DATA_DIR,
    "Thư mục metadata": METADATA_DIR,
    "Thư mục dữ liệu gốc": DATA_DIR / "rawdata",
    "Thư mục dữ liệu đã xử lý": PROCESSED_DIR,
}

directory_records = []

for directory_name, directory_path in project_directories.items():

    directory_records.append({
        "ten_thu_muc": directory_name,
        "duong_dan": str(directory_path),
        "ton_tai": directory_path.is_dir()
    })

directory_check_df = pd.DataFrame(directory_records)

print("=" * 70)
print("KIỂM TRA CẤU TRÚC THƯ MỤC")
print("=" * 70)

display(directory_check_df)

KIỂM TRA CẤU TRÚC THƯ MỤC


,ten_thu_muc,duong_dan,ton_tai
0,Thư mục dự án,C:\Breast-Cancer-CBIS-DDSM,True
1,Thư mục dữ liệu,C:\Breast-Cancer-CBIS-DDSM\Data,True
2,Thư mục metadata,C:\Breast-Cancer-CBIS-DDSM\Data\metadata,True
3,Thư mục dữ liệu gốc,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata,True
4,Thư mục dữ liệu đã xử lý,C:\Breast-Cancer-CBIS-DDSM\Data\processed,True


In [67]:
# KIỂM TRA ĐIỀU KIỆN TIẾP TỤC NOTEBOOK 03

required_files = (
    list(ANNOTATION_FILES.keys())
    + [
        "dicom_index",
        "csv_dicom_linkage",
        "instance_role_final",
        "roi_mask_validation"
    ]
)

required_missing = input_validation_df.loc[
    input_validation_df["ten_file"].isin(required_files)
    & ~input_validation_df["ton_tai"],
    "ten_file"
].tolist()

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục dự án: {PROJECT_ROOT}"
    )

if not METADATA_DIR.is_dir():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục metadata: {METADATA_DIR}"
    )

if not PROCESSED_DIR.is_dir():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục processed: {PROCESSED_DIR}"
    )

if required_missing:

    raise FileNotFoundError(
        "Không thể tiếp tục vì còn thiếu các file đầu vào:\n"
        + "\n".join(required_missing)
    )

print("=" * 70)
print("KIỂM TRA ĐẦU VÀO HOÀN TẤT")
print("=" * 70)

print("Các file đầu vào bắt buộc đã tồn tại.")
print("Cấu trúc thư mục dự án hợp lệ.")
print("Có thể chuyển sang Phần 02 - Đọc và chuẩn hóa dữ liệu nguồn.")

KIỂM TRA ĐẦU VÀO HOÀN TẤT
Các file đầu vào bắt buộc đã tồn tại.
Cấu trúc thư mục dự án hợp lệ.
Có thể chuyển sang Phần 02 - Đọc và chuẩn hóa dữ liệu nguồn.


### PHẦN 02 — ĐỌC VÀ CHUẨN HÓA DỮ LIỆU NGUỒN


## Các nội dung thực hiện

1. Đọc dữ liệu từ bốn file CSV.
2. Kiểm tra số dòng, số cột và tên cột.
3. Đối chiếu sự khác biệt về schema giữa bốn bảng.
4. Chuẩn hóa tên cột theo một cấu trúc thống nhất.
5. Bổ sung thông tin loại tổn thương, tập dữ liệu và nguồn gốc bản ghi.
6. Kiểm tra dữ liệu thiếu và các giá trị của trường pathology.

In [68]:
# 2.1. ĐỌC BỐN BẢNG METADATA NGUỒN

source_tables = {}

for source_name, file_path in ANNOTATION_FILES.items():

    if not file_path.is_file():
        raise FileNotFoundError(
            f"Không tìm thấy file dữ liệu: {file_path}"
        )

    source_tables[source_name] = pd.read_csv(
        file_path,
        low_memory=False
    )

    print(
        f"[ĐÃ ĐỌC] {source_name}: "
        f"{source_tables[source_name].shape[0]:,} dòng, "
        f"{source_tables[source_name].shape[1]} cột"
    )

print("\nĐã đọc hoàn tất bốn bảng metadata.")

[ĐÃ ĐỌC] calc_train: 1,546 dòng, 14 cột
[ĐÃ ĐỌC] calc_test: 326 dòng, 14 cột
[ĐÃ ĐỌC] mass_train: 1,318 dòng, 14 cột
[ĐÃ ĐỌC] mass_test: 378 dòng, 14 cột

Đã đọc hoàn tất bốn bảng metadata.


In [69]:
# 2.2. THỐNG KÊ CẤU TRÚC CÁC BẢNG NGUỒN

schema_records = []

for source_name, df in source_tables.items():

    for column_position, column_name in enumerate(df.columns):

        schema_records.append({
            "bang_nguon": source_name,
            "vi_tri_cot": column_position,
            "ten_cot_goc": column_name,
            "kieu_du_lieu": str(df[column_name].dtype),
            "so_gia_tri_thieu": int(df[column_name].isna().sum()),
            "so_gia_tri_khong_thieu": int(df[column_name].notna().sum())
        })

schema_overview_df = pd.DataFrame(schema_records)

print("=" * 80)
print("TỔNG QUAN CẤU TRÚC CÁC BẢNG METADATA")
print("=" * 80)

table_summary_df = pd.DataFrame([
    {
        "bang_nguon": source_name,
        "so_dong": df.shape[0],
        "so_cot": df.shape[1]
    }
    for source_name, df in source_tables.items()
])

display(table_summary_df)

print("\nCHI TIẾT TÊN CỘT VÀ KIỂU DỮ LIỆU")
display(schema_overview_df)

TỔNG QUAN CẤU TRÚC CÁC BẢNG METADATA


,bang_nguon,so_dong,so_cot
0,calc_train,1546,14
1,calc_test,326,14
2,mass_train,1318,14
3,mass_test,378,14



CHI TIẾT TÊN CỘT VÀ KIỂU DỮ LIỆU


,bang_nguon,vi_tri_cot,ten_cot_goc,kieu_du_lieu,so_gia_tri_thieu,so_gia_tri_khong_thieu
0,calc_train,0,patient_id,str,0,1546
1,calc_train,1,breast density,int64,0,1546
2,calc_train,2,left or right breast,str,0,1546
3,calc_train,3,image view,str,0,1546
4,calc_train,4,abnormality id,int64,0,1546
5,calc_train,5,abnormality type,str,0,1546
6,calc_train,6,calc type,str,20,1526
7,calc_train,7,calc distribution,str,376,1170
8,calc_train,8,assessment,int64,0,1546
9,calc_train,9,pathology,str,0,1546


In [70]:
# 2.3. SO SÁNH SCHEMA GIỮA BỐN BẢNG

schema_by_table = {
    source_name: set(df.columns)
    for source_name, df in source_tables.items()
}

all_source_columns = sorted(
    set().union(*schema_by_table.values())
)

schema_comparison_records = []

for column_name in all_source_columns:

    record = {
        "ten_cot_goc": column_name
    }

    for source_name, columns in schema_by_table.items():
        record[source_name] = column_name in columns

    record["so_bang_co_cot"] = sum(
        column_name in columns
        for columns in schema_by_table.values()
    )

    schema_comparison_records.append(record)

schema_comparison_df = pd.DataFrame(
    schema_comparison_records
)

print("=" * 80)
print("ĐỐI CHIẾU SCHEMA GIỮA BỐN BẢNG")
print("=" * 80)

display(schema_comparison_df)

print("\nSỐ LƯỢNG CỘT KHÁC NHAU:")
print(f"Tổng số tên cột phân biệt: {len(all_source_columns)}")
print(
    "Số cột xuất hiện trong cả bốn bảng:",
    int((schema_comparison_df["so_bang_co_cot"] == 4).sum())
)
print(
    "Số cột chỉ xuất hiện ở một đến ba bảng:",
    int((schema_comparison_df["so_bang_co_cot"] < 4).sum())
)

ĐỐI CHIẾU SCHEMA GIỮA BỐN BẢNG


,ten_cot_goc,calc_train,calc_test,mass_train,mass_test,so_bang_co_cot
0,ROI mask file path,True,True,True,True,4
1,abnormality id,True,True,True,True,4
2,abnormality type,True,True,True,True,4
3,assessment,True,True,True,True,4
4,breast density,True,True,False,False,2
5,breast_density,False,False,True,True,2
6,calc distribution,True,True,False,False,2
7,calc type,True,True,False,False,2
8,cropped image file path,True,True,True,True,4
9,image file path,True,True,True,True,4



SỐ LƯỢNG CỘT KHÁC NHAU:
Tổng số tên cột phân biệt: 17
Số cột xuất hiện trong cả bốn bảng: 11
Số cột chỉ xuất hiện ở một đến ba bảng: 6


In [71]:
# 2.4. XÂY DỰNG QUY TẮC CHUẨN HÓA TÊN CỘT

import re
import unicodedata


def normalize_column_key(column_name):
    """
    Chuẩn hóa tên cột để phục vụ đối chiếu.

    Không thay đổi giá trị dữ liệu bên trong cột.
    """

    text = str(column_name).strip().lower()

    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        char for char in text
        if not unicodedata.combining(char)
    )

    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text)

    return text.strip("_")


# Ánh xạ tên cột nguồn sang tên cột chuẩn
COLUMN_RENAME_MAP = {
    "patient_id": "patient_id",

    "breast_density": "breast_density",

    "left_or_right_breast": "laterality",

    "image_view": "image_view",

    "abnormality_id": "abnormality_id",

    "abnormalitytype": "abnormality_type",

    "calc_type": "calc_type",

    "calc_distribution": "calc_distribution",

    "mass_shape": "mass_shape",

    "mass_margins": "mass_margins",

    "assessment": "assessment",

    "pathology": "pathology",

    "subtlety": "subtlety",

    "image_file_path": "full_image_csv_path",

    "cropped_image_file_path": "crop_image_csv_path",

    "roi_mask_file_path": "roi_mask_csv_path",
}

print("Đã khai báo quy tắc chuẩn hóa tên cột.")
print(f"Số quy tắc ánh xạ: {len(COLUMN_RENAME_MAP)}")

Đã khai báo quy tắc chuẩn hóa tên cột.
Số quy tắc ánh xạ: 16


In [72]:
# 2.5. CHUẨN HÓA BỐN BẢNG VÀ GIỮ THÔNG TIN NGUỒN

normalized_tables = {}

for source_name, original_df in source_tables.items():

    df = original_df.copy()

    # Chuẩn hóa tên cột để đối chiếu với quy tắc ánh xạ
    normalized_keys = [
        normalize_column_key(column)
        for column in df.columns
    ]

    # Kiểm tra tên cột có bị trùng sau chuẩn hóa hay không
    if len(normalized_keys) != len(set(normalized_keys)):
        raise ValueError(
            f"Bảng {source_name} có tên cột trùng "
            "sau khi chuẩn hóa."
        )

    # Xây dựng tên cột chuẩn
    canonical_columns = [
        COLUMN_RENAME_MAP.get(key, key)
        for key in normalized_keys
    ]

    # Kiểm tra nhiều cột nguồn có cùng ánh xạ đến một cột chuẩn
    if len(canonical_columns) != len(set(canonical_columns)):
        raise ValueError(
            f"Bảng {source_name} có nhiều cột "
            "cùng ánh xạ đến một tên cột chuẩn."
        )

    # Giữ lại toàn bộ cột gốc dưới tiền tố raw__
    raw_df = original_df.copy()

    raw_df.columns = [
        f"raw__{column}"
        for column in original_df.columns
    ]

    # Tạo bản chuẩn hóa
    canonical_df = original_df.copy()
    canonical_df.columns = canonical_columns

    # Ghép dữ liệu gốc và dữ liệu chuẩn hóa
    normalized_df = pd.concat(
        [
            canonical_df.reset_index(drop=True),
            raw_df.reset_index(drop=True)
        ],
        axis=1
    )

    # Bổ sung thông tin nguồn
    normalized_df["source_split"] = (
        "train" if source_name.endswith("train") else "test"
    )

    normalized_df["lesion_type"] = (
        "calcification"
        if source_name.startswith("calc")
        else "mass"
    )

    normalized_df["source_table"] = source_name

    normalized_df["source_csv"] = (
        ANNOTATION_FILES[source_name].name
    )

    normalized_df["source_row_index"] = np.arange(
        len(normalized_df)
    )

    normalized_df["source_row_number"] = (
        normalized_df["source_row_index"] + 1
    )

    normalized_tables[source_name] = normalized_df

    print(
        f"[HOÀN TẤT] {source_name}: "
        f"{len(normalized_df):,} dòng, "
        f"{len(normalized_df.columns)} cột sau chuẩn hóa"
    )

print("\nĐã chuẩn hóa xong bốn bảng.")
print("Dữ liệu gốc được giữ lại dưới các cột có tiền tố raw__.")

[HOÀN TẤT] calc_train: 1,546 dòng, 34 cột sau chuẩn hóa
[HOÀN TẤT] calc_test: 326 dòng, 34 cột sau chuẩn hóa
[HOÀN TẤT] mass_train: 1,318 dòng, 34 cột sau chuẩn hóa
[HOÀN TẤT] mass_test: 378 dòng, 34 cột sau chuẩn hóa

Đã chuẩn hóa xong bốn bảng.
Dữ liệu gốc được giữ lại dưới các cột có tiền tố raw__.


In [73]:
# 2.6. TẠO BẢNG ANNOTATION CHUẨN HÓA

annotation_master_df = pd.concat(
    [
        normalized_tables["calc_train"],
        normalized_tables["calc_test"],
        normalized_tables["mass_train"],
        normalized_tables["mass_test"]
    ],
    ignore_index=True,
    sort=False
)

# Kiểm tra tổng số dòng sau khi gộp
expected_annotation_count = sum(
    len(df)
    for df in source_tables.values()
)

actual_annotation_count = len(annotation_master_df)

print("=" * 80)
print("KẾT QUẢ TẠO BẢNG ANNOTATION CHUẨN HÓA")
print("=" * 80)

print(
    f"Tổng số dòng từ bốn bảng nguồn: "
    f"{expected_annotation_count:,}"
)

print(
    f"Tổng số dòng sau khi gộp: "
    f"{actual_annotation_count:,}"
)

print(
    f"Số cột sau chuẩn hóa: "
    f"{annotation_master_df.shape[1]}"
)

if actual_annotation_count != expected_annotation_count:
    raise ValueError(
        "Số dòng sau khi gộp không khớp với tổng số dòng nguồn."
    )

print("\n[ĐẠT] Số dòng được bảo toàn sau khi gộp.")

display(
    annotation_master_df[
        [
            "patient_id",
            "laterality",
            "image_view",
            "abnormality_id",
            "abnormality_type",
            "lesion_type",
            "pathology",
            "source_split",
            "source_table",
            "source_csv",
            "source_row_index"
        ]
    ].head(10)
)

KẾT QUẢ TẠO BẢNG ANNOTATION CHUẨN HÓA
Tổng số dòng từ bốn bảng nguồn: 3,568
Tổng số dòng sau khi gộp: 3,568
Số cột sau chuẩn hóa: 39

[ĐẠT] Số dòng được bảo toàn sau khi gộp.


,patient_id,laterality,image_view,abnormality_id,abnormality_type,lesion_type,pathology,source_split,source_table,source_csv,source_row_index
0,P_00005,RIGHT,CC,1,calcification,calcification,MALIGNANT,train,calc_train,calc_case_description_train_set.csv,0
1,P_00005,RIGHT,MLO,1,calcification,calcification,MALIGNANT,train,calc_train,calc_case_description_train_set.csv,1
2,P_00007,LEFT,CC,1,calcification,calcification,BENIGN,train,calc_train,calc_case_description_train_set.csv,2
3,P_00007,LEFT,MLO,1,calcification,calcification,BENIGN,train,calc_train,calc_case_description_train_set.csv,3
4,P_00008,LEFT,CC,1,calcification,calcification,BENIGN_WITHOUT_CALLBACK,train,calc_train,calc_case_description_train_set.csv,4
5,P_00008,LEFT,CC,2,calcification,calcification,BENIGN_WITHOUT_CALLBACK,train,calc_train,calc_case_description_train_set.csv,5
6,P_00008,LEFT,CC,3,calcification,calcification,BENIGN_WITHOUT_CALLBACK,train,calc_train,calc_case_description_train_set.csv,6
7,P_00008,LEFT,MLO,1,calcification,calcification,BENIGN_WITHOUT_CALLBACK,train,calc_train,calc_case_description_train_set.csv,7
8,P_00008,LEFT,MLO,2,calcification,calcification,BENIGN_WITHOUT_CALLBACK,train,calc_train,calc_case_description_train_set.csv,8
9,P_00008,LEFT,MLO,3,calcification,calcification,BENIGN_WITHOUT_CALLBACK,train,calc_train,calc_case_description_train_set.csv,9


In [74]:
# 2.7. THỐNG KÊ SỐ LƯỢNG ANNOTATION THEO NGUỒN

annotation_distribution_df = (
    annotation_master_df
    .groupby(
        ["lesion_type", "source_split", "source_table"],
        dropna=False
    )
    .size()
    .reset_index(name="so_luong_annotation")
)

annotation_distribution_df = annotation_distribution_df.sort_values(
    ["lesion_type", "source_split"]
).reset_index(drop=True)

print("=" * 80)
print("PHÂN BỐ ANNOTATION THEO LOẠI TỔN THƯƠNG VÀ TẬP DỮ LIỆU")
print("=" * 80)

display(annotation_distribution_df)

print("\nTỔNG SỐ ANNOTATION:")
print(
    f"{annotation_distribution_df['so_luong_annotation'].sum():,}"
)

PHÂN BỐ ANNOTATION THEO LOẠI TỔN THƯƠNG VÀ TẬP DỮ LIỆU


,lesion_type,source_split,source_table,so_luong_annotation
0,calcification,test,calc_test,326
1,calcification,train,calc_train,1546
2,mass,test,mass_test,378
3,mass,train,mass_train,1318



TỔNG SỐ ANNOTATION:
3,568


In [75]:
# 2.8. KIỂM TRA DỮ LIỆU THIẾU

important_columns = [
    "patient_id",
    "laterality",
    "image_view",
    "abnormality_id",
    "abnormality_type",
    "lesion_type",
    "pathology",
    "assessment",
    "full_image_csv_path",
    "crop_image_csv_path",
    "roi_mask_csv_path"
]

missing_records = []

for column_name in important_columns:

    if column_name not in annotation_master_df.columns:
        missing_records.append({
            "ten_cot": column_name,
            "so_gia_tri_thieu": np.nan,
            "ty_le_thieu_phan_tram": np.nan,
            "trang_thai": "Không có cột"
        })
        continue

    missing_count = int(
        annotation_master_df[column_name].isna().sum()
    )

    missing_ratio = (
        missing_count / len(annotation_master_df) * 100
        if len(annotation_master_df) > 0
        else 0
    )

    missing_records.append({
        "ten_cot": column_name,
        "so_gia_tri_thieu": missing_count,
        "ty_le_thieu_phan_tram": round(missing_ratio, 3),
        "trang_thai": (
            "Có giá trị thiếu"
            if missing_count > 0
            else "Đầy đủ"
        )
    })

missing_summary_df = pd.DataFrame(missing_records)

print("=" * 80)
print("BÁO CÁO GIÁ TRỊ THIẾU")
print("=" * 80)

display(missing_summary_df)

print("\nLưu ý: Chưa loại bỏ hoặc tự động điền bất kỳ giá trị thiếu nào.")

BÁO CÁO GIÁ TRỊ THIẾU


,ten_cot,so_gia_tri_thieu,ty_le_thieu_phan_tram,trang_thai
0,patient_id,0,0.0,Đầy đủ
1,laterality,0,0.0,Đầy đủ
2,image_view,0,0.0,Đầy đủ
3,abnormality_id,0,0.0,Đầy đủ
4,abnormality_type,0,0.0,Đầy đủ
5,lesion_type,0,0.0,Đầy đủ
6,pathology,0,0.0,Đầy đủ
7,assessment,0,0.0,Đầy đủ
8,full_image_csv_path,0,0.0,Đầy đủ
9,crop_image_csv_path,0,0.0,Đầy đủ



Lưu ý: Chưa loại bỏ hoặc tự động điền bất kỳ giá trị thiếu nào.


In [76]:
# 2.9. KHẢO SÁT GIÁ TRỊ PATHOLOGY

if "pathology" not in annotation_master_df.columns:
    raise KeyError(
        "Không tìm thấy cột pathology sau khi chuẩn hóa."
    )

pathology_values_df = (
    annotation_master_df
    .groupby(
        ["lesion_type", "source_split", "pathology"],
        dropna=False
    )
    .size()
    .reset_index(name="so_luong")
    .sort_values(
        ["lesion_type", "source_split", "so_luong"],
        ascending=[True, True, False]
    )
    .reset_index(drop=True)
)

print("=" * 80)
print("PHÂN BỐ GIÁ TRỊ PATHOLOGY")
print("=" * 80)

display(pathology_values_df)

print("\nCÁC GIÁ TRỊ PATHOLOGY PHÂN BIỆT:")

for value in annotation_master_df["pathology"].drop_duplicates():
    print(repr(value))

PHÂN BỐ GIÁ TRỊ PATHOLOGY


,lesion_type,source_split,pathology,so_luong
0,calcification,test,BENIGN,130
1,calcification,test,MALIGNANT,129
2,calcification,test,BENIGN_WITHOUT_CALLBACK,67
3,calcification,train,MALIGNANT,544
4,calcification,train,BENIGN,528
5,calcification,train,BENIGN_WITHOUT_CALLBACK,474
6,mass,test,BENIGN,194
7,mass,test,MALIGNANT,147
8,mass,test,BENIGN_WITHOUT_CALLBACK,37
9,mass,train,MALIGNANT,637



CÁC GIÁ TRỊ PATHOLOGY PHÂN BIỆT:
'MALIGNANT'
'BENIGN'
'BENIGN_WITHOUT_CALLBACK'


In [77]:
# 2.10. KIỂM TRA KHẢ NĂNG TRUY XUẤT NGUỒN

provenance_columns = [
    "source_table",
    "source_csv",
    "source_row_index",
    "source_row_number",
    "source_split",
    "lesion_type"
]

# Kiểm tra sự tồn tại của các trường nguồn
missing_provenance_columns = [
    column
    for column in provenance_columns
    if column not in annotation_master_df.columns
]

if missing_provenance_columns:
    raise KeyError(
        "Thiếu các trường truy xuất nguồn:\n"
        + "\n".join(missing_provenance_columns)
    )

# Kiểm tra khóa xác định bản ghi nguồn
source_record_key = [
    "source_table",
    "source_row_index"
]

duplicate_source_records = annotation_master_df.duplicated(
    subset=source_record_key,
    keep=False
)

# Kiểm tra giá trị thiếu trong thông tin nguồn
missing_provenance = (
    annotation_master_df[provenance_columns]
    .isna()
    .sum()
)

print("=" * 80)
print("KIỂM TRA TÍNH TOÀN VẸN THÔNG TIN NGUỒN")
print("=" * 80)

print(
    "Số bản ghi có khóa nguồn trùng:",
    int(duplicate_source_records.sum())
)

print("\nSố giá trị thiếu theo trường nguồn:")
display(missing_provenance.to_frame("so_gia_tri_thieu"))

if duplicate_source_records.any():
    print("\n[CẢNH BÁO] Phát hiện khóa bản ghi nguồn bị trùng.")
else:
    print("\n[ĐẠT] Không phát hiện khóa bản ghi nguồn bị trùng.")

if missing_provenance.sum() > 0:
    print("[CẢNH BÁO] Có thông tin nguồn bị thiếu.")
else:
    print("[ĐẠT] Các trường thông tin nguồn đều đầy đủ.")

KIỂM TRA TÍNH TOÀN VẸN THÔNG TIN NGUỒN
Số bản ghi có khóa nguồn trùng: 0

Số giá trị thiếu theo trường nguồn:


,so_gia_tri_thieu
source_table,0
source_csv,0
source_row_index,0
source_row_number,0
source_split,0
lesion_type,0



[ĐẠT] Không phát hiện khóa bản ghi nguồn bị trùng.
[ĐẠT] Các trường thông tin nguồn đều đầy đủ.


### 2.11. KẾT LUẬN PHẦN 02

### Các nội dung đã kiểm tra:

- Số dòng và số cột của từng bảng nguồn.
- Sự khác biệt về tên cột giữa các bảng.
- Quy tắc chuẩn hóa tên cột.
- Khả năng bảo toàn dữ liệu và thông tin nguồn.
- Số lượng annotation theo loại tổn thương và tập dữ liệu.
- Giá trị thiếu trong các trường thông tin quan trọng.
- Các giá trị pathology thực tế.



In [78]:
# 2.12. ĐỐI CHIẾU ĐƯỜNG DẪN ROI MASK TRONG BỐN BẢNG NGUỒN

roi_path_records = []

for source_name, df in source_tables.items():

    roi_column = next(
        (
            column for column in df.columns
            if normalize_column_key(column) == "roi_mask_file_path"
        ),
        None
    )

    if roi_column is None:
        roi_path_records.append({
            "bang_nguon": source_name,
            "ten_cot_roi_goc": None,
            "tong_so_dong": len(df),
            "so_duong_dan_co_gia_tri": 0,
            "so_duong_dan_thieu": len(df),
            "so_chuoi_rong": np.nan
        })
        continue

    values = df[roi_column]

    non_null = values.notna()
    non_empty = values.astype("string").str.strip().ne("")

    roi_path_records.append({
        "bang_nguon": source_name,
        "ten_cot_roi_goc": roi_column,
        "tong_so_dong": len(df),
        "so_duong_dan_co_gia_tri": int((non_null & non_empty).sum()),
        "so_duong_dan_thieu": int((~non_null).sum()),
        "so_chuoi_rong": int((non_null & ~non_empty).sum())
    })

roi_path_audit_df = pd.DataFrame(roi_path_records)

display(roi_path_audit_df)

,bang_nguon,ten_cot_roi_goc,tong_so_dong,so_duong_dan_co_gia_tri,so_duong_dan_thieu,so_chuoi_rong
0,calc_train,ROI mask file path,1546,1546,0,0
1,calc_test,ROI mask file path,326,326,0,0
2,mass_train,ROI mask file path,1318,1318,0,0
3,mass_test,ROI mask file path,378,378,0,0


In [79]:
# 2.13. KIỂM TRA CHI TIẾT ROI PATH CỦA MASS TEST

mass_test_raw = source_tables["mass_test"].copy()

roi_source_column = next(
    column for column in mass_test_raw.columns
    if normalize_column_key(column) == "roi_mask_file_path"
)

mass_test_raw["_roi_path_missing"] = (
    mass_test_raw[roi_source_column].isna()
)

mass_test_raw["_roi_path_empty"] = (
    mass_test_raw[roi_source_column]
    .astype("string")
    .str.strip()
    .eq("")
    .fillna(False)
)

print("Tên cột ROI gốc:", roi_source_column)

print("\nPhân bố trạng thái ROI path:")
display(
    mass_test_raw[
        ["_roi_path_missing", "_roi_path_empty"]
    ].value_counts(dropna=False).reset_index(name="so_luong")
)

print("\nMột số dòng đầu tiên:")
display(
    mass_test_raw[
        [
            "patient_id",
            "left or right breast",
            "image view",
            "abnormality id",
            roi_source_column,
            "_roi_path_missing",
            "_roi_path_empty"
        ]
    ].head(20)
)

Tên cột ROI gốc: ROI mask file path

Phân bố trạng thái ROI path:


,_roi_path_missing,_roi_path_empty,so_luong
0,False,False,378



Một số dòng đầu tiên:


,patient_id,left or right breast,image view,abnormality id,ROI mask file path,_roi_path_missing,_roi_path_empty
0,P_00016,LEFT,CC,1,Mass-Test_P_00016_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.259596319110047779433501728143778409887/1.3...,False,False
1,P_00016,LEFT,MLO,1,Mass-Test_P_00016_LEFT_MLO_1/1.3.6.1.4.1.9590.100.1.2.207144238612220754118040203520552715563/1....,False,False
2,P_00017,LEFT,CC,1,Mass-Test_P_00017_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.265130777712709757209861180301164730118/1.3...,False,False
3,P_00017,LEFT,MLO,1,Mass-Test_P_00017_LEFT_MLO_1/1.3.6.1.4.1.9590.100.1.2.66862495211464129620756742774012127297/1.3...,False,False
4,P_00032,RIGHT,CC,1,Mass-Test_P_00032_RIGHT_CC_1/1.3.6.1.4.1.9590.100.1.2.362221700813915332616463215463274950690/1....,False,False
5,P_00032,RIGHT,MLO,1,Mass-Test_P_00032_RIGHT_MLO_1/1.3.6.1.4.1.9590.100.1.2.247343732911042250406750122270776949655/1...,False,False
6,P_00037,RIGHT,CC,1,Mass-Test_P_00037_RIGHT_CC_1/1.3.6.1.4.1.9590.100.1.2.51371400012281630410382188863446338107/1.3...,False,False
7,P_00037,RIGHT,MLO,1,Mass-Test_P_00037_RIGHT_MLO_1/1.3.6.1.4.1.9590.100.1.2.212527445411669219116084453571752862492/1...,False,False
8,P_00056,LEFT,MLO,1,Mass-Test_P_00056_LEFT_MLO_1/1.3.6.1.4.1.9590.100.1.2.261323583512853897235755734391630707499/1....,False,False
9,P_00066,LEFT,CC,1,Mass-Test_P_00066_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.282824026411717122435269948653880963255/1.3...,False,False


In [80]:
# 2.14. ĐỐI CHIẾU ROI PATH VỚI KẾT QUẢ NOTEBOOK 02

from pathlib import Path
import pandas as pd


# 1. Khai báo đường dẫn đến bảng liên kết của Notebook 02

NB02_LINKAGE_FILE = (
    Path(r"C:\Breast-Cancer-CBIS-DDSM")
    / "Data"
    / "processed"
    / "nb02_csv_dicom_linkage.csv"
)

# Kiểm tra file trước khi đọc
if not NB02_LINKAGE_FILE.exists():
    raise FileNotFoundError(
        f"Không tìm thấy bảng liên kết Notebook 02:\n"
        f"{NB02_LINKAGE_FILE}"
    )

print("Đường dẫn bảng liên kết:")
print(NB02_LINKAGE_FILE)

# 2. Đọc bảng liên kết

linkage_df = pd.read_csv(
    NB02_LINKAGE_FILE,
    low_memory=False
)

print("\nCác cột trong bảng liên kết Notebook 02:")
print(linkage_df.columns.tolist())

print(f"\nTổng số dòng: {len(linkage_df):,}")

# 3. Kiểm tra các cột cần thiết

required_columns = [
    "split",
    "path_type",
]

missing_columns = [
    col for col in required_columns
    if col not in linkage_df.columns
]

if missing_columns:
    raise ValueError(
        f"Bảng liên kết thiếu các cột cần thiết: {missing_columns}"
    )

# 4. Thống kê số dòng theo loại đường dẫn

print("\n" + "=" * 80)
print("SỐ DÒNG THEO LOẠI ĐƯỜNG DẪN")
print("=" * 80)

display(
    linkage_df["path_type"]
    .value_counts(dropna=False)
    .rename_axis("loai_duong_dan")
    .reset_index(name="so_luong")
)

# 5. Thống kê số dòng liên kết theo bảng nguồn và loại đường dẫn

print("\n" + "=" * 80)
print("SỐ DÒNG LIÊN KẾT THEO BẢNG NGUỒN")
print("=" * 80)

display(
    linkage_df
    .groupby(
        ["split", "path_type"],
        dropna=False
    )
    .size()
    .reset_index(name="so_luong")
)

Đường dẫn bảng liên kết:
C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb02_csv_dicom_linkage.csv

Các cột trong bảng liên kết Notebook 02:
['split', 'csv_row', 'path_type', 'csv_path', 'case_folder', 'study_uid', 'series_uid', 'alias_filename', 'path_components', 'candidate_count', 'candidate_paths', 'candidate_sop_uids', 'series_descriptions', 'linkage_status']

Tổng số dòng: 10,704

SỐ DÒNG THEO LOẠI ĐƯỜNG DẪN


,loai_duong_dan,so_luong
0,image file path,3568
1,cropped image file path,3568
2,ROI mask file path,3568



SỐ DÒNG LIÊN KẾT THEO BẢNG NGUỒN


,split,path_type,so_luong
0,calc_test,ROI mask file path,326
1,calc_test,cropped image file path,326
2,calc_test,image file path,326
3,calc_train,ROI mask file path,1546
4,calc_train,cropped image file path,1546
5,calc_train,image file path,1546
6,mass_test,ROI mask file path,378
7,mass_test,cropped image file path,378
8,mass_test,image file path,378
9,mass_train,ROI mask file path,1318


### PHẦN 03 — KIỂM TRA CẤU TRÚC VÀ NHẬN DIỆN TỔN THƯƠNG ỨNG VIÊN

### 3.1. Kiểm tra các trường nhận diện tổn thương

Trước khi xây dựng bảng tổn thương cấp độ lesion, cần kiểm tra các trường có khả năng định danh một tổn thương trong dữ liệu nguồn.

Các trường dự kiến sử dụng gồm:

- `patient_id`
- `laterality`
- `lesion_type`
- `abnormality_id`

Đồng thời, kiểm tra các trường mô tả ảnh và nhãn bệnh lý nhằm phục vụ việc đối chiếu giữa những annotation có cùng khóa nhận diện.

In [81]:
# 3.1 — KIỂM TRA CÁC TRƯỜNG NHẬN DIỆN TỔN THƯƠNG

# 1. Kiểm tra bảng dữ liệu đầu vào

if "annotation_master_df" not in globals():
    raise NameError(
        "Không tìm thấy annotation_master_df. "
        "Hãy chạy hoàn tất Phần 02 trước."
    )

nb03_audit_df = annotation_master_df.copy()

print("=" * 80)
print("KIỂM TRA DỮ LIỆU ĐẦU VÀO")
print("=" * 80)

print(f"Tổng số annotation: {len(nb03_audit_df):,}")
print(f"Tổng số cột: {len(nb03_audit_df.columns):,}")

# 2. Kiểm tra sự tồn tại của các cột cần thiết

required_columns = [
    "patient_id",
    "laterality",
    "lesion_type",
    "abnormality_id",
    "image_view",
    "pathology",
    "source_split",
    "abnormality_type",
]

missing_columns = [
    column
    for column in required_columns
    if column not in nb03_audit_df.columns
]

if missing_columns:
    raise ValueError(
        f"Không tìm thấy các cột cần thiết: {missing_columns}"
    )

print("\n[ĐẠT] Tất cả các cột cần thiết đều tồn tại.")

# 3. Kiểm tra giá trị thiếu, chuỗi rỗng và số lượng giá trị phân biệt

check_columns = [
    "patient_id",
    "laterality",
    "lesion_type",
    "abnormality_id",
    "image_view",
    "pathology",
    "source_split",
    "abnormality_type",
]

quality_records = []

for column in check_columns:

    series = nb03_audit_df[column]

    normalized = series.astype("string").str.strip()

    quality_records.append({
        "ten_cot": column,
        "kieu_du_lieu": str(series.dtype),
        "so_gia_tri_thieu": int(series.isna().sum()),
        "so_chuoi_rong": int(
            (series.notna() & normalized.eq("")).sum()
        ),
        "so_gia_tri_phan_biet": int(normalized.nunique(dropna=True)),
    })

key_field_quality_df = pd.DataFrame(quality_records)

print("\n" + "=" * 80)
print("BÁO CÁO CHẤT LƯỢNG CÁC TRƯỜNG")
print("=" * 80)

display(key_field_quality_df)

# 4. Kiểm tra các giá trị phân biệt của từng trường

value_check_columns = [
    "laterality",
    "lesion_type",
    "image_view",
    "pathology",
    "source_split",
    "abnormality_type",
]

for column in value_check_columns:

    print("\n" + "=" * 80)
    print(f"PHÂN BỐ GIÁ TRỊ: {column}")
    print("=" * 80)

    display(
        nb03_audit_df[column]
        .value_counts(dropna=False)
        .rename_axis("gia_tri")
        .reset_index(name="so_luong")
    )

# 5. Kiểm tra phân bố abnormality_id

print("\n" + "=" * 80)
print("PHÂN BỐ ABNORMALITY ID")
print("=" * 80)

display(
    nb03_audit_df["abnormality_id"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("abnormality_id")
    .reset_index(name="so_luong")
)

KIỂM TRA DỮ LIỆU ĐẦU VÀO
Tổng số annotation: 3,568
Tổng số cột: 39

[ĐẠT] Tất cả các cột cần thiết đều tồn tại.

BÁO CÁO CHẤT LƯỢNG CÁC TRƯỜNG


,ten_cot,kieu_du_lieu,so_gia_tri_thieu,so_chuoi_rong,so_gia_tri_phan_biet
0,patient_id,str,0,0,1566
1,laterality,str,0,0,2
2,lesion_type,str,0,0,2
3,abnormality_id,int64,0,0,7
4,image_view,str,0,0,2
5,pathology,str,0,0,3
6,source_split,str,0,0,2
7,abnormality_type,str,0,0,2



PHÂN BỐ GIÁ TRỊ: laterality


,gia_tri,so_luong
0,LEFT,1819
1,RIGHT,1749



PHÂN BỐ GIÁ TRỊ: lesion_type


,gia_tri,so_luong
0,calcification,1872
1,mass,1696


... (6 outputs left out)


PHÂN BỐ GIÁ TRỊ: abnormality_type


,gia_tri,so_luong
0,calcification,1872
1,mass,1696



PHÂN BỐ ABNORMALITY ID


,abnormality_id,so_luong
0,1,3011
1,2,351
2,3,122
3,4,47
4,5,23
5,6,12
6,7,2


### lưu ý:
- abnormality_id chỉ có 7 giá trị phân biệt, nên không thể dùng riêng trường này để định danh tổn thương.
- patient_id cũng không đủ vì một bệnh nhân có thể có nhiều tổn thương.
- CC và MLO là hai góc chụp cần xem xét khi nhóm các annotation.
- abnormality_type và lesion_type có phân bố giống nhau, nhưng vẫn cần kiểm tra từng dòng xem hai trường có thực sự tương ứng hay không.
- Không gộp BENIGN_WITHOUT_CALLBACK vào BENIGN.

### 3.2. Xây dựng và đánh giá khóa nhận diện tổn thương ứng viên

- Tạm thời sử dụng tổ hợp bốn trường:
Candidate Key = (patient_id,laterality,lesion_type,abnormality_id)
- Đây mới là khóa nhận diện ứng viên, chưa được xác nhận là định danh tổn thương cuối cùng.
### kiểm tra:
1. Số nhóm ứng viên tạo được từ 3.568 annotation.
2. Phân bố số annotation trong mỗi nhóm.
3. Những nhóm có cả hai góc chụp CC và MLO.
4. Tính nhất quán của pathology, source_split, abnormality_type.
5. Các trường hợp có nhiều annotation cùng một góc chụp.

In [82]:
# 3.2 — XÂY DỰNG VÀ ĐÁNH GIÁ NHÓM TỔN THƯƠNG ỨNG VIÊN

# 1. Tạo bản sao phục vụ kiểm tra

if "nb03_audit_df" not in globals():
    raise NameError(
        "Không tìm thấy nb03_audit_df. "
        "Hãy chạy hoàn tất Phần 03.1 trước."
    )

candidate_audit_df = nb03_audit_df.copy()

print("=" * 80)
print("KIỂM TRA DỮ LIỆU ĐẦU VÀO")
print("=" * 80)

print(f"Tổng số annotation: {len(candidate_audit_df):,}")

# 2. Chuẩn hóa tạm thời các trường khóa

candidate_audit_df["_patient_key"] = (
    candidate_audit_df["patient_id"]
    .astype("string")
    .str.strip()
    .str.upper()
)

candidate_audit_df["_laterality_key"] = (
    candidate_audit_df["laterality"]
    .astype("string")
    .str.strip()
    .str.upper()
)

candidate_audit_df["_lesion_type_key"] = (
    candidate_audit_df["lesion_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)

candidate_audit_df["_abnormality_id_key"] = (
    candidate_audit_df["abnormality_id"]
    .astype("string")
    .str.strip()
)

candidate_audit_df["_image_view_key"] = (
    candidate_audit_df["image_view"]
    .astype("string")
    .str.strip()
    .str.upper()
)

candidate_audit_df["_pathology_key"] = (
    candidate_audit_df["pathology"]
    .astype("string")
    .str.strip()
    .str.upper()
)

candidate_audit_df["_source_split_key"] = (
    candidate_audit_df["source_split"]
    .astype("string")
    .str.strip()
    .str.lower()
)

candidate_audit_df["_abnormality_type_key"] = (
    candidate_audit_df["abnormality_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# 3. Kiểm tra tính đầy đủ của khóa ứng viên

candidate_key_columns = [
    "_patient_key",
    "_laterality_key",
    "_lesion_type_key",
    "_abnormality_id_key",
]

missing_key_mask = candidate_audit_df[
    candidate_key_columns
].isna().any(axis=1)

empty_key_mask = (
    candidate_audit_df[candidate_key_columns]
    .eq("")
    .any(axis=1)
)

invalid_key_rows_df = candidate_audit_df.loc[
    missing_key_mask | empty_key_mask
].copy()

print("\n" + "=" * 80)
print("KIỂM TRA TÍNH ĐẦY ĐỦ CỦA KHÓA ỨNG VIÊN")
print("=" * 80)

print(f"Số bản ghi có khóa hợp lệ: {len(candidate_audit_df) - len(invalid_key_rows_df):,}")
print(f"Số bản ghi có thành phần khóa thiếu/rỗng: {len(invalid_key_rows_df):,}")

if not invalid_key_rows_df.empty:
    display(
        invalid_key_rows_df[
            [
                "patient_id",
                "laterality",
                "lesion_type",
                "abnormality_id",
                "source_table",
            ]
        ].head(20)
    )


# Chỉ tiếp tục nhóm khi tất cả bản ghi có khóa hợp lệ.
if not invalid_key_rows_df.empty:
    raise ValueError(
        "Có bản ghi thiếu thành phần khóa nhận diện. "
        "Cần kiểm tra trước khi tiếp tục."
    )

# 4. Hàm tổng hợp giá trị phân biệt trong mỗi nhóm

def summarize_unique_values(series):
    values = {
        str(value).strip()
        for value in series.dropna()
        if str(value).strip()
    }

    return " | ".join(sorted(values))

# 5. Nhóm annotation theo khóa ứng viên

grouped = candidate_audit_df.groupby(
    candidate_key_columns,
    dropna=False,
    sort=True,
)

candidate_group_summary_df = grouped.agg(
    so_annotation=("_patient_key", "size"),

    so_goc_chup=("_image_view_key", "nunique"),
    danh_sach_goc_chup=(
        "_image_view_key",
        summarize_unique_values,
    ),

    so_pathology=("_pathology_key", "nunique"),
    danh_sach_pathology=(
        "_pathology_key",
        summarize_unique_values,
    ),

    so_source_split=("_source_split_key", "nunique"),
    danh_sach_source_split=(
        "_source_split_key",
        summarize_unique_values,
    ),

    so_abnormality_type=("_abnormality_type_key", "nunique"),
    danh_sach_abnormality_type=(
        "_abnormality_type_key",
        summarize_unique_values,
    ),

    so_bang_nguon=("source_table", "nunique"),
).reset_index()

candidate_group_summary_df = candidate_group_summary_df.rename(
    columns={
        "_patient_key": "patient_id",
        "_laterality_key": "laterality",
        "_lesion_type_key": "lesion_type",
        "_abnormality_id_key": "abnormality_id",
    }
)

# Phân loại kiểu xuất hiện của góc chụp.
def classify_view_pattern(value):
    views = set(value.split(" | ")) if value else set()

    if views == {"CC", "MLO"}:
        return "CC + MLO"
    elif views == {"CC"}:
        return "Chỉ CC"
    elif views == {"MLO"}:
        return "Chỉ MLO"
    else:
        return "Giá trị góc chụp khác"


candidate_group_summary_df["mau_goc_chup"] = (
    candidate_group_summary_df["danh_sach_goc_chup"]
    .apply(classify_view_pattern)
)

candidate_group_summary_df = (
    candidate_group_summary_df
    .sort_values(
        by=["so_annotation", "patient_id"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

# 6. Kiểm tra khả năng bảo toàn số lượng annotation

total_annotations_after_grouping = int(
    candidate_group_summary_df["so_annotation"].sum()
)

print("\n" + "=" * 80)
print("KẾT QUẢ NHÓM ỨNG VIÊN")
print("=" * 80)

print(f"Tổng số annotation ban đầu: {len(candidate_audit_df):,}")
print(f"Tổng số nhóm ứng viên: {len(candidate_group_summary_df):,}")
print(
    "Tổng số annotation sau khi nhóm:",
    f"{total_annotations_after_grouping:,}"
)

print(
    "[ĐẠT] Bảo toàn số lượng annotation."
    if total_annotations_after_grouping == len(candidate_audit_df)
    else "[CẢNH BÁO] Số lượng annotation không khớp."
)

# 7. Phân bố số annotation trong mỗi nhóm

group_size_distribution_df = (
    candidate_group_summary_df["so_annotation"]
    .value_counts()
    .sort_index()
    .rename_axis("so_annotation_moi_nhom")
    .reset_index(name="so_nhom")
)

print("\n" + "=" * 80)
print("PHÂN BỐ SỐ ANNOTATION TRONG MỖI NHÓM")
print("=" * 80)

display(group_size_distribution_df)

# 8. Phân bố nhóm theo góc chụp

view_pattern_distribution_df = (
    candidate_group_summary_df["mau_goc_chup"]
    .value_counts()
    .rename_axis("mau_goc_chup")
    .reset_index(name="so_nhom")
)

print("\n" + "=" * 80)
print("PHÂN BỐ NHÓM THEO GÓC CHỤP")
print("=" * 80)

display(view_pattern_distribution_df)

# 9. Phân bố nhóm theo loại tổn thương

lesion_type_distribution_df = (
    candidate_group_summary_df
    .groupby("lesion_type", dropna=False)
    .agg(
        so_nhom=("patient_id", "size"),
        so_annotation=("so_annotation", "sum"),
        so_nhom_nhieu_annotation=(
            "so_annotation",
            lambda values: int((values > 1).sum()),
        ),
    )
    .reset_index()
)

print("\n" + "=" * 80)
print("PHÂN BỐ NHÓM THEO LOẠI TỔN THƯƠNG")
print("=" * 80)

display(lesion_type_distribution_df)

# 10. Phát hiện nhóm có thông tin không nhất quán

inconsistent_candidate_groups_df = candidate_group_summary_df.loc[
    (candidate_group_summary_df["so_pathology"] > 1)
    | (candidate_group_summary_df["so_source_split"] > 1)
    | (candidate_group_summary_df["so_abnormality_type"] > 1)
].copy()

print("\n" + "=" * 80)
print("NHÓM CÓ THÔNG TIN KHÔNG NHẤT QUÁN")
print("=" * 80)

print(
    "Số nhóm không nhất quán:",
    f"{len(inconsistent_candidate_groups_df):,}"
)

display(inconsistent_candidate_groups_df.head(30))

# 11. Kiểm tra annotation trùng cùng một góc chụp trong một nhóm

same_view_group_counts_df = (
    candidate_audit_df
    .groupby(
        candidate_key_columns + ["_image_view_key"],
        dropna=False,
    )
    .size()
    .reset_index(name="so_annotation_cung_goc_chup")
)

duplicate_same_view_df = same_view_group_counts_df.loc[
    same_view_group_counts_df["so_annotation_cung_goc_chup"] > 1
].copy()

print("\n" + "=" * 80)
print("NHÓM CÓ NHIỀU ANNOTATION TRÊN CÙNG GÓC CHỤP")
print("=" * 80)

print(
    "Số trường hợp nhóm và góc chụp có nhiều annotation:",
    f"{len(duplicate_same_view_df):,}"
)

display(duplicate_same_view_df.head(30))

# 12. Kiểm tra sự tương ứng giữa lesion_type và abnormality_type

type_mismatch_df = candidate_audit_df.loc[
    candidate_audit_df["_lesion_type_key"]
    != candidate_audit_df["_abnormality_type_key"]
].copy()

print("\n" + "=" * 80)
print("ĐỐI CHIẾU LESION_TYPE VÀ ABNORMALITY_TYPE")
print("=" * 80)

print(
    "Số annotation có hai trường không tương ứng:",
    f"{len(type_mismatch_df):,}"
)

if not type_mismatch_df.empty:
    display(
        type_mismatch_df[
            [
                "patient_id",
                "lesion_type",
                "abnormality_type",
                "source_table",
                "source_row_number",
            ]
        ].head(30)
    )

# 13. Xem trước các nhóm có nhiều annotation

print("\n" + "=" * 80)
print("20 NHÓM ỨNG VIÊN CÓ NHIỀU ANNOTATION NHẤT")
print("=" * 80)

display(
    candidate_group_summary_df[
        [
            "patient_id",
            "laterality",
            "lesion_type",
            "abnormality_id",
            "so_annotation",
            "danh_sach_goc_chup",
            "danh_sach_pathology",
            "danh_sach_source_split",
        ]
    ].head(20)
)

KIỂM TRA DỮ LIỆU ĐẦU VÀO
Tổng số annotation: 3,568

KIỂM TRA TÍNH ĐẦY ĐỦ CỦA KHÓA ỨNG VIÊN
Số bản ghi có khóa hợp lệ: 3,568
Số bản ghi có thành phần khóa thiếu/rỗng: 0

KẾT QUẢ NHÓM ỨNG VIÊN
Tổng số annotation ban đầu: 3,568
Tổng số nhóm ứng viên: 2,050
Tổng số annotation sau khi nhóm: 3,568
[ĐẠT] Bảo toàn số lượng annotation.

PHÂN BỐ SỐ ANNOTATION TRONG MỖI NHÓM


,so_annotation_moi_nhom,so_nhom
0,1,532
1,2,1518



PHÂN BỐ NHÓM THEO GÓC CHỤP


,mau_goc_chup,so_nhom
0,CC + MLO,1518
1,Chỉ MLO,378
2,Chỉ CC,154



PHÂN BỐ NHÓM THEO LOẠI TỔN THƯƠNG


,lesion_type,so_nhom,so_annotation,so_nhom_nhieu_annotation
0,calcification,1045,1872,827
1,mass,1005,1696,691


... (2 outputs left out)


NHÓM CÓ NHIỀU ANNOTATION TRÊN CÙNG GÓC CHỤP
Số trường hợp nhóm và góc chụp có nhiều annotation: 0


,_patient_key,_laterality_key,_lesion_type_key,_abnormality_id_key,_image_view_key,so_annotation_cung_goc_chup



ĐỐI CHIẾU LESION_TYPE VÀ ABNORMALITY_TYPE
Số annotation có hai trường không tương ứng: 0

20 NHÓM ỨNG VIÊN CÓ NHIỀU ANNOTATION NHẤT


,patient_id,laterality,lesion_type,abnormality_id,so_annotation,danh_sach_goc_chup,danh_sach_pathology,danh_sach_source_split
0,P_00001,LEFT,mass,1,2,CC | MLO,MALIGNANT,train
1,P_00004,LEFT,mass,1,2,CC | MLO,BENIGN,train
2,P_00005,RIGHT,calcification,1,2,CC | MLO,MALIGNANT,train
3,P_00007,LEFT,calcification,1,2,CC | MLO,BENIGN,train
4,P_00008,LEFT,calcification,1,2,CC | MLO,BENIGN_WITHOUT_CALLBACK,train
5,P_00008,LEFT,calcification,2,2,CC | MLO,BENIGN_WITHOUT_CALLBACK,train
6,P_00008,LEFT,calcification,3,2,CC | MLO,BENIGN_WITHOUT_CALLBACK,train
7,P_00008,RIGHT,calcification,1,2,CC | MLO,BENIGN_WITHOUT_CALLBACK,train
8,P_00008,RIGHT,calcification,2,2,CC | MLO,BENIGN_WITHOUT_CALLBACK,train
9,P_00008,RIGHT,calcification,3,2,CC | MLO,BENIGN_WITHOUT_CALLBACK,train


### 3.3. Truy xuất và kiểm tra các annotation có nhãn bệnh lý bất nhất

Kết quả nhóm ứng viên phát hiện 6 nhóm có cùng khóa nhận diện nhưng có nhiều giá trị `pathology` khác nhau.

Để xác minh, truy xuất toàn bộ annotation thuộc 6 nhóm này và đối chiếu:

- Mã bệnh nhân, bên vú, loại tổn thương và mã bất thường.
- Góc chụp CC/MLO.
- Nhãn `pathology` và mức đánh giá `assessment`.
- Bảng nguồn và số thứ tự bản ghi gốc.
- Đường dẫn ảnh đầy đủ, ảnh cắt và ROI mask.

Mục tiêu là xác định liệu các bản ghi có thực sự thuộc cùng một tổn thương hay có dấu hiệu bất nhất trong dữ liệu nguồn.

In [83]:
# 3.3 — TRUY XUẤT ANNOTATION CÓ PATHOLOGY BẤT NHẤT

# 1. Lấy danh sách khóa ứng viên có pathology không nhất quán

pathology_conflict_keys_df = candidate_group_summary_df.loc[
    candidate_group_summary_df["so_pathology"] > 1,
    [
        "patient_id",
        "laterality",
        "lesion_type",
        "abnormality_id",
    ],
].copy()

print("=" * 80)
print("DANH SÁCH KHÓA ỨNG VIÊN CÓ PATHOLOGY BẤT NHẤT")
print("=" * 80)

display(pathology_conflict_keys_df)

# 2. Truy xuất toàn bộ annotation thuộc các khóa bất nhất

conflict_key_set = set(
    pathology_conflict_keys_df.itertuples(
        index=False,
        name=None,
    )
)

row_key_tuples = list(
    zip(
        candidate_audit_df["_patient_key"],
        candidate_audit_df["_laterality_key"],
        candidate_audit_df["_lesion_type_key"],
        candidate_audit_df["_abnormality_id_key"],
    )
)

conflict_mask = pd.Series(
    [
        key in conflict_key_set
        for key in row_key_tuples
    ],
    index=candidate_audit_df.index,
)

pathology_conflict_rows_df = (
    candidate_audit_df.loc[conflict_mask]
    .copy()
    .sort_values(
        by=[
            "_patient_key",
            "_laterality_key",
            "_lesion_type_key",
            "_abnormality_id_key",
            "_image_view_key",
        ]
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("SỐ LƯỢNG ANNOTATION ĐƯỢC TRUY XUẤT")
print("=" * 80)

print(
    f"Số nhóm bất nhất: {len(pathology_conflict_keys_df):,}"
)

print(
    f"Số annotation liên quan: {len(pathology_conflict_rows_df):,}"
)

# 3. Chọn các trường cần đối chiếu

detail_columns = [
    "patient_id",
    "laterality",
    "lesion_type",
    "abnormality_id",
    "image_view",
    "pathology",
    "assessment",
    "abnormality_type",
    "source_split",
    "source_table",
    "source_csv",
    "source_row_index",
    "source_row_number",
]

# Bổ sung các thuộc tính đặc trưng nếu có.
optional_columns = [
    "breast_density",
    "subtlety",
    "calc_type",
    "calc_distribution",
    "mass_shape",
    "mass_margins",
    "full_image_csv_path",
    "crop_image_csv_path",
    "roi_mask_csv_path",
]

detail_columns.extend([
    column
    for column in optional_columns
    if column in pathology_conflict_rows_df.columns
])

# Giữ lại thứ tự cột và loại bỏ tên cột trùng.
detail_columns = list(dict.fromkeys(detail_columns))

print("\n" + "=" * 80)
print("CHI TIẾT CÁC ANNOTATION BẤT NHẤT")
print("=" * 80)

with pd.option_context("display.max_colwidth", 100):
    display(
        pathology_conflict_rows_df[detail_columns]
    )

# 4. Tổng hợp đối chiếu theo từng nhóm

pathology_conflict_overview_df = (
    pathology_conflict_rows_df
    .groupby(
        [
            "_patient_key",
            "_laterality_key",
            "_lesion_type_key",
            "_abnormality_id_key",
        ],
        dropna=False,
    )
    .agg(
        so_annotation=("patient_id", "size"),
        danh_sach_goc_chup=(
            "_image_view_key",
            summarize_unique_values,
        ),
        danh_sach_pathology=(
            "_pathology_key",
            summarize_unique_values,
        ),
        danh_sach_assessment=(
            "assessment",
            summarize_unique_values,
        ),
        danh_sach_bang_nguon=(
            "source_table",
            summarize_unique_values,
        ),
        danh_sach_dong_nguon=(
            "source_row_number",
            summarize_unique_values,
        ),
    )
    .reset_index()
)

pathology_conflict_overview_df = (
    pathology_conflict_overview_df.rename(
        columns={
            "_patient_key": "patient_id",
            "_laterality_key": "laterality",
            "_lesion_type_key": "lesion_type",
            "_abnormality_id_key": "abnormality_id",
        }
    )
)

print("\n" + "=" * 80)
print("TỔNG HỢP 6 NHÓM PATHOLOGY BẤT NHẤT")
print("=" * 80)

display(pathology_conflict_overview_df)

# 5. Kiểm tra các trường raw liên quan nếu được lưu trong bảng

raw_check_columns = [
    column
    for column in pathology_conflict_rows_df.columns
    if column in [
        "raw__pathology",
        "raw__assessment",
        "raw__image view",
        "raw__abnormality id",
    ]
]

if raw_check_columns:
    print("\n" + "=" * 80)
    print("ĐỐI CHIẾU GIÁ TRỊ TỪ CỘT GỐC")
    print("=" * 80)

    display(
        pathology_conflict_rows_df[
            [
                "patient_id",
                "image_view",
                "pathology",
                "source_table",
                "source_row_number",
            ] + raw_check_columns
        ]
    )

else:
    print(
        "\nKhông tìm thấy các cột raw tương ứng trong bảng hiện tại. "
        "Có thể tiếp tục đối chiếu bằng source_csv và source_row_number."
    )

DANH SÁCH KHÓA ỨNG VIÊN CÓ PATHOLOGY BẤT NHẤT


,patient_id,laterality,lesion_type,abnormality_id
323,P_00419,LEFT,mass,1
324,P_00419,LEFT,mass,2
677,P_00858,RIGHT,calcification,1
678,P_00858,RIGHT,calcification,2
928,P_01200,LEFT,calcification,1
1247,P_01623,RIGHT,mass,1



SỐ LƯỢNG ANNOTATION ĐƯỢC TRUY XUẤT
Số nhóm bất nhất: 6
Số annotation liên quan: 12

CHI TIẾT CÁC ANNOTATION BẤT NHẤT


,patient_id,laterality,lesion_type,abnormality_id,image_view,pathology,assessment,abnormality_type,source_split,source_table,source_csv,source_row_index,source_row_number,breast_density,subtlety,calc_type,calc_distribution,mass_shape,mass_margins,full_image_csv_path,crop_image_csv_path,roi_mask_csv_path
0,P_00419,LEFT,mass,1,CC,BENIGN,3,mass,train,mass_train,mass_case_description_train_set.csv,286,287,2,5,NaN,NaN,OVAL,CIRCUMSCRIBED,Mass-Training_P_00419_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.185385464112640817231484525363691131563/1...,Mass-Training_P_00419_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.364380877410451490835437482611931111685...,Mass-Training_P_00419_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.364380877410451490835437482611931111685...
1,P_00419,LEFT,mass,1,MLO,MALIGNANT,5,mass,train,mass_train,mass_case_description_train_set.csv,288,289,2,5,NaN,NaN,NaN,ILL_DEFINED,Mass-Training_P_00419_LEFT_MLO/1.3.6.1.4.1.9590.100.1.2.35251714711534395303147864661551477629/1...,Mass-Training_P_00419_LEFT_MLO_1/1.3.6.1.4.1.9590.100.1.2.37591886681288580470826691517355817152...,Mass-Training_P_00419_LEFT_MLO_1/1.3.6.1.4.1.9590.100.1.2.37591886681288580470826691517355817152...
2,P_00419,LEFT,mass,2,CC,MALIGNANT,5,mass,train,mass_train,mass_case_description_train_set.csv,287,288,2,5,NaN,NaN,NaN,ILL_DEFINED,Mass-Training_P_00419_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.185385464112640817231484525363691131563/1...,Mass-Training_P_00419_LEFT_CC_2/1.3.6.1.4.1.9590.100.1.2.67263285411337048620966122671267935581/...,Mass-Training_P_00419_LEFT_CC_2/1.3.6.1.4.1.9590.100.1.2.67263285411337048620966122671267935581/...
3,P_00419,LEFT,mass,2,MLO,BENIGN,3,mass,train,mass_train,mass_case_description_train_set.csv,289,290,2,5,NaN,NaN,OVAL,CIRCUMSCRIBED,Mass-Training_P_00419_LEFT_MLO/1.3.6.1.4.1.9590.100.1.2.35251714711534395303147864661551477629/1...,Mass-Training_P_00419_LEFT_MLO_2/1.3.6.1.4.1.9590.100.1.2.20032692721311178093390846104407778063...,Mass-Training_P_00419_LEFT_MLO_2/1.3.6.1.4.1.9590.100.1.2.20032692721311178093390846104407778063...
4,P_00858,RIGHT,calcification,1,CC,BENIGN,4,calcification,train,calc_train,calc_case_description_train_set.csv,681,682,3,4,PLEOMORPHIC,CLUSTERED,NaN,NaN,Calc-Training_P_00858_RIGHT_CC/1.3.6.1.4.1.9590.100.1.2.378015788313560566134597838150194477905/...,Calc-Training_P_00858_RIGHT_CC_1/1.3.6.1.4.1.9590.100.1.2.13415240991294965013646063783401984280...,Calc-Training_P_00858_RIGHT_CC_1/1.3.6.1.4.1.9590.100.1.2.13415240991294965013646063783401984280...
5,P_00858,RIGHT,calcification,1,MLO,BENIGN_WITHOUT_CALLBACK,3,calcification,train,calc_train,calc_case_description_train_set.csv,683,684,3,4,PUNCTATE,CLUSTERED,NaN,NaN,Calc-Training_P_00858_RIGHT_MLO/1.3.6.1.4.1.9590.100.1.2.154701552912026387537768204120519793340...,Calc-Training_P_00858_RIGHT_MLO_1/1.3.6.1.4.1.9590.100.1.2.1912045448115844893059947176434128413...,Calc-Training_P_00858_RIGHT_MLO_1/1.3.6.1.4.1.9590.100.1.2.1912045448115844893059947176434128413...
6,P_00858,RIGHT,calcification,2,CC,BENIGN_WITHOUT_CALLBACK,3,calcification,train,calc_train,calc_case_description_train_set.csv,682,683,3,4,PUNCTATE,CLUSTERED,NaN,NaN,Calc-Training_P_00858_RIGHT_CC/1.3.6.1.4.1.9590.100.1.2.378015788313560566134597838150194477905/...,Calc-Training_P_00858_RIGHT_CC_2/1.3.6.1.4.1.9590.100.1.2.39902308581305785692817254797130661186...,Calc-Training_P_00858_RIGHT_CC_2/1.3.6.1.4.1.9590.100.1.2.39902308581305785692817254797130661186...
7,P_00858,RIGHT,calcification,2,MLO,BENIGN,4,calcification,train,calc_train,calc_case_description_train_set.csv,684,685,3,4,PLEOMORPHIC,CLUSTERED,NaN,NaN,Calc-Training_P_00858_RIGHT_MLO/1.3.6.1.4.1.9590.100.1.2.154701552912026387537768204120519793340...,Calc-Training_P_00858_RIGHT_MLO_2/1.3.6.1.4.1.9590.100.1.2.9396998901116382760243392950336869300...,Calc-Training_P_00858_RIGHT_MLO_2/1.3.6.1.4.1.9590.100.1.2.9396998901116382760243392950336869300...
8,P_01200,LEFT,calcification,1,CC,BENIGN_WITHOUT_CALLBACK,2,calcification,train,calc_train,calc_case_description_train_set.csv,


TỔNG HỢP 6 NHÓM PATHOLOGY BẤT NHẤT


,patient_id,laterality,lesion_type,abnormality_id,so_annotation,danh_sach_goc_chup,danh_sach_pathology,danh_sach_assessment,danh_sach_bang_nguon,danh_sach_dong_nguon
0,P_00419,LEFT,mass,1,2,CC | MLO,BENIGN | MALIGNANT,3 | 5,mass_train,287 | 289
1,P_00419,LEFT,mass,2,2,CC | MLO,BENIGN | MALIGNANT,3 | 5,mass_train,288 | 290
2,P_00858,RIGHT,calcification,1,2,CC | MLO,BENIGN | BENIGN_WITHOUT_CALLBACK,3 | 4,calc_train,682 | 684
3,P_00858,RIGHT,calcification,2,2,CC | MLO,BENIGN | BENIGN_WITHOUT_CALLBACK,3 | 4,calc_train,683 | 685
4,P_01200,LEFT,calcification,1,2,CC | MLO,BENIGN | BENIGN_WITHOUT_CALLBACK,2,calc_train,939 | 940
5,P_01623,RIGHT,mass,1,2,CC | MLO,BENIGN | MALIGNANT,5,mass_test,329 | 330



ĐỐI CHIẾU GIÁ TRỊ TỪ CỘT GỐC


,patient_id,image_view,pathology,source_table,source_row_number,raw__image view,raw__abnormality id,raw__assessment,raw__pathology
0,P_00419,CC,BENIGN,mass_train,287,CC,1,3,BENIGN
1,P_00419,MLO,MALIGNANT,mass_train,289,MLO,1,5,MALIGNANT
2,P_00419,CC,MALIGNANT,mass_train,288,CC,2,5,MALIGNANT
3,P_00419,MLO,BENIGN,mass_train,290,MLO,2,3,BENIGN
4,P_00858,CC,BENIGN,calc_train,682,CC,1,4,BENIGN
5,P_00858,MLO,BENIGN_WITHOUT_CALLBACK,calc_train,684,MLO,1,3,BENIGN_WITHOUT_CALLBACK
6,P_00858,CC,BENIGN_WITHOUT_CALLBACK,calc_train,683,CC,2,3,BENIGN_WITHOUT_CALLBACK
7,P_00858,MLO,BENIGN,calc_train,685,MLO,2,4,BENIGN
8,P_01200,CC,BENIGN_WITHOUT_CALLBACK,calc_train,939,CC,1,2,BENIGN_WITHOUT_CALLBACK
9,P_01200,MLO,BENIGN,calc_train,940,MLO,1,2,BENIGN


### 3.4. Kiểm tra khả năng ghép cặp CC/MLO theo đặc trưng tổn thương

Kết quả kiểm tra Phần 3.3 cho thấy `abnormality_id` có thể không đồng nhất giữa hai góc chụp khi một bệnh nhân có nhiều bất thường cùng loại.

Trong bước này, kiểm tra khả năng ghép các annotation CC và MLO dựa trên những đặc trưng mô tả tổn thương.

Đối với khối u (mass), sử dụng:
- `assessment`
- `subtlety`
- `mass_shape`
- `mass_margins`

Đối với vi vôi hóa (calcification), sử dụng:
- `assessment`
- `subtlety`
- `calc_type`
- `calc_distribution`

Không sử dụng `abnormality_id` và `pathology` làm thành phần của chữ ký ghép cặp.

Kết quả được phân loại thành:
- Có một bản ghi CC và một bản ghi MLO cùng đặc trưng.
- Có nhiều bản ghi cùng đặc trưng, chưa thể ghép duy nhất.
- Chỉ xuất hiện ở một góc chụp.

Các kết quả chỉ dùng để xác định ứng viên ghép cặp, không tự động gộp annotation.

In [84]:
# 3.4 — KIỂM TRA KHẢ NĂNG GHÉP CẶP CC/MLO

# 1. Lấy danh sách bệnh nhân và nhóm tổn thương cần kiểm tra

case_keys_df = (
    pathology_conflict_keys_df[
        [
            "patient_id",
            "laterality",
            "lesion_type",
        ]
    ]
    .drop_duplicates()
    .copy()
)

case_keys = set(
    case_keys_df.itertuples(
        index=False,
        name=None,
    )
)

row_case_keys = list(
    zip(
        candidate_audit_df["_patient_key"],
        candidate_audit_df["_laterality_key"],
        candidate_audit_df["_lesion_type_key"],
    )
)

case_mask = pd.Series(
    [
        key in case_keys
        for key in row_case_keys
    ],
    index=candidate_audit_df.index,
)

affected_case_rows_df = (
    candidate_audit_df.loc[case_mask]
    .copy()
    .reset_index(drop=True)
)

print("=" * 80)
print("CÁC NHÓM BỆNH NHÂN ĐƯỢC KIỂM TRA")
print("=" * 80)

display(case_keys_df)

print(
    f"\nTổng số annotation được truy xuất: "
    f"{len(affected_case_rows_df):,}"
)

# 2. Tạo chữ ký đặc trưng, không sử dụng pathology và abnormality_id

def normalize_feature_value(value):

    if pd.isna(value):
        return "<THIEU>"

    normalized_value = str(value).strip().upper()

    if normalized_value == "":
        return "<RONG>"

    return normalized_value

def build_descriptor_signature(row):

    common_features = [
        "assessment",
        "subtlety",
    ]

    if row["_lesion_type_key"] == "mass":

        specific_features = [
            "mass_shape",
            "mass_margins",
        ]

    elif row["_lesion_type_key"] == "calcification":

        specific_features = [
            "calc_type",
            "calc_distribution",
        ]

    else:
        raise ValueError(
            f"Loại tổn thương không hợp lệ: "
            f"{row['_lesion_type_key']}"
        )

    feature_columns = common_features + specific_features

    return tuple(
        (
            column,
            normalize_feature_value(row[column]),
        )
        for column in feature_columns
    )

affected_case_rows_df["_descriptor_signature"] = (
    affected_case_rows_df.apply(
        build_descriptor_signature,
        axis=1,
    )
)

def format_descriptor_signature(signature):

    return " | ".join(
        f"{column}={value}"
        for column, value in signature
    )

affected_case_rows_df["_descriptor_label"] = (
    affected_case_rows_df["_descriptor_signature"]
    .apply(format_descriptor_signature)
)

# 3. Hiển thị các annotation và đặc trưng của từng góc chụp

detail_columns = [
    "patient_id",
    "laterality",
    "lesion_type",
    "image_view",
    "abnormality_id",
    "pathology",
    "assessment",
    "subtlety",
    "mass_shape",
    "mass_margins",
    "calc_type",
    "calc_distribution",
    "source_table",
    "source_row_number",
    "_descriptor_label",
]

# Chỉ giữ các cột thực sự tồn tại.
detail_columns = [
    column
    for column in detail_columns
    if column in affected_case_rows_df.columns
]

# Sắp xếp trên DataFrame đầy đủ trước khi chọn các cột hiển thị.
affected_case_rows_sorted_df = (
    affected_case_rows_df
    .sort_values(
        by=[
            "_patient_key",
            "_laterality_key",
            "_lesion_type_key",
            "_image_view_key",
            "abnormality_id",
        ]
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("ĐỐI CHIẾU CHI TIẾT CC/MLO")
print("=" * 80)

with pd.option_context("display.max_colwidth", 100):
    display(
        affected_case_rows_sorted_df[detail_columns]
    )

# 4. Tổng hợp các đặc trưng xuất hiện trong từng góc chụp

match_group_columns = [
    "_patient_key",
    "_laterality_key",
    "_lesion_type_key",
    "_descriptor_signature",
]

def summarize_one_view(data, view):

    view_df = data.loc[
        data["_image_view_key"] == view
    ].copy()

    result = (
        view_df
        .groupby(
            match_group_columns,
            dropna=False,
        )
        .agg(
            so_annotation=("patient_id", "size"),

            danh_sach_abnormality_id=(
                "abnormality_id",
                summarize_unique_values,
            ),

            danh_sach_pathology=(
                "pathology",
                summarize_unique_values,
            ),

            dong_nguon=(
                "source_row_number",
                summarize_unique_values,
            ),
        )
        .reset_index()
    )

    return result.rename(
        columns={
            "so_annotation": f"so_annotation_{view}",
            "danh_sach_abnormality_id": f"abnormality_id_{view}",
            "danh_sach_pathology": f"pathology_{view}",
            "dong_nguon": f"dong_nguon_{view}",
        }
    )

cc_summary_df = summarize_one_view(
    affected_case_rows_df,
    "CC",
)

mlo_summary_df = summarize_one_view(
    affected_case_rows_df,
    "MLO",
)

# 5. Đối chiếu hai góc chụp theo chữ ký đặc trưng

descriptor_pairing_audit_df = pd.merge(
    cc_summary_df,
    mlo_summary_df,
    on=match_group_columns,
    how="outer",
)

count_columns = [
    "so_annotation_CC",
    "so_annotation_MLO",
]

for column in count_columns:

    descriptor_pairing_audit_df[column] = (
        descriptor_pairing_audit_df[column]
        .fillna(0)
        .astype(int)
    )

text_columns = [
    "abnormality_id_CC",
    "pathology_CC",
    "dong_nguon_CC",
    "abnormality_id_MLO",
    "pathology_MLO",
    "dong_nguon_MLO",
]

for column in text_columns:

    descriptor_pairing_audit_df[column] = (
        descriptor_pairing_audit_df[column]
        .fillna("")
    )

# 6. Phân loại khả năng ghép cặp

def classify_pairing(row):

    cc_count = row["so_annotation_CC"]
    mlo_count = row["so_annotation_MLO"]

    if cc_count == 1 and mlo_count == 1:
        return "Ứng viên ghép duy nhất"

    elif cc_count > 0 and mlo_count > 0:
        return "Có ở hai góc nhưng ghép chưa duy nhất"

    elif cc_count > 0:
        return "Chỉ có CC"

    else:
        return "Chỉ có MLO"

descriptor_pairing_audit_df["trang_thai_ghep"] = (
    descriptor_pairing_audit_df.apply(
        classify_pairing,
        axis=1,
    )
)

descriptor_pairing_audit_df = (
    descriptor_pairing_audit_df
    .sort_values(
        by=[
            "_patient_key",
            "_laterality_key",
            "_lesion_type_key",
            "_descriptor_signature",
        ]
    )
    .reset_index(drop=True)
)

# 7. Hiển thị kết quả ghép cặp ứng viên

descriptor_pairing_audit_df["dac_trung"] = (
    descriptor_pairing_audit_df[
        "_descriptor_signature"
    ].apply(format_descriptor_signature)
)

result_columns = [
    "_patient_key",
    "_laterality_key",
    "_lesion_type_key",
    "dac_trung",
    "so_annotation_CC",
    "abnormality_id_CC",
    "pathology_CC",
    "dong_nguon_CC",
    "so_annotation_MLO",
    "abnormality_id_MLO",
    "pathology_MLO",
    "dong_nguon_MLO",
    "trang_thai_ghep",
]

print("\n" + "=" * 80)
print("KẾT QUẢ ĐỐI CHIẾU ĐẶC TRƯNG CC/MLO")
print("=" * 80)

with pd.option_context("display.max_colwidth", 100):

    display(
        descriptor_pairing_audit_df[
            result_columns
        ]
    )

# 8. Thống kê trạng thái ghép cặp

pairing_status_summary_df = (
    descriptor_pairing_audit_df[
        "trang_thai_ghep"
    ]
    .value_counts()
    .rename_axis("trang_thai_ghep")
    .reset_index(name="so_nhom_dac_trung")
)

print("\n" + "=" * 80)
print("THỐNG KÊ TRẠNG THÁI GHÉP CẶP")
print("=" * 80)

display(pairing_status_summary_df)

CÁC NHÓM BỆNH NHÂN ĐƯỢC KIỂM TRA


,patient_id,laterality,lesion_type
323,P_00419,LEFT,mass
677,P_00858,RIGHT,calcification
928,P_01200,LEFT,calcification
1247,P_01623,RIGHT,mass



Tổng số annotation được truy xuất: 12

ĐỐI CHIẾU CHI TIẾT CC/MLO


,patient_id,laterality,lesion_type,image_view,abnormality_id,pathology,assessment,subtlety,mass_shape,mass_margins,calc_type,calc_distribution,source_table,source_row_number,_descriptor_label
0,P_00419,LEFT,mass,CC,1,BENIGN,3,5,OVAL,CIRCUMSCRIBED,NaN,NaN,mass_train,287,assessment=3 | subtlety=5 | mass_shape=OVAL | mass_margins=CIRCUMSCRIBED
1,P_00419,LEFT,mass,CC,2,MALIGNANT,5,5,NaN,ILL_DEFINED,NaN,NaN,mass_train,288,assessment=5 | subtlety=5 | mass_shape=<THIEU> | mass_margins=ILL_DEFINED
2,P_00419,LEFT,mass,MLO,1,MALIGNANT,5,5,NaN,ILL_DEFINED,NaN,NaN,mass_train,289,assessment=5 | subtlety=5 | mass_shape=<THIEU> | mass_margins=ILL_DEFINED
3,P_00419,LEFT,mass,MLO,2,BENIGN,3,5,OVAL,CIRCUMSCRIBED,NaN,NaN,mass_train,290,assessment=3 | subtlety=5 | mass_shape=OVAL | mass_margins=CIRCUMSCRIBED
4,P_00858,RIGHT,calcification,CC,1,BENIGN,4,4,NaN,NaN,PLEOMORPHIC,CLUSTERED,calc_train,682,assessment=4 | subtlety=4 | calc_type=PLEOMORPHIC | calc_distribution=CLUSTERED
5,P_00858,RIGHT,calcification,CC,2,BENIGN_WITHOUT_CALLBACK,3,4,NaN,NaN,PUNCTATE,CLUSTERED,calc_train,683,assessment=3 | subtlety=4 | calc_type=PUNCTATE | calc_distribution=CLUSTERED
6,P_00858,RIGHT,calcification,MLO,1,BENIGN_WITHOUT_CALLBACK,3,4,NaN,NaN,PUNCTATE,CLUSTERED,calc_train,684,assessment=3 | subtlety=4 | calc_type=PUNCTATE | calc_distribution=CLUSTERED
7,P_00858,RIGHT,calcification,MLO,2,BENIGN,4,4,NaN,NaN,PLEOMORPHIC,CLUSTERED,calc_train,685,assessment=4 | subtlety=4 | calc_type=PLEOMORPHIC | calc_distribution=CLUSTERED
8,P_01200,LEFT,calcification,CC,1,BENIGN_WITHOUT_CALLBACK,2,5,NaN,NaN,COARSE,NaN,calc_train,939,assessment=2 | subtlety=5 | calc_type=COARSE | calc_distribution=<THIEU>
9,P_01200,LEFT,calcification,MLO,1,BENIGN,2,5,NaN,NaN,COARSE,NaN,calc_train,940,assessment=2 | subtlety=5 | calc_type=COARSE | calc_distribution=<THIEU>



KẾT QUẢ ĐỐI CHIẾU ĐẶC TRƯNG CC/MLO


,_patient_key,_laterality_key,_lesion_type_key,dac_trung,so_annotation_CC,abnormality_id_CC,pathology_CC,dong_nguon_CC,so_annotation_MLO,abnormality_id_MLO,pathology_MLO,dong_nguon_MLO,trang_thai_ghep
0,P_00419,LEFT,mass,assessment=3 | subtlety=5 | mass_shape=OVAL | mass_margins=CIRCUMSCRIBED,1,1,BENIGN,287,1,2,BENIGN,290,Ứng viên ghép duy nhất
1,P_00419,LEFT,mass,assessment=5 | subtlety=5 | mass_shape=<THIEU> | mass_margins=ILL_DEFINED,1,2,MALIGNANT,288,1,1,MALIGNANT,289,Ứng viên ghép duy nhất
2,P_00858,RIGHT,calcification,assessment=3 | subtlety=4 | calc_type=PUNCTATE | calc_distribution=CLUSTERED,1,2,BENIGN_WITHOUT_CALLBACK,683,1,1,BENIGN_WITHOUT_CALLBACK,684,Ứng viên ghép duy nhất
3,P_00858,RIGHT,calcification,assessment=4 | subtlety=4 | calc_type=PLEOMORPHIC | calc_distribution=CLUSTERED,1,1,BENIGN,682,1,2,BENIGN,685,Ứng viên ghép duy nhất
4,P_01200,LEFT,calcification,assessment=2 | subtlety=5 | calc_type=COARSE | calc_distribution=<THIEU>,1,1,BENIGN_WITHOUT_CALLBACK,939,1,1,BENIGN,940,Ứng viên ghép duy nhất
5,P_01623,RIGHT,mass,assessment=5 | subtlety=2 | mass_shape=IRREGULAR | mass_margins=SPICULATED,1,1,MALIGNANT,329,1,1,BENIGN,330,Ứng viên ghép duy nhất



THỐNG KÊ TRẠNG THÁI GHÉP CẶP


,trang_thai_ghep,so_nhom_dac_trung
0,Ứng viên ghép duy nhất,6


### 3.5. Kiểm tra khả năng ghép cặp CC/MLO trên toàn bộ dữ liệu

Các bước trước, một số trường hợp có `abnormality_id` và `pathology` không nhất quán giữa hai góc chụp CC và MLO đã được phát hiện.

Để đánh giá khả năng áp dụng quy tắc ghép cặp trên toàn bộ bộ dữ liệu, bước này mở rộng việc đối chiếu sang tất cả annotation.

Các đặc trưng sử dụng:

Đối với khối u (mass):
- `assessment`
- `subtlety`
- `mass_shape`
- `mass_margins`

Đối với vi vôi hóa (calcification):
- `assessment`
- `subtlety`
- `calc_type`
- `calc_distribution`

Không sử dụng `abnormality_id` và `pathology` để tạo chữ ký đặc trưng.

Mỗi chữ ký được đối chiếu riêng giữa hai góc chụp CC và MLO trong cùng bệnh nhân, cùng bên vú và cùng loại tổn thương.

Các kết quả được phân loại thành:
- Ghép duy nhất: có đúng một annotation CC và một annotation MLO cùng đặc trưng.
- Ghép chưa duy nhất: có nhiều annotation cùng đặc trưng ở ít nhất một góc chụp.
- Chỉ có CC.
- Chỉ có MLO.

Đối với các cặp ghép duy nhất, tiếp tục kiểm tra sự tương ứng của `abnormality_id` và tính nhất quán của `pathology`.

In [85]:
# 3.5 — KIỂM TRA GHÉP CẶP CC/MLO TRÊN TOÀN BỘ DỮ LIỆU

# 1. Kiểm tra và tạo bản sao dữ liệu

if "nb03_audit_df" not in globals():
    raise NameError(
        "Không tìm thấy nb03_audit_df. "
        "Hãy chạy hoàn tất Phần 03.1 trước."
    )

pairing_audit_full_df = nb03_audit_df.copy()

print("=" * 80)
print("KIỂM TRA DỮ LIỆU ĐẦU VÀO")
print("=" * 80)

print(f"Tổng số annotation: {len(pairing_audit_full_df):,}")

# 2. Chuẩn hóa các trường cần sử dụng

pairing_audit_full_df["_patient_key"] = (
    pairing_audit_full_df["patient_id"]
    .astype("string")
    .str.strip()
    .str.upper()
)

pairing_audit_full_df["_laterality_key"] = (
    pairing_audit_full_df["laterality"]
    .astype("string")
    .str.strip()
    .str.upper()
)

pairing_audit_full_df["_lesion_type_key"] = (
    pairing_audit_full_df["lesion_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)

pairing_audit_full_df["_image_view_key"] = (
    pairing_audit_full_df["image_view"]
    .astype("string")
    .str.strip()
    .str.upper()
)

pairing_audit_full_df["_abnormality_id_key"] = (
    pairing_audit_full_df["abnormality_id"]
    .astype("string")
    .str.strip()
)

pairing_audit_full_df["_pathology_key"] = (
    pairing_audit_full_df["pathology"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# 3. Xác định các trường đặc trưng theo loại tổn thương

required_feature_columns = [
    "assessment",
    "subtlety",
    "mass_shape",
    "mass_margins",
    "calc_type",
    "calc_distribution",
]

missing_feature_columns = [
    column
    for column in required_feature_columns
    if column not in pairing_audit_full_df.columns
]

if missing_feature_columns:
    raise ValueError(
        f"Thiếu các trường đặc trưng: {missing_feature_columns}"
    )

# 4. Chuẩn hóa giá trị đặc trưng

def normalize_pairing_value(value):

    if pd.isna(value):
        return "<THIEU>"

    normalized_value = str(value).strip().upper()

    if normalized_value == "":
        return "<RONG>"

    return normalized_value

# 5. Xây dựng chữ ký đặc trưng cho toàn bộ annotation

def create_pairing_signature(row):

    common_features = [
        "assessment",
        "subtlety",
    ]

    if row["_lesion_type_key"] == "mass":

        specific_features = [
            "mass_shape",
            "mass_margins",
        ]

    elif row["_lesion_type_key"] == "calcification":

        specific_features = [
            "calc_type",
            "calc_distribution",
        ]

    else:
        raise ValueError(
            f"Loại tổn thương không hợp lệ: "
            f"{row['_lesion_type_key']}"
        )

    feature_columns = common_features + specific_features

    return tuple(
        (
            column,
            normalize_pairing_value(row[column])
        )
        for column in feature_columns
    )


pairing_audit_full_df["_descriptor_signature"] = (
    pairing_audit_full_df.apply(
        create_pairing_signature,
        axis=1,
    )
)

# Đếm số đặc trưng bị thiếu hoặc chuỗi rỗng.
pairing_audit_full_df["_so_dac_trung_thieu"] = (
    pairing_audit_full_df["_descriptor_signature"]
    .apply(
        lambda signature: sum(
            value in ["<THIEU>", "<RONG>"]
            for _, value in signature
        )
    )
)

# 6. Xác định khóa nhóm phục vụ đối chiếu

pairing_group_columns = [
    "_patient_key",
    "_laterality_key",
    "_lesion_type_key",
    "_descriptor_signature",
]

# 7. Hàm tổng hợp thông tin theo từng góc chụp

def summarize_pairing_view(data, view):

    view_data = data.loc[
        data["_image_view_key"] == view
    ].copy()

    result = (
        view_data
        .groupby(
            pairing_group_columns,
            dropna=False,
            sort=True,
        )
        .agg(
            so_annotation=(
                "_patient_key",
                "size",
            ),

            danh_sach_abnormality_id=(
                "_abnormality_id_key",
                summarize_unique_values,
            ),

            danh_sach_pathology=(
                "_pathology_key",
                summarize_unique_values,
            ),

            dong_nguon=(
                "source_row_number",
                summarize_unique_values,
            ),

            so_dac_trung_thieu=(
                "_so_dac_trung_thieu",
                "max",
            ),
        )
        .reset_index()
    )

    return result.rename(
        columns={
            "so_annotation": f"so_annotation_{view}",
            "danh_sach_abnormality_id": f"abnormality_id_{view}",
            "danh_sach_pathology": f"pathology_{view}",
            "dong_nguon": f"dong_nguon_{view}",
            "so_dac_trung_thieu": f"so_dac_trung_thieu_{view}",
        }
    )

cc_pairing_summary_df = summarize_pairing_view(
    pairing_audit_full_df,
    "CC",
)

mlo_pairing_summary_df = summarize_pairing_view(
    pairing_audit_full_df,
    "MLO",
)

# 8. Đối chiếu hai góc chụp

full_pairing_audit_df = pd.merge(
    cc_pairing_summary_df,
    mlo_pairing_summary_df,
    on=pairing_group_columns,
    how="outer",
)

# Điền số lượng bằng 0 cho những góc chụp không có dữ liệu.
count_columns = [
    "so_annotation_CC",
    "so_annotation_MLO",
]

for column in count_columns:

    full_pairing_audit_df[column] = (
        full_pairing_audit_df[column]
        .fillna(0)
        .astype(int)
    )

# Điền chuỗi rỗng cho trường thông tin không có dữ liệu ở một góc.
text_columns = [
    "abnormality_id_CC",
    "pathology_CC",
    "dong_nguon_CC",
    "abnormality_id_MLO",
    "pathology_MLO",
    "dong_nguon_MLO",
]

for column in text_columns:

    full_pairing_audit_df[column] = (
        full_pairing_audit_df[column]
        .fillna("")
    )

full_pairing_audit_df["so_annotation"] = (
    full_pairing_audit_df["so_annotation_CC"]
    + full_pairing_audit_df["so_annotation_MLO"]
)

# 9. Phân loại trạng thái ghép cặp

def classify_full_pairing(row):

    cc_count = row["so_annotation_CC"]
    mlo_count = row["so_annotation_MLO"]

    if cc_count == 1 and mlo_count == 1:
        return "Ghep_duy_nhat"

    elif cc_count > 0 and mlo_count > 0:
        return "Ghep_chua_duy_nhat"

    elif cc_count > 0:
        return "Chi_CC"

    else:
        return "Chi_MLO"

full_pairing_audit_df["trang_thai_ghep"] = (
    full_pairing_audit_df.apply(
        classify_full_pairing,
        axis=1,
    )
)

# 10. Kiểm tra sự tương ứng của abnormality_id và pathology

def classify_id_relation(row):

    if row["trang_thai_ghep"] != "Ghep_duy_nhat":
        return "Chua_xac_dinh"

    if (
        row["abnormality_id_CC"]
        == row["abnormality_id_MLO"]
    ):
        return "Cung_abnormality_id"

    return "Khac_abnormality_id"

def classify_pathology_relation(row):

    if row["trang_thai_ghep"] != "Ghep_duy_nhat":
        return "Chua_danh_gia"

    if (
        row["pathology_CC"]
        == row["pathology_MLO"]
    ):
        return "Nhat_quan"

    return "Bat_nhat"

full_pairing_audit_df["quan_he_abnormality_id"] = (
    full_pairing_audit_df.apply(
        classify_id_relation,
        axis=1,
    )
)

full_pairing_audit_df["tinh_nhat_quan_pathology"] = (
    full_pairing_audit_df.apply(
        classify_pathology_relation,
        axis=1,
    )
)

# 11. Kiểm tra mức độ đầy đủ của đặc trưng

full_pairing_audit_df["so_dac_trung_thieu"] = (
    full_pairing_audit_df[
        [
            "so_dac_trung_thieu_CC",
            "so_dac_trung_thieu_MLO",
        ]
    ]
    .fillna(0)
    .max(axis=1)
    .astype(int)
)

full_pairing_audit_df["trang_thai_dac_trung"] = (
    full_pairing_audit_df["so_dac_trung_thieu"]
    .apply(
        lambda count:
            "Day_du_4_dac_trung"
            if count == 0
            else f"Thieu_{count}_dac_trung"
    )
)

# 12. Tạo nhãn mô tả chữ ký đặc trưng

full_pairing_audit_df["dac_trung"] = (
    full_pairing_audit_df["_descriptor_signature"]
    .apply(
        lambda signature: " | ".join(
            f"{column}={value}"
            for column, value in signature
        )
    )
)

# 13. Sắp xếp kết quả

full_pairing_audit_df = (
    full_pairing_audit_df
    .sort_values(
        by=[
            "_patient_key",
            "_laterality_key",
            "_lesion_type_key",
            "trang_thai_ghep",
        ]
    )
    .reset_index(drop=True)
)

# 14. Kiểm tra bảo toàn số lượng annotation

total_annotations_after_pairing = int(
    full_pairing_audit_df["so_annotation"].sum()
)

print("\n" + "=" * 80)
print("KIỂM TRA BẢO TOÀN SỐ LƯỢNG ANNOTATION")
print("=" * 80)

print(
    f"Tổng annotation ban đầu: "
    f"{len(pairing_audit_full_df):,}"
)

print(
    f"Tổng annotation sau đối chiếu: "
    f"{total_annotations_after_pairing:,}"
)

if total_annotations_after_pairing != len(pairing_audit_full_df):
    raise ValueError(
        "Số lượng annotation sau đối chiếu không khớp dữ liệu đầu vào."
    )

print("[ĐẠT] Bảo toàn toàn bộ annotation.")

# 15. Thống kê trạng thái ghép cặp

pairing_status_summary_full_df = (
    full_pairing_audit_df
    .groupby(
        "trang_thai_ghep",
        dropna=False,
    )
    .agg(
        so_chu_ky_dac_trung=(
            "trang_thai_ghep",
            "size",
        ),
        so_annotation=(
            "so_annotation",
            "sum",
        ),
    )
    .reset_index()
)

print("\n" + "=" * 80)
print("PHÂN BỐ TRẠNG THÁI GHÉP CẶP TRÊN TOÀN BỘ DỮ LIỆU")
print("=" * 80)

display(pairing_status_summary_full_df)

# 16. Thống kê quan hệ abnormality_id trong các cặp ghép duy nhất
unique_pair_df = full_pairing_audit_df.loc[
    full_pairing_audit_df["trang_thai_ghep"]
    == "Ghep_duy_nhat"
].copy()

id_relation_summary_df = (
    unique_pair_df["quan_he_abnormality_id"]
    .value_counts()
    .rename_axis("quan_he_abnormality_id")
    .reset_index(name="so_cap")
)

print("\n" + "=" * 80)
print("QUAN HỆ ABNORMALITY ID TRONG CÁC CẶP GHÉP DUY NHẤT")
print("=" * 80)

display(id_relation_summary_df)

# 17. Thống kê tính nhất quán pathology

pathology_relation_summary_df = (
    unique_pair_df["tinh_nhat_quan_pathology"]
    .value_counts()
    .rename_axis("tinh_nhat_quan_pathology")
    .reset_index(name="so_cap")
)

print("\n" + "=" * 80)
print("TÍNH NHẤT QUÁN PATHOLOGY TRONG CÁC CẶP GHÉP DUY NHẤT")
print("=" * 80)

display(pathology_relation_summary_df)

# 18. Các cặp ghép duy nhất có abnormality_id khác nhau

different_id_pairs_df = unique_pair_df.loc[
    unique_pair_df["quan_he_abnormality_id"]
    == "Khac_abnormality_id"
].copy()

print("\n" + "=" * 80)
print("CÁC CẶP GHÉP DUY NHẤT CÓ ABNORMALITY ID KHÁC NHAU")
print("=" * 80)

print(
    f"Số cặp: {len(different_id_pairs_df):,}"
)

display(
    different_id_pairs_df[
        [
            "_patient_key",
            "_laterality_key",
            "_lesion_type_key",
            "abnormality_id_CC",
            "abnormality_id_MLO",
            "pathology_CC",
            "pathology_MLO",
            "dac_trung",
        ]
    ].head(30)
)

# 19. Các cặp ghép duy nhất có pathology bất nhất

different_pathology_pairs_df = unique_pair_df.loc[
    unique_pair_df["tinh_nhat_quan_pathology"]
    == "Bat_nhat"
].copy()

print("\n" + "=" * 80)
print("CÁC CẶP GHÉP DUY NHẤT CÓ PATHOLOGY BẤT NHẤT")
print("=" * 80)

print(f"Số cặp: {len(different_pathology_pairs_df):,}")

display(
    different_pathology_pairs_df[
        [
            "_patient_key",
            "_laterality_key",
            "_lesion_type_key",
            "abnormality_id_CC",
            "abnormality_id_MLO",
            "pathology_CC",
            "pathology_MLO",
            "dac_trung",
            "trang_thai_dac_trung",
        ]
    ].head(30)
)

# 20. Kiểm tra các trường hợp ghép chưa duy nhất

ambiguous_pairing_df = full_pairing_audit_df.loc[
    full_pairing_audit_df["trang_thai_ghep"]
    == "Ghep_chua_duy_nhat"
].copy()

print("\n" + "=" * 80)
print("CÁC TRƯỜNG HỢP GHÉP CHƯA DUY NHẤT")
print("=" * 80)

print(
    f"Số chữ ký đặc trưng chưa ghép duy nhất: "
    f"{len(ambiguous_pairing_df):,}"
)

display(
    ambiguous_pairing_df[
        [
            "_patient_key",
            "_laterality_key",
            "_lesion_type_key",
            "so_annotation_CC",
            "so_annotation_MLO",
            "abnormality_id_CC",
            "abnormality_id_MLO",
            "pathology_CC",
            "pathology_MLO",
            "dac_trung",
        ]
    ].head(30)
)

KIỂM TRA DỮ LIỆU ĐẦU VÀO
Tổng số annotation: 3,568

KIỂM TRA BẢO TOÀN SỐ LƯỢNG ANNOTATION
Tổng annotation ban đầu: 3,568
Tổng annotation sau đối chiếu: 3,568
[ĐẠT] Bảo toàn toàn bộ annotation.

PHÂN BỐ TRẠNG THÁI GHÉP CẶP TRÊN TOÀN BỘ DỮ LIỆU


,trang_thai_ghep,so_chu_ky_dac_trung,so_annotation
0,Chi_CC,432,445
1,Chi_MLO,626,646
2,Ghep_chua_duy_nhat,115,565
3,Ghep_duy_nhat,956,1912



QUAN HỆ ABNORMALITY ID TRONG CÁC CẶP GHÉP DUY NHẤT


,quan_he_abnormality_id,so_cap
0,Cung_abnormality_id,924
1,Khac_abnormality_id,32



TÍNH NHẤT QUÁN PATHOLOGY TRONG CÁC CẶP GHÉP DUY NHẤT


,tinh_nhat_quan_pathology,so_cap
0,Nhat_quan,954
1,Bat_nhat,2


... (2 outputs left out)


CÁC CẶP GHÉP DUY NHẤT CÓ PATHOLOGY BẤT NHẤT
Số cặp: 2


,_patient_key,_laterality_key,_lesion_type_key,abnormality_id_CC,abnormality_id_MLO,pathology_CC,pathology_MLO,dac_trung,trang_thai_dac_trung
1326,P_01200,LEFT,calcification,1,1,BENIGN_WITHOUT_CALLBACK,BENIGN,assessment=2 | subtlety=5 | calc_type=COARSE | calc_distribution=<THIEU>,Thieu_1_dac_trung
1777,P_01623,RIGHT,mass,1,1,MALIGNANT,BENIGN,assessment=5 | subtlety=2 | mass_shape=IRREGULAR | mass_margins=SPICULATED,Day_du_4_dac_trung



CÁC TRƯỜNG HỢP GHÉP CHƯA DUY NHẤT
Số chữ ký đặc trưng chưa ghép duy nhất: 115


,_patient_key,_laterality_key,_lesion_type_key,so_annotation_CC,so_annotation_MLO,abnormality_id_CC,abnormality_id_MLO,pathology_CC,pathology_MLO,dac_trung
5,P_00008,LEFT,calcification,3,3,1 | 2 | 3,1 | 2 | 3,BENIGN_WITHOUT_CALLBACK,BENIGN_WITHOUT_CALLBACK,assessment=2 | subtlety=3 | calc_type=<THIEU> | calc_distribution=REGIONAL
6,P_00008,RIGHT,calcification,5,5,1 | 2 | 3 | 4 | 5,1 | 2 | 3 | 4 | 5,BENIGN_WITHOUT_CALLBACK,BENIGN_WITHOUT_CALLBACK,assessment=2 | subtlety=3 | calc_type=<THIEU> | calc_distribution=REGIONAL
43,P_00038,RIGHT,calcification,2,2,1 | 2,1 | 2,BENIGN_WITHOUT_CALLBACK,BENIGN_WITHOUT_CALLBACK,assessment=2 | subtlety=5 | calc_type=VASCULAR | calc_distribution=<THIEU>
49,P_00044,RIGHT,mass,3,1,1 | 2 | 3,2,BENIGN,BENIGN,assessment=3 | subtlety=5 | mass_shape=OVAL | mass_margins=CIRCUMSCRIBED
53,P_00047,LEFT,calcification,3,3,1 | 2 | 3,1 | 2 | 3,MALIGNANT,MALIGNANT,assessment=4 | subtlety=3 | calc_type=FINE_LINEAR_BRANCHING | calc_distribution=CLUSTERED-LINEAR
54,P_00048,RIGHT,calcification,2,2,1 | 2,1 | 2,BENIGN_WITHOUT_CALLBACK,BENIGN_WITHOUT_CALLBACK,assessment=2 | subtlety=5 | calc_type=LARGE_RODLIKE | calc_distribution=SEGMENTAL
98,P_00092,RIGHT,mass,1,2,1,1 | 2,MALIGNANT,MALIGNANT,assessment=4 | subtlety=5 | mass_shape=ROUND | mass_margins=CIRCUMSCRIBED
104,P_00098,LEFT,calcification,1,2,1,1 | 2,BENIGN_WITHOUT_CALLBACK,BENIGN_WITHOUT_CALLBACK,assessment=2 | subtlety=5 | calc_type=PUNCTATE-LUCENT_CENTER | calc_distribution=REGIONAL
111,P_00106,LEFT,mass,3,5,1 | 2 | 3,1 | 2 | 3 | 4 | 5,BENIGN_WITHOUT_CALLBACK,BENIGN_WITHOUT_CALLBACK,assessment=2 | subtlety=4 | mass_shape=LYMPH_NODE | mass_margins=CIRCUMSCRIBED
113,P_00106,RIGHT,mass,3,3,2 | 3 | 4,1 | 2 | 3,BENIGN_WITHOUT_CALLBACK,BENIGN_WITHOUT_CALLBACK,assessment=2 | subtlety=4 | mass_shape=LYMPH_NODE | mass_margins=CIRCUMSCRIBED


### 3.6. Đánh giá mức độ tin cậy và phân loại các trường hợp liên kết

Sau khi kiểm tra toàn bộ dữ liệu, các chữ ký đặc trưng được phân loại theo khả năng liên kết giữa hai góc chụp CC và MLO.

Việc phân loại dựa trên các yếu tố:

- Số lượng annotation ở mỗi góc chụp.
- Mức độ đầy đủ của các đặc trưng tổn thương.
- Sự tương ứng của `abnormality_id`.
- Tính nhất quán của `pathology`.

Các nhóm có đúng một annotation CC và một annotation MLO được tiếp tục kiểm tra theo từng yếu tố trên.

Các nhóm có nhiều annotation cùng chữ ký ở một hoặc cả hai góc được giữ ở trạng thái chưa xác định. Những nhóm chỉ xuất hiện ở một góc chụp cũng được phân biệt giữa trường hợp có một annotation và trường hợp có nhiều annotation cùng chữ ký.

In [86]:
# 3.6 — ĐÁNH GIÁ MỨC ĐỘ TIN CẬY VÀ PHÂN LOẠI NGOẠI LỆ

# 1. Kiểm tra dữ liệu đầu vào

if "full_pairing_audit_df" not in globals():
    raise NameError(
        "Không tìm thấy full_pairing_audit_df. "
        "Hãy chạy hoàn tất Phần 03.5 trước."
    )

review_group_df = full_pairing_audit_df.copy()

required_review_columns = [
    "trang_thai_ghep",
    "so_annotation_CC",
    "so_annotation_MLO",
    "abnormality_id_CC",
    "abnormality_id_MLO",
    "pathology_CC",
    "pathology_MLO",
    "so_dac_trung_thieu",
]

missing_review_columns = [
    column
    for column in required_review_columns
    if column not in review_group_df.columns
]

if missing_review_columns:
    raise ValueError(
        f"Thiếu cột cần thiết: {missing_review_columns}"
    )

# 2. Kiểm tra trạng thái ghép duy nhất

review_group_df["_la_cap_ghep_duy_nhat"] = (
    review_group_df["trang_thai_ghep"]
    == "Ghep_duy_nhat"
)

review_group_df["_la_cap_ghep_chua_duy_nhat"] = (
    review_group_df["trang_thai_ghep"]
    == "Ghep_chua_duy_nhat"
)

# 3. Tạo các cờ kiểm tra cho cặp ghép duy nhất

review_group_df["_id_nhat_quan"] = (
    review_group_df["abnormality_id_CC"]
    == review_group_df["abnormality_id_MLO"]
).astype("boolean")

review_group_df["_pathology_nhat_quan"] = (
    review_group_df["pathology_CC"]
    == review_group_df["pathology_MLO"]
).astype("boolean")

review_group_df["_dac_trung_day_du"] = (
    review_group_df["so_dac_trung_thieu"] == 0
).astype("boolean")

# Chỉ đánh giá các cờ này khi có đúng một CC và một MLO.
unique_pair_mask = review_group_df["_la_cap_ghep_duy_nhat"]

review_group_df.loc[
    ~unique_pair_mask,
    [
        "_id_nhat_quan",
        "_pathology_nhat_quan",
        "_dac_trung_day_du",
    ],
] = pd.NA

# 4. Phân loại chi tiết tình trạng của từng nhóm

def classify_review_status(row):

    cc_count = row["so_annotation_CC"]
    mlo_count = row["so_annotation_MLO"]

    # Trường hợp có cả hai góc chụp.
    if cc_count > 0 and mlo_count > 0:

        if cc_count != 1 or mlo_count != 1:
            return "Ghep_chua_duy_nhat"

        issues = []

        if row["so_dac_trung_thieu"] > 0:
            issues.append("Thieu_dac_trung")

        if row["abnormality_id_CC"] != row["abnormality_id_MLO"]:
            issues.append("Khac_abnormality_id")

        if row["pathology_CC"] != row["pathology_MLO"]:
            issues.append("Bat_nhat_pathology")

        if not issues:
            return "Cap_ghép_day_du_va_nhat_quan"

        return "Cap_ghép_can_xem_xet__" + "__".join(issues)

    # Chỉ có CC.
    if cc_count > 0:

        if cc_count == 1:
            return "Don_goc_CC_mot_annotation"

        return "Don_goc_CC_nhieu_annotation_cung_chu_ky"

    # Chỉ có MLO.
    if mlo_count == 1:
        return "Don_goc_MLO_mot_annotation"

    return "Don_goc_MLO_nhieu_annotation_cung_chu_ky"


review_group_df["trang_thai_ra_soat"] = (
    review_group_df.apply(
        classify_review_status,
        axis=1,
    )
)

# 5. Tổng hợp trạng thái rà soát

review_status_summary_df = (
    review_group_df
    .groupby(
        "trang_thai_ra_soat",
        dropna=False,
    )
    .agg(
        so_nhom=("trang_thai_ra_soat", "size"),
        so_annotation=("so_annotation", "sum"),
    )
    .reset_index()
    .sort_values(
        by="so_nhom",
        ascending=False,
    )
    .reset_index(drop=True)
)

print("=" * 80)
print("TỔNG HỢP TRẠNG THÁI RÀ SOÁT")
print("=" * 80)

display(review_status_summary_df)


# 6. Kiểm tra mức độ đầy đủ đặc trưng của các cặp ghép duy nhất

unique_pair_review_df = review_group_df.loc[
    review_group_df["_la_cap_ghep_duy_nhat"]
].copy()

feature_completeness_summary_df = (
    unique_pair_review_df
    .groupby(
        "trang_thai_dac_trung",
        dropna=False,
    )
    .agg(
        so_cap=("trang_thai_dac_trung", "size"),
        so_annotation=("so_annotation", "sum"),
        so_cap_pathology_bat_nhat=(
            "tinh_nhat_quan_pathology",
            lambda values: int((values == "Bat_nhat").sum()),
        ),
        so_cap_abnormality_id_khac=(
            "quan_he_abnormality_id",
            lambda values: int((values == "Khac_abnormality_id").sum()),
        ),
    )
    .reset_index()
)

print("\n" + "=" * 80)
print("MỨC ĐỘ ĐẦY ĐỦ ĐẶC TRƯNG TRONG CÁC CẶP GHÉP DUY NHẤT")
print("=" * 80)

display(feature_completeness_summary_df)

# 7. Kiểm tra nhóm chỉ có một góc nhưng có nhiều annotation

single_view_duplicate_signature_df = review_group_df.loc[
    review_group_df["trang_thai_ra_soat"].isin([
        "Don_goc_CC_nhieu_annotation_cung_chu_ky",
        "Don_goc_MLO_nhieu_annotation_cung_chu_ky",
    ])
].copy()

print("\n" + "=" * 80)
print("NHÓM CHỈ CÓ MỘT GÓC NHƯNG NHIỀU ANNOTATION")
print("=" * 80)

print(
    "Số nhóm cần xem xét:",
    f"{len(single_view_duplicate_signature_df):,}"
)

display(
    single_view_duplicate_signature_df[
        [
            "_patient_key",
            "_laterality_key",
            "_lesion_type_key",
            "so_annotation_CC",
            "so_annotation_MLO",
            "abnormality_id_CC",
            "abnormality_id_MLO",
            "pathology_CC",
            "pathology_MLO",
            "dac_trung",
        ]
    ].head(30)
)

# 8. Trích xuất các nhóm ghép chưa duy nhất

ambiguous_review_df = review_group_df.loc[
    review_group_df["trang_thai_ra_soat"]
    == "Ghep_chua_duy_nhat"
].copy()

print("\n" + "=" * 80)
print("CÁC NHÓM CÓ CẢ CC/MLO NHƯNG GHÉP CHƯA DUY NHẤT")
print("=" * 80)

print(
    "Số nhóm:",
    f"{len(ambiguous_review_df):,}"
)

display(
    ambiguous_review_df[
        [
            "_patient_key",
            "_laterality_key",
            "_lesion_type_key",
            "so_annotation_CC",
            "so_annotation_MLO",
            "abnormality_id_CC",
            "abnormality_id_MLO",
            "pathology_CC",
            "pathology_MLO",
            "dac_trung",
        ]
    ].head(30)
)

# 9. Trích xuất cặp ghép duy nhất nhưng có vấn đề

unique_pair_exception_df = review_group_df.loc[
    review_group_df["trang_thai_ra_soat"].str.startswith(
        "Cap_ghép_can_xem_xet",
        na=False,
    )
].copy()

print("\n" + "=" * 80)
print("CÁC CẶP GHÉP DUY NHẤT CẦN XEM XÉT")
print("=" * 80)

print(
    "Số cặp:",
    f"{len(unique_pair_exception_df):,}"
)

display(
    unique_pair_exception_df[
        [
            "_patient_key",
            "_laterality_key",
            "_lesion_type_key",
            "abnormality_id_CC",
            "abnormality_id_MLO",
            "pathology_CC",
            "pathology_MLO",
            "so_dac_trung_thieu",
            "trang_thai_ra_soat",
        ]
    ].head(50)
)

# 10. Kiểm tra bảo toàn số annotation

total_annotations_reviewed = int(
    review_group_df["so_annotation"].sum()
)

print("\n" + "=" * 80)
print("KIỂM TRA BẢO TOÀN DỮ LIỆU")
print("=" * 80)

print(
    f"Tổng annotation ban đầu: "
    f"{len(pairing_audit_full_df):,}"
)

print(
    f"Tổng annotation đã phân loại: "
    f"{total_annotations_reviewed:,}"
)

if total_annotations_reviewed != len(pairing_audit_full_df):
    raise ValueError(
        "Số lượng annotation sau rà soát không khớp."
    )

print("[ĐẠT] Bảo toàn toàn bộ annotation.")

TỔNG HỢP TRẠNG THÁI RÀ SOÁT


,trang_thai_ra_soat,so_nhom,so_annotation
0,Cap_ghép_day_du_va_nhat_quan,850,1700
1,Don_goc_MLO_mot_annotation,611,611
2,Don_goc_CC_mot_annotation,422,422
3,Ghep_chua_duy_nhat,115,565
4,Cap_ghép_can_xem_xet__Thieu_dac_trung,72,144
5,Cap_ghép_can_xem_xet__Khac_abnormality_id,22,44
6,Don_goc_MLO_nhieu_annotation_cung_chu_ky,15,35
7,Don_goc_CC_nhieu_annotation_cung_chu_ky,10,23
8,Cap_ghép_can_xem_xet__Thieu_dac_trung__Khac_abnormality_id,10,20
9,Cap_ghép_can_xem_xet__Bat_nhat_pathology,1,2



MỨC ĐỘ ĐẦY ĐỦ ĐẶC TRƯNG TRONG CÁC CẶP GHÉP DUY NHẤT


,trang_thai_dac_trung,so_cap,so_annotation,so_cap_pathology_bat_nhat,so_cap_abnormality_id_khac
0,Day_du_4_dac_trung,873,1746,1,22
1,Thieu_1_dac_trung,82,164,1,10
2,Thieu_2_dac_trung,1,2,0,0



NHÓM CHỈ CÓ MỘT GÓC NHƯNG NHIỀU ANNOTATION
Số nhóm cần xem xét: 25


,_patient_key,_laterality_key,_lesion_type_key,so_annotation_CC,so_annotation_MLO,abnormality_id_CC,abnormality_id_MLO,pathology_CC,pathology_MLO,dac_trung
334,P_00328,RIGHT,mass,2,0,1 | 2,,MALIGNANT,,assessment=4 | subtlety=3 | mass_shape=ARCHITECTURAL_DISTORTION | mass_margins=ILL_DEFINED-SPICU...
335,P_00328,RIGHT,mass,0,2,,1 | 2,,MALIGNANT,assessment=4 | subtlety=4 | mass_shape=ARCHITECTURAL_DISTORTION | mass_margins=ILL_DEFINED-SPICU...
385,P_00377,LEFT,calcification,2,0,1 | 2,,BENIGN,,assessment=4 | subtlety=4 | calc_type=PLEOMORPHIC | calc_distribution=CLUSTERED
410,P_00394,LEFT,calcification,0,2,,1 | 2,,BENIGN_WITHOUT_CALLBACK,assessment=2 | subtlety=5 | calc_type=LUCENT_CENTER | calc_distribution=<THIEU>
576,P_00528,LEFT,mass,0,2,,1 | 2,,BENIGN,assessment=3 | subtlety=5 | mass_shape=OVAL | mass_margins=CIRCUMSCRIBED
614,P_00558,RIGHT,calcification,0,2,,1 | 2,,BENIGN,assessment=4 | subtlety=3 | calc_type=PUNCTATE-PLEOMORPHIC | calc_distribution=CLUSTERED
630,P_00573,RIGHT,mass,0,2,,1 | 2,,MALIGNANT,assessment=5 | subtlety=5 | mass_shape=IRREGULAR | mass_margins=MICROLOBULATED
754,P_00673,RIGHT,mass,0,3,,1 | 3 | 4,,BENIGN_WITHOUT_CALLBACK,assessment=2 | subtlety=4 | mass_shape=OVAL-LYMPH_NODE | mass_margins=CIRCUMSCRIBED
794,P_00710,LEFT,mass,2,0,1 | 2,,BENIGN,,assessment=4 | subtlety=5 | mass_shape=ROUND-OVAL | mass_margins=CIRCUMSCRIBED-OBSCURED
963,P_00865,RIGHT,mass,0,2,,1 | 2,,MALIGNANT,assessment=4 | subtlety=4 | mass_shape=OVAL | mass_margins=ILL_DEFINED



CÁC NHÓM CÓ CẢ CC/MLO NHƯNG GHÉP CHƯA DUY NHẤT
Số nhóm: 115


,_patient_key,_laterality_key,_lesion_type_key,so_annotation_CC,so_annotation_MLO,abnormality_id_CC,abnormality_id_MLO,pathology_CC,pathology_MLO,dac_trung
5,P_00008,LEFT,calcification,3,3,1 | 2 | 3,1 | 2 | 3,BENIGN_WITHOUT_CALLBACK,BENIGN_WITHOUT_CALLBACK,assessment=2 | subtlety=3 | calc_type=<THIEU> | calc_distribution=REGIONAL
6,P_00008,RIGHT,calcification,5,5,1 | 2 | 3 | 4 | 5,1 | 2 | 3 | 4 | 5,BENIGN_WITHOUT_CALLBACK,BENIGN_WITHOUT_CALLBACK,assessment=2 | subtlety=3 | calc_type=<THIEU> | calc_distribution=REGIONAL
43,P_00038,RIGHT,calcification,2,2,1 | 2,1 | 2,BENIGN_WITHOUT_CALLBACK,BENIGN_WITHOUT_CALLBACK,assessment=2 | subtlety=5 | calc_type=VASCULAR | calc_distribution=<THIEU>
49,P_00044,RIGHT,mass,3,1,1 | 2 | 3,2,BENIGN,BENIGN,assessment=3 | subtlety=5 | mass_shape=OVAL | mass_margins=CIRCUMSCRIBED
53,P_00047,LEFT,calcification,3,3,1 | 2 | 3,1 | 2 | 3,MALIGNANT,MALIGNANT,assessment=4 | subtlety=3 | calc_type=FINE_LINEAR_BRANCHING | calc_distribution=CLUSTERED-LINEAR
54,P_00048,RIGHT,calcification,2,2,1 | 2,1 | 2,BENIGN_WITHOUT_CALLBACK,BENIGN_WITHOUT_CALLBACK,assessment=2 | subtlety=5 | calc_type=LARGE_RODLIKE | calc_distribution=SEGMENTAL
98,P_00092,RIGHT,mass,1,2,1,1 | 2,MALIGNANT,MALIGNANT,assessment=4 | subtlety=5 | mass_shape=ROUND | mass_margins=CIRCUMSCRIBED
104,P_00098,LEFT,calcification,1,2,1,1 | 2,BENIGN_WITHOUT_CALLBACK,BENIGN_WITHOUT_CALLBACK,assessment=2 | subtlety=5 | calc_type=PUNCTATE-LUCENT_CENTER | calc_distribution=REGIONAL
111,P_00106,LEFT,mass,3,5,1 | 2 | 3,1 | 2 | 3 | 4 | 5,BENIGN_WITHOUT_CALLBACK,BENIGN_WITHOUT_CALLBACK,assessment=2 | subtlety=4 | mass_shape=LYMPH_NODE | mass_margins=CIRCUMSCRIBED
113,P_00106,RIGHT,mass,3,3,2 | 3 | 4,1 | 2 | 3,BENIGN_WITHOUT_CALLBACK,BENIGN_WITHOUT_CALLBACK,assessment=2 | subtlety=4 | mass_shape=LYMPH_NODE | mass_margins=CIRCUMSCRIBED



CÁC CẶP GHÉP DUY NHẤT CẦN XEM XÉT
Số cặp: 106


,_patient_key,_laterality_key,_lesion_type_key,abnormality_id_CC,abnormality_id_MLO,pathology_CC,pathology_MLO,so_dac_trung_thieu,trang_thai_ra_soat
33,P_00030,LEFT,calcification,1,1,BENIGN_WITHOUT_CALLBACK,BENIGN_WITHOUT_CALLBACK,1,Cap_ghép_can_xem_xet__Thieu_dac_trung
34,P_00030,LEFT,calcification,2,2,BENIGN_WITHOUT_CALLBACK,BENIGN_WITHOUT_CALLBACK,1,Cap_ghép_can_xem_xet__Thieu_dac_trung
35,P_00030,RIGHT,calcification,1,1,BENIGN_WITHOUT_CALLBACK,BENIGN_WITHOUT_CALLBACK,1,Cap_ghép_can_xem_xet__Thieu_dac_trung
45,P_00041,LEFT,calcification,2,2,BENIGN_WITHOUT_CALLBACK,BENIGN_WITHOUT_CALLBACK,1,Cap_ghép_can_xem_xet__Thieu_dac_trung
50,P_00044,RIGHT,mass,4,1,BENIGN,BENIGN,0,Cap_ghép_can_xem_xet__Khac_abnormality_id
76,P_00077,LEFT,calcification,1,1,BENIGN_WITHOUT_CALLBACK,BENIGN_WITHOUT_CALLBACK,1,Cap_ghép_can_xem_xet__Thieu_dac_trung
77,P_00077,RIGHT,calcification,1,2,BENIGN_WITHOUT_CALLBACK,BENIGN_WITHOUT_CALLBACK,1,Cap_ghép_can_xem_xet__Thieu_dac_trung__Khac_abnormality_id
78,P_00077,RIGHT,calcification,2,1,BENIGN_WITHOUT_CALLBACK,BENIGN_WITHOUT_CALLBACK,1,Cap_ghép_can_xem_xet__Thieu_dac_trung__Khac_abnormality_id
80,P_00079,RIGHT,mass,1,1,MALIGNANT,MALIGNANT,1,Cap_ghép_can_xem_xet__Thieu_dac_trung
106,P_00098,RIGHT,calcification,1,2,BENIGN_WITHOUT_CALLBACK,BENIGN_WITHOUT_CALLBACK,0,Cap_ghép_can_xem_xet__Khac_abnormality_id



KIỂM TRA BẢO TOÀN DỮ LIỆU
Tổng annotation ban đầu: 3,568
Tổng annotation đã phân loại: 3,568
[ĐẠT] Bảo toàn toàn bộ annotation.


Trong 106 cặp cần xem xét:
- 22 cặp có abnormality_id khác nhau nhưng đặc trưng đầy đủ.
- 72 cặp thiếu một đặc trưng.
- 10 cặp vừa thiếu đặc trưng vừa khác abnormality_id.
- 2 cặp pathology bất nhất, trong đó một cặp còn thiếu đặc trưng.

Ngoài ra, 140 nhóm chưa xác định gồm 115 nhóm có cả CC/MLO nhưng ghép chưa duy nhất và 25 nhóm chỉ có một góc nhưng có nhiều annotation cùng chữ ký.

### 3.7. Xây dựng bảng dữ liệu tổn thương ứng viên

Quy trình sử dụng kết quả rà soát ở Phần 03.6 để phân chia dữ liệu thành:

1. Bảng tổn thương ứng viên: gồm các nhóm có thể xác định được một ứng viên ghép hoặc một annotation đơn góc.
2. Bảng nhóm cần rà soát: gồm các trường hợp có liên kết chưa duy nhất.
3. Bảng ánh xạ annotation: bảo toàn quan hệ giữa từng bản ghi nguồn và nhóm tương ứng.

Các trường hợp pathology bất nhất, thiếu đặc trưng hoặc có `abnormality_id` khác nhau vẫn được giữ lại cùng thông tin kiểm soát.

In [87]:
# 3.7.2 — TẠO MÃ TỔN THƯƠNG ỨNG VIÊN VÀ MÃ NHÓM RÀ SOÁT

# 1. Kiểm tra dữ liệu đầu vào

required_objects = [
    "review_group_df",
    "pairing_audit_full_df",
]

for object_name in required_objects:

    if object_name not in globals():
        raise NameError(
            f"Không tìm thấy biến {object_name}. "
            "Hãy chạy hoàn tất Phần 03.5 và 03.6."
        )

required_group_columns = [
    "_patient_key",
    "_laterality_key",
    "_lesion_type_key",
    "_descriptor_signature",
    "trang_thai_ghep",
    "trang_thai_ra_soat",
    "so_annotation",
    "so_annotation_CC",
    "so_annotation_MLO",
    "so_dac_trung_thieu",
    "pathology_CC",
    "pathology_MLO",
    "abnormality_id_CC",
    "abnormality_id_MLO",
]

missing_columns = [
    column
    for column in required_group_columns
    if column not in review_group_df.columns
]

if missing_columns:
    raise ValueError(
        f"Thiếu cột trong bảng rà soát: {missing_columns}"
    )

# 2. Tạo bản sao dữ liệu
lesion_group_build_df = review_group_df.copy()

# 3. Xác định nhóm đủ điều kiện tạo ứng viên
single_view_statuses = [
    "Don_goc_CC_mot_annotation",
    "Don_goc_MLO_mot_annotation",
]

is_unique_pair = (
    lesion_group_build_df["trang_thai_ghep"]
    == "Ghep_duy_nhat"
)

is_single_view_candidate = (
    lesion_group_build_df["trang_thai_ra_soat"]
    .isin(single_view_statuses)
)

candidate_mask = (
    is_unique_pair | is_single_view_candidate
)

lesion_group_build_df["loai_nhom"] = "Chua_xac_dinh"

lesion_group_build_df.loc[
    is_unique_pair,
    "loai_nhom",
] = "Cap_CC_MLO"

lesion_group_build_df.loc[
    is_single_view_candidate,
    "loai_nhom",
] = "Don_goc"

# Tách hai bảng độc lập.
lesion_master_df = lesion_group_build_df.loc[
    candidate_mask
].copy()

lesion_review_groups_df = lesion_group_build_df.loc[
    ~candidate_mask
].copy()

# 4. Tạo khóa sắp xếp ổn định

def signature_to_text(signature):

    return " | ".join(
        f"{column}={value}"
        for column, value in signature
    )


for dataframe in [
    lesion_master_df,
    lesion_review_groups_df,
]:

    dataframe["_signature_sort_key"] = (
        dataframe["_descriptor_signature"]
        .apply(signature_to_text)
    )

    dataframe.sort_values(
        by=[
            "_patient_key",
            "_laterality_key",
            "_lesion_type_key",
            "_signature_sort_key",
        ],
        inplace=True,
    )

    dataframe.reset_index(drop=True, inplace=True)

# 5. Cấp mã tổn thương ứng viên

lesion_master_df.insert(
    0,
    "lesion_candidate_id",
    [
        f"LCAND_{number:06d}"
        for number in range(1, len(lesion_master_df) + 1)
    ],
)

lesion_review_groups_df.insert(
    0,
    "review_group_id",
    [
        f"LREV_{number:06d}"
        for number in range(1, len(lesion_review_groups_df) + 1)
    ],
)

# 6. Phân loại ứng viên theo trạng thái

def classify_candidate_type(row):
    if row["loai_nhom"] == "Cap_CC_MLO":
        if row["trang_thai_ra_soat"] == ("Cap_ghép_day_du_va_nhat_quan"):
            return "Cap_CC_MLO_nhat_quan"
        return "Cap_CC_MLO_can_ra_soat"
    return "Ung_vien_don_annotation"

lesion_master_df["candidate_status"] = (
    lesion_master_df.apply(
        classify_candidate_type,
        axis=1,
    )
)

# 7. Tổng hợp pathology nhưng không tự sửa nhãn

def get_pathology_consensus(row):

    cc_pathology = str(
        row["pathology_CC"]
    ).strip()

    mlo_pathology = str(
        row["pathology_MLO"]
    ).strip()

    if cc_pathology and mlo_pathology:

        if cc_pathology == mlo_pathology:
            return cc_pathology

        return pd.NA

    return cc_pathology or mlo_pathology or pd.NA

def classify_pathology_status(row):

    cc_pathology = str(
        row["pathology_CC"]
    ).strip()

    mlo_pathology = str(
        row["pathology_MLO"]
    ).strip()

    if cc_pathology and mlo_pathology:

        if cc_pathology == mlo_pathology:
            return "Nhat_quan"

        return "Bat_nhat"

    if cc_pathology or mlo_pathology:
        return "Chi_co_mot_goc"

    return "Thieu_nhan"

lesion_master_df["pathology_consensus"] = (
    lesion_master_df.apply(
        get_pathology_consensus,
        axis=1,
    )
)

lesion_master_df["pathology_status"] = (
    lesion_master_df.apply(
        classify_pathology_status,
        axis=1,
    )
)

# 8. Phân loại quan hệ abnormality_id

def classify_abnormality_relation(row):

    cc_id = str(row["abnormality_id_CC"]).strip()
    mlo_id = str(row["abnormality_id_MLO"]).strip()

    if cc_id and mlo_id:

        if cc_id == mlo_id:
            return "Cung_abnormality_id"

        return "Khac_abnormality_id"

    if cc_id:
        return "Chi_CC"

    if mlo_id:
        return "Chi_MLO"

    return "Thieu_abnormality_id"


lesion_master_df["abnormality_id_relation"] = (
    lesion_master_df.apply(
        classify_abnormality_relation,
        axis=1,
    )
)

# 9. Đánh dấu tình trạng cần rà soát

lesion_master_df["needs_review"] = (
    lesion_master_df["candidate_status"]
    != "Cap_CC_MLO_nhat_quan"
)

lesion_master_df["is_verified_lesion"] = False

# 10. Kiểm tra nhóm có source_split nhất quán

source_group_meta_df = (
    pairing_audit_full_df
    .groupby(
        [
            "_patient_key",
            "_laterality_key",
            "_lesion_type_key",
            "_descriptor_signature",
        ],
        dropna=False,
    )
    .agg(
        so_source_split=("source_split", "nunique"),
        danh_sach_source_split=(
            "source_split",
            summarize_unique_values,
        ),
        so_bang_nguon=("source_table", "nunique"),
        danh_sach_bang_nguon=(
            "source_table",
            summarize_unique_values,
        ),
    )
    .reset_index()
)

merge_keys = [
    "_patient_key",
    "_laterality_key",
    "_lesion_type_key",
    "_descriptor_signature",
]

lesion_master_df = lesion_master_df.merge(
    source_group_meta_df,
    on=merge_keys,
    how="left",
    validate="one_to_one",
)

lesion_review_groups_df = lesion_review_groups_df.merge(
    source_group_meta_df,
    on=merge_keys,
    how="left",
    validate="one_to_one",
)

# 11. Xác định trạng thái tập nguồn

for dataframe in [
    lesion_master_df,
    lesion_review_groups_df,
]:

    dataframe["source_split_status"] = (
        dataframe["so_source_split"]
        .map({
            1: "Nhat_quan",
        })
        .fillna("Can_ra_soat")
    )

# 12. Tổng hợp thông tin đặc trưng

for dataframe in [
    lesion_master_df,
    lesion_review_groups_df,
]:

    dataframe["descriptor_summary"] = (
        dataframe["_descriptor_signature"]
        .apply(signature_to_text)
    )

# 13. Kiểm tra số lượng nhóm và annotation

print("=" * 80)
print("KẾT QUẢ XÂY DỰNG NHÓM TỔN THƯƠNG ỨNG VIÊN")
print("=" * 80)

print(
    f"Số tổn thương ứng viên: "
    f"{len(lesion_master_df):,}"
)

print(
    f"Số nhóm cần rà soát riêng: "
    f"{len(lesion_review_groups_df):,}"
)

print(
    f"Tổng số nhóm trước phân loại: "
    f"{len(review_group_df):,}"
)

print(
    f"Tổng annotation trong bảng ứng viên: "
    f"{lesion_master_df['so_annotation'].sum():,}"
)

print(
    f"Tổng annotation trong nhóm rà soát: "
    f"{lesion_review_groups_df['so_annotation'].sum():,}"
)

total_grouped_annotations = int(
    lesion_master_df["so_annotation"].sum()
    + lesion_review_groups_df["so_annotation"].sum()
)

print(
    f"Tổng annotation được bảo toàn: "
    f"{total_grouped_annotations:,}"
)

if total_grouped_annotations != len(pairing_audit_full_df):
    raise ValueError(
        "Số annotation sau khi phân nhóm không khớp dữ liệu nguồn."
    )

print("[ĐẠT] Bảo toàn số lượng annotation.")

# 14. Hiển thị phân bố trạng thái ứng viên

print("\n" + "=" * 80)
print("PHÂN BỐ TRẠNG THÁI ỨNG VIÊN")
print("=" * 80)

display(
    lesion_master_df["candidate_status"]
    .value_counts()
    .rename_axis("candidate_status")
    .reset_index(name="so_ung_vien")
)

# 15. Hiển thị trạng thái pathology

print("\n" + "=" * 80)
print("PHÂN BỐ TRẠNG THÁI PATHOLOGY")
print("=" * 80)

display(
    lesion_master_df["pathology_status"]
    .value_counts(dropna=False)
    .rename_axis("pathology_status")
    .reset_index(name="so_ung_vien")
)

# 16. Xem trước bảng tổn thương ứng viên

master_preview_columns = [
    "lesion_candidate_id",
    "_patient_key",
    "_laterality_key",
    "_lesion_type_key",
    "candidate_status",
    "so_annotation",
    "so_annotation_CC",
    "so_annotation_MLO",
    "abnormality_id_CC",
    "abnormality_id_MLO",
    "pathology_CC",
    "pathology_MLO",
    "pathology_consensus",
    "pathology_status",
    "needs_review",
]

print("\n" + "=" * 80)
print("XEM TRƯỚC BẢNG TỔN THƯƠNG ỨNG VIÊN")
print("=" * 80)

display(
    lesion_master_df[
        master_preview_columns
    ].head(20)
)

C:\Users\HOANG YEN\AppData\Local\Temp\ipykernel_15864\2192387649.py:110: Pandas4Warning: The inplace keyword in DataFrame.sort_values is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dataframe.sort_values(
C:\Users\HOANG YEN\AppData\Local\Temp\ipykernel_15864\2192387649.py:120: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dataframe.reset_index(drop=True, inplace=True)


KẾT QUẢ XÂY DỰNG NHÓM TỔN THƯƠNG ỨNG VIÊN
Số tổn thương ứng viên: 1,989
Số nhóm cần rà soát riêng: 140
Tổng số nhóm trước phân loại: 2,129
Tổng annotation trong bảng ứng viên: 2,945
Tổng annotation trong nhóm rà soát: 623
Tổng annotation được bảo toàn: 3,568
[ĐẠT] Bảo toàn số lượng annotation.

PHÂN BỐ TRẠNG THÁI ỨNG VIÊN


,candidate_status,so_ung_vien
0,Ung_vien_don_annotation,1033
1,Cap_CC_MLO_nhat_quan,850
2,Cap_CC_MLO_can_ra_soat,106



PHÂN BỐ TRẠNG THÁI PATHOLOGY


,pathology_status,so_ung_vien
0,Chi_co_mot_goc,1033
1,Nhat_quan,954
2,Bat_nhat,2



XEM TRƯỚC BẢNG TỔN THƯƠNG ỨNG VIÊN


,lesion_candidate_id,_patient_key,_laterality_key,_lesion_type_key,candidate_status,so_annotation,so_annotation_CC,so_annotation_MLO,abnormality_id_CC,abnormality_id_MLO,pathology_CC,pathology_MLO,pathology_consensus,pathology_status,needs_review
0,LCAND_000001,P_00001,LEFT,mass,Cap_CC_MLO_nhat_quan,2,1,1,1,1,MALIGNANT,MALIGNANT,MALIGNANT,Nhat_quan,False
1,LCAND_000002,P_00004,LEFT,mass,Cap_CC_MLO_nhat_quan,2,1,1,1,1,BENIGN,BENIGN,BENIGN,Nhat_quan,False
2,LCAND_000003,P_00004,RIGHT,mass,Ung_vien_don_annotation,1,0,1,,1,,BENIGN,BENIGN,Chi_co_mot_goc,True
3,LCAND_000004,P_00005,RIGHT,calcification,Cap_CC_MLO_nhat_quan,2,1,1,1,1,MALIGNANT,MALIGNANT,MALIGNANT,Nhat_quan,False
4,LCAND_000005,P_00007,LEFT,calcification,Cap_CC_MLO_nhat_quan,2,1,1,1,1,BENIGN,BENIGN,BENIGN,Nhat_quan,False
5,LCAND_000006,P_00009,RIGHT,mass,Cap_CC_MLO_nhat_quan,2,1,1,1,1,MALIGNANT,MALIGNANT,MALIGNANT,Nhat_quan,False
6,LCAND_000007,P_00010,LEFT,calcification,Cap_CC_MLO_nhat_quan,2,1,1,1,1,BENIGN_WITHOUT_CALLBACK,BENIGN_WITHOUT_CALLBACK,BENIGN_WITHOUT_CALLBACK,Nhat_quan,False
7,LCAND_000008,P_00011,LEFT,calcification,Cap_CC_MLO_nhat_quan,2,1,1,1,1,BENIGN,BENIGN,BENIGN,Nhat_quan,False
8,LCAND_000009,P_00012,LEFT,calcification,Ung_vien_don_annotation,1,0,1,,1,,MALIGNANT,MALIGNANT,Chi_co_mot_goc,True
9,LCAND_000010,P_00012,LEFT,calcification,Ung_vien_don_annotation,1,1,0,1,,MALIGNANT,,MALIGNANT,Chi_co_mot_goc,True


### 3.7.3. Xây dựng bảng ánh xạ annotation

Xây dựng bảng liên kết từng annotation nguồn với nhóm dữ liệu tương ứng.

Mỗi annotation chỉ được liên kết với một trong hai đối tượng:

- Một tổn thương ứng viên đã được tạo mã.
- Một nhóm cần rà soát riêng.

Các trường đường dẫn ảnh và ROI được giữ lại để phục vụ việc xây dựng bảng ánh xạ ảnh và ROI ở các bước tiếp theo.

Kiểm tra tính duy nhất của mã annotation, tính toàn vẹn của quan hệ liên kết và khả năng truy xuất toàn bộ 3.568 bản ghi nguồn.

In [88]:
# 3.7.3 — XÂY DỰNG BẢNG ÁNH XẠ ANNOTATION

# 1. Tạo bảng mã liên kết từ hai nhóm dữ liệu

candidate_assignment_df = lesion_master_df[
    merge_keys + ["lesion_candidate_id"]
].copy()

candidate_assignment_df["review_group_id"] = pd.NA

review_assignment_df = lesion_review_groups_df[
    merge_keys + ["review_group_id"]
].copy()

review_assignment_df["lesion_candidate_id"] = pd.NA

assignment_df = pd.concat(
    [
        candidate_assignment_df,
        review_assignment_df,
    ],
    ignore_index=True,
)

# Kiểm tra mỗi chữ ký chỉ có một mã liên kết.
if assignment_df.duplicated(merge_keys).any():
    raise ValueError(
        "Một chữ ký đặc trưng được gán nhiều mã liên kết."
    )

# 2. Ánh xạ các annotation nguồn tới mã tương ứng

lesion_annotation_mapping_df = pairing_audit_full_df.merge(
    assignment_df[
        merge_keys
        + [
            "lesion_candidate_id",
            "review_group_id",
        ]
    ],
    on=merge_keys,
    how="left",
    validate="many_to_one",
)

# 3. Tạo mã annotation có thể truy xuất về nguồn

lesion_annotation_mapping_df["annotation_id"] = (
    lesion_annotation_mapping_df["source_table"].astype("string")
    + "__"
    + lesion_annotation_mapping_df["source_row_number"]
    .astype("string")
    .str.zfill(5)
)

# 4. Kiểm tra mã annotation

if lesion_annotation_mapping_df["annotation_id"].duplicated().any():
    duplicated_annotation_ids_df = (
        lesion_annotation_mapping_df.loc[
            lesion_annotation_mapping_df["annotation_id"].duplicated(
                keep=False
            )
        ]
        .sort_values("annotation_id")
    )

    display(duplicated_annotation_ids_df)

    raise ValueError(
        "Phát hiện annotation_id bị trùng."
    )

# 5. Kiểm tra mỗi annotation chỉ thuộc một loại liên kết

has_candidate = (
    lesion_annotation_mapping_df["lesion_candidate_id"].notna()
)

has_review_group = (
    lesion_annotation_mapping_df["review_group_id"].notna()
)

invalid_assignment_mask = (
    has_candidate == has_review_group
)

if invalid_assignment_mask.any():

    display(
        lesion_annotation_mapping_df.loc[
            invalid_assignment_mask,
            [
                "annotation_id",
                "source_table",
                "source_row_number",
                "lesion_candidate_id",
                "review_group_id",
            ],
        ].head(20)
    )

    raise ValueError(
        "Có annotation chưa được gán hoặc được gán đồng thời "
        "vào hai loại nhóm."
    )

# 6. Phân loại trạng thái liên kết annotation

lesion_annotation_mapping_df["annotation_link_status"] = (
    has_candidate.map({
        True: "Co_ma_ung_vien",
        False: "Thuoc_nhom_can_ra_soat",
    })
)

# 7. Kiểm tra số lượng annotation được ánh xạ

expected_annotation_count = len(annotation_master_df)

actual_annotation_count = len(
    lesion_annotation_mapping_df
)

unique_annotation_count = (
    lesion_annotation_mapping_df["annotation_id"].nunique()
)

candidate_annotation_count = int(
    has_candidate.sum()
)

review_annotation_count = int(
    has_review_group.sum()
)

print("\n" + "=" * 80)
print("KIỂM TRA TÍNH TOÀN VẸN BẢNG ÁNH XẠ")
print("=" * 80)

print(
    f"Số annotation nguồn: "
    f"{expected_annotation_count:,}"
)

print(
    f"Số dòng ánh xạ: "
    f"{actual_annotation_count:,}"
)

print(
    f"Số annotation_id duy nhất: "
    f"{unique_annotation_count:,}"
)

print(
    f"Annotation thuộc ứng viên: "
    f"{candidate_annotation_count:,}"
)

print(
    f"Annotation thuộc nhóm rà soát: "
    f"{review_annotation_count:,}"
)

if (
    actual_annotation_count != expected_annotation_count
    or unique_annotation_count != expected_annotation_count
):
    raise ValueError(
        "Số lượng annotation hoặc mã annotation không khớp."
    )

if (
    candidate_annotation_count + review_annotation_count
    != expected_annotation_count
):
    raise ValueError(
        "Không bảo toàn được toàn bộ annotation."
    )

print("[ĐẠT] Tất cả annotation đã được ánh xạ đúng một lần.")

# 8. Xem trước bảng ánh xạ annotation

mapping_preview_columns = [
    "annotation_id",
    "patient_id",
    "laterality",
    "lesion_type",
    "image_view",
    "abnormality_id",
    "pathology",
    "source_split",
    "source_table",
    "source_row_number",
    "lesion_candidate_id",
    "review_group_id",
    "annotation_link_status",
    "full_image_csv_path",
    "crop_image_csv_path",
    "roi_mask_csv_path",
]

mapping_preview_columns = [
    column
    for column in mapping_preview_columns
    if column in lesion_annotation_mapping_df.columns
]

print("\n" + "=" * 80)
print("XEM TRƯỚC BẢNG ÁNH XẠ ANNOTATION")
print("=" * 80)

with pd.option_context("display.max_colwidth", 100):

    display(
        lesion_annotation_mapping_df[
            mapping_preview_columns
        ].head(15)
    )


KIỂM TRA TÍNH TOÀN VẸN BẢNG ÁNH XẠ
Số annotation nguồn: 3,568
Số dòng ánh xạ: 3,568
Số annotation_id duy nhất: 3,568
Annotation thuộc ứng viên: 2,945
Annotation thuộc nhóm rà soát: 623
[ĐẠT] Tất cả annotation đã được ánh xạ đúng một lần.

XEM TRƯỚC BẢNG ÁNH XẠ ANNOTATION


,annotation_id,patient_id,laterality,lesion_type,image_view,abnormality_id,pathology,source_split,source_table,source_row_number,lesion_candidate_id,review_group_id,annotation_link_status,full_image_csv_path,crop_image_csv_path,roi_mask_csv_path
0,calc_train__00001,P_00005,RIGHT,calcification,CC,1,MALIGNANT,train,calc_train,1,LCAND_000004,<NA>,Co_ma_ung_vien,Calc-Training_P_00005_RIGHT_CC/1.3.6.1.4.1.9590.100.1.2.408909860712120272633130274602115723157/...,Calc-Training_P_00005_RIGHT_CC_1/1.3.6.1.4.1.9590.100.1.2.32877891901241276921808012421408870908...,Calc-Training_P_00005_RIGHT_CC_1/1.3.6.1.4.1.9590.100.1.2.32877891901241276921808012421408870908...
1,calc_train__00002,P_00005,RIGHT,calcification,MLO,1,MALIGNANT,train,calc_train,2,LCAND_000004,<NA>,Co_ma_ung_vien,Calc-Training_P_00005_RIGHT_MLO/1.3.6.1.4.1.9590.100.1.2.427517897311902339923511678883689433338...,Calc-Training_P_00005_RIGHT_MLO_1/1.3.6.1.4.1.9590.100.1.2.6751236221031963610814850438268078193...,Calc-Training_P_00005_RIGHT_MLO_1/1.3.6.1.4.1.9590.100.1.2.6751236221031963610814850438268078193...
2,calc_train__00003,P_00007,LEFT,calcification,CC,1,BENIGN,train,calc_train,3,LCAND_000005,<NA>,Co_ma_ung_vien,Calc-Training_P_00007_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.201322325113694962619881476352450072222/1...,Calc-Training_P_00007_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.241202057913673145232234613012384759880...,Calc-Training_P_00007_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.241202057913673145232234613012384759880...
3,calc_train__00004,P_00007,LEFT,calcification,MLO,1,BENIGN,train,calc_train,4,LCAND_000005,<NA>,Co_ma_ung_vien,Calc-Training_P_00007_LEFT_MLO/1.3.6.1.4.1.9590.100.1.2.370479499712916693322010643793108454887/...,Calc-Training_P_00007_LEFT_MLO_1/1.3.6.1.4.1.9590.100.1.2.31425027291117028920388234902422986882...,Calc-Training_P_00007_LEFT_MLO_1/1.3.6.1.4.1.9590.100.1.2.31425027291117028920388234902422986882...
4,calc_train__00005,P_00008,LEFT,calcification,CC,1,BENIGN_WITHOUT_CALLBACK,train,calc_train,5,<NA>,LREV_000001,Thuoc_nhom_can_ra_soat,Calc-Training_P_00008_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.162256682111885666305889708873412464189/1...,Calc-Training_P_00008_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.336811694512764490002272925921108351157...,Calc-Training_P_00008_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.336811694512764490002272925921108351157...
5,calc_train__00006,P_00008,LEFT,calcification,CC,2,BENIGN_WITHOUT_CALLBACK,train,calc_train,6,<NA>,LREV_000001,Thuoc_nhom_can_ra_soat,Calc-Training_P_00008_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.162256682111885666305889708873412464189/1...,Calc-Training_P_00008_LEFT_CC_2/1.3.6.1.4.1.9590.100.1.2.207558519313356140106496367610041842177...,Calc-Training_P_00008_LEFT_CC_2/1.3.6.1.4.1.9590.100.1.2.207558519313356140106496367610041842177...
6,calc_train__00007,P_00008,LEFT,calcification,CC,3,BENIGN_WITHOUT_CALLBACK,train,calc_train,7,<NA>,LREV_000001,Thuoc_nhom_can_ra_soat,Calc-Training_P_00008_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.162256682111885666305889708873412464189/1...,Calc-Training_P_00008_LEFT_CC_3/1.3.6.1.4.1.9590.100.1.2.74480335312443720440103727953726257477/...,Calc-Training_P_00008_LEFT_CC_3/1.3.6.1.4.1.9590.100.1.2.74480335312443720440103727953726257477/...
7,calc_train__00008,P_00008,LEFT,calcification,MLO,1,BENIGN_WITHOUT_CALLBACK,train,calc_train,8,<NA>,LREV_000001,Thuoc_nhom_can_ra_soat,Calc-Training_P_00008_LEFT_MLO/1.3.6.1.4.1.9590.100.1.2.84155842712765660429462608981255328663/1...,Calc-Training_P_00008_LEFT_MLO_1/1.3.6.1.4.1.9590.100.1.2.19421664301294301022274117045309492678...,Calc-Training_P_00008_LEFT_MLO_1/1.3.6.1.4.1.9590.100.1.2.19421664301294301022274117045309492678...
8,calc_train__00009,P_00008,LEFT,calcification,MLO,2,BENIGN_WITHOUT_CALLBACK,train,calc_train,9,<NA>,LREV_000001,Thuoc_nhom_can_ra_soat,Calc-Training_P_00008_LEFT_MLO/1.3.6.1.4.1.9590.100.1.2.84155842712765660429462608981255328663/1...,Calc-Training_P_00008_LEFT_MLO_2/1.3.6.1.4.1.9590.100.1.2.36373558861103632922611167132350714147...,Calc-Training_P_00008_L

### 3.8. Xây dựng bảng ánh xạ ảnh

### 03.8.1. Kiểm tra và chuẩn bị đường dẫn ảnh

Mỗi annotation trong bảng dữ liệu nguồn có hai loại đường dẫn ảnh:

- `full_image_csv_path`: đường dẫn ảnh chụp đầy đủ.
- `crop_image_csv_path`: đường dẫn ảnh cắt vùng bất thường.

Kiểm tra tính đầy đủ của hai loại đường dẫn và chuyển dữ liệu từ dạng rộng sang dạng dài để xây dựng bảng liên kết ảnh.

Mã ảnh được tạo mới chỉ phục vụ quản lý nội bộ, không thay thế SOPInstanceUID hoặc các mã định danh DICOM gốc.

In [89]:
# 3.8.1 — KIỂM TRA VÀ CHUẨN BỊ ĐƯỜNG DẪN ẢNH

# 1. Kiểm tra bảng ánh xạ annotation

if "lesion_annotation_mapping_df" not in globals():
    raise NameError(
        "Không tìm thấy lesion_annotation_mapping_df. "
        "Hãy chạy hoàn tất Phần 03.7.3 trước."
    )

image_source_df = lesion_annotation_mapping_df.copy()

image_path_columns = [
    "full_image_csv_path",
    "crop_image_csv_path",
]

missing_columns = [
    column
    for column in image_path_columns
    if column not in image_source_df.columns
]

if missing_columns:
    raise ValueError(
        f"Thiếu các cột đường dẫn ảnh: {missing_columns}"
    )

# 2. Kiểm tra giá trị thiếu và chuỗi rỗng

path_quality_records = []

for column in image_path_columns:

    series = image_source_df[column]

    normalized = series.astype("string").str.strip()

    path_quality_records.append({
        "loai_duong_dan": column,
        "tong_so_annotation": len(series),
        "so_gia_tri_thieu": int(series.isna().sum()),
        "so_chuoi_rong": int(
            (series.notna() & normalized.eq("")).sum()
        ),
        "so_duong_dan_co_gia_tri": int(
            (series.notna() & normalized.ne("")).sum()
        ),
        "so_duong_dan_phan_biet": int(
            normalized[
                series.notna() & normalized.ne("")
            ].nunique()
        ),
    })

image_path_quality_df = pd.DataFrame(
    path_quality_records
)

print("=" * 80)
print("KIỂM TRA CHẤT LƯỢNG ĐƯỜNG DẪN ẢNH")
print("=" * 80)

display(image_path_quality_df)

# 3. Kiểm tra những annotation có đường dẫn ảnh không hợp lệ

invalid_image_path_mask = pd.Series(
    False,
    index=image_source_df.index,
)

for column in image_path_columns:

    normalized = image_source_df[column].astype("string").str.strip()

    invalid_image_path_mask |= (
        image_source_df[column].isna()
        | normalized.eq("").fillna(True)
    )

invalid_image_path_rows_df = (
    image_source_df.loc[
        invalid_image_path_mask,
        [
            "annotation_id",
            "patient_id",
            "image_view",
            "source_table",
            "source_row_number",
        ] + image_path_columns,
    ].copy()
)

print("\n" + "=" * 80)
print("ANNOTATION CÓ ĐƯỜNG DẪN ẢNH THIẾU HOẶC RỖNG")
print("=" * 80)

print(
    f"Số annotation cần kiểm tra: "
    f"{len(invalid_image_path_rows_df):,}"
)

display(
    invalid_image_path_rows_df.head(20)
)

# Không tiếp tục tạo danh mục khi có đường dẫn thiếu.
if not invalid_image_path_rows_df.empty:
    raise ValueError(
        "Có đường dẫn ảnh thiếu hoặc rỗng. "
        "Cần kiểm tra trước khi xây dựng bảng ánh xạ."
    )

# 4. Kiểm tra tính đầy đủ của các thông tin liên kết

required_mapping_columns = [
    "annotation_id",
    "lesion_candidate_id",
    "review_group_id",
    "patient_id",
    "laterality",
    "lesion_type",
    "image_view",
    "source_split",
    "source_table",
    "source_row_number",
]

missing_mapping_columns = [
    column
    for column in required_mapping_columns
    if column not in image_source_df.columns
]

if missing_mapping_columns:
    raise ValueError(
        f"Thiếu cột ánh xạ cần thiết: {missing_mapping_columns}"
    )

print("\n[ĐẠT] Đường dẫn ảnh và thông tin liên kết đã đầy đủ.")

KIỂM TRA CHẤT LƯỢNG ĐƯỜNG DẪN ẢNH


,loai_duong_dan,tong_so_annotation,so_gia_tri_thieu,so_chuoi_rong,so_duong_dan_co_gia_tri,so_duong_dan_phan_biet
0,full_image_csv_path,3568,0,0,3568,3103
1,crop_image_csv_path,3568,0,0,3568,3568



ANNOTATION CÓ ĐƯỜNG DẪN ẢNH THIẾU HOẶC RỖNG
Số annotation cần kiểm tra: 0


,annotation_id,patient_id,image_view,source_table,source_row_number,full_image_csv_path,crop_image_csv_path



[ĐẠT] Đường dẫn ảnh và thông tin liên kết đã đầy đủ.


### 3.8.2. Xây dựng danh mục ảnh và bảng ánh xạ ảnh

Từ hai loại đường dẫn ảnh đã kiểm tra, xây dựng danh mục ảnh duy nhất.

Mỗi đường dẫn kết hợp với loại ảnh được cấp một mã quản lý nội bộ.

Sau đó, tạo bảng ánh xạ giữa annotation và ảnh. Một annotation sẽ có tối đa hai bản ghi liên kết:

- Một bản ghi ảnh đầy đủ.
- Một bản ghi ảnh cắt.

Một ảnh đầy đủ có thể được liên kết với nhiều annotation khác nhau. Vì vậy, số dòng của bảng ánh xạ không được đồng nhất với số lượng ảnh duy nhất.

Bảng danh mục ảnh và bảng ánh xạ được lưu tách biệt để bảo toàn cấu trúc quan hệ dữ liệu.

In [90]:
# 3.8.2 — XÂY DỰNG DANH MỤC ẢNH VÀ BẢNG ÁNH XẠ

# KIỂM TRA VÀ KHỞI TẠO DỮ LIỆU ĐẦU VÀO

if "lesion_annotation_mapping_df" not in globals():
    raise NameError(
        "Không tìm thấy lesion_annotation_mapping_df. "
        "Hãy chạy hoàn tất Phần 03.7.3 trước."
    )

# Tạo bản sao bảng ánh xạ annotation để xây dựng bảng ảnh.
image_source_df = lesion_annotation_mapping_df.copy()

print("=" * 80)
print("KIỂM TRA DỮ LIỆU ĐẦU VÀO PHẦN 03.8.2")
print("=" * 80)

print(f"Tổng số annotation: {len(image_source_df):,}")

required_columns = [
    "annotation_id",
    "full_image_csv_path",
    "crop_image_csv_path",
]

missing_columns = [
    column
    for column in required_columns
    if column not in image_source_df.columns
]

if missing_columns:
    raise ValueError(
        f"Thiếu các cột cần thiết: {missing_columns}"
    )

print("[ĐẠT] Dữ liệu đầu vào đầy đủ.")

# 1. Xây dựng bảng đường dẫn ảnh dạng dài

image_common_columns = [
    "annotation_id",
    "lesion_candidate_id",
    "review_group_id",
    "patient_id",
    "laterality",
    "lesion_type",
    "image_view",
    "pathology",
    "source_split",
    "source_table",
    "source_csv",
    "source_row_index",
    "source_row_number",
]

image_common_columns = [
    column
    for column in image_common_columns
    if column in image_source_df.columns
]


full_image_rows_df = image_source_df[
    image_common_columns + ["full_image_csv_path"]
].copy()

full_image_rows_df = full_image_rows_df.rename(
    columns={
        "full_image_csv_path": "image_csv_path"
    }
)

full_image_rows_df["image_role"] = "full_image"


crop_image_rows_df = image_source_df[
    image_common_columns + ["crop_image_csv_path"]
].copy()

crop_image_rows_df = crop_image_rows_df.rename(
    columns={
        "crop_image_csv_path": "image_csv_path"
    }
)

crop_image_rows_df["image_role"] = "cropped_image"


lesion_image_mapping_df = pd.concat(
    [
        full_image_rows_df,
        crop_image_rows_df,
    ],
    ignore_index=True,
)

# 2. Chuẩn hóa tạm thời đường dẫn để kiểm tra trùng lặp

lesion_image_mapping_df["image_path_key"] = (
    lesion_image_mapping_df["image_csv_path"]
    .astype("string")
    .str.strip()
    .str.replace("\\", "/", regex=False)
)

# 3. Tạo danh mục đường dẫn ảnh duy nhất

image_path_catalog_df = (
    lesion_image_mapping_df[
        [
            "image_role",
            "image_path_key",
            "image_csv_path",
        ]
    ]
    .drop_duplicates(
        subset=[
            "image_role",
            "image_path_key",
        ]
    )
    .sort_values(
        by=[
            "image_role",
            "image_path_key",
        ]
    )
    .reset_index(drop=True)
)

# 4. Tạo mã ảnh nội bộ theo từng loại ảnh
image_path_catalog_df["image_path_id"] = pd.NA

for image_role, prefix in [
    ("full_image", "IMG_FULL"),
    ("cropped_image", "IMG_CROP"),
]:

    role_mask = (
        image_path_catalog_df["image_role"]
        == image_role
    )

    role_count = int(role_mask.sum())

    image_path_catalog_df.loc[
        role_mask,
        "image_path_id",
    ] = [
        f"{prefix}_{number:06d}"
        for number in range(1, role_count + 1)
    ]

# 5. Liên kết mã ảnh vào bảng ánh xạ

lesion_image_mapping_df = lesion_image_mapping_df.merge(
    image_path_catalog_df[
        [
            "image_role",
            "image_path_key",
            "image_path_id",
        ]
    ],
    on=[
        "image_role",
        "image_path_key",
    ],
    how="left",
    validate="many_to_one",
)

# 6. Kiểm tra tính duy nhất của quan hệ annotation - loại ảnh

duplicate_image_mapping_mask = (
    lesion_image_mapping_df.duplicated(
        subset=[
            "annotation_id",
            "image_role",
        ],
        keep=False,
    )
)

if duplicate_image_mapping_mask.any():

    display(
        lesion_image_mapping_df.loc[
            duplicate_image_mapping_mask
        ].head(20)
    )

    raise ValueError(
        "Phát hiện annotation được ánh xạ trùng loại ảnh."
    )

# 7. Kiểm tra mỗi annotation có đủ hai loại ảnh

image_count_per_annotation_df = (
    lesion_image_mapping_df
    .groupby("annotation_id")
    .agg(
        so_loai_anh=("image_role", "nunique"),
        so_duong_dan=("image_path_id", "count"),
    )
    .reset_index()
)

invalid_image_count_df = (
    image_count_per_annotation_df.loc[
        image_count_per_annotation_df["so_loai_anh"] != 2
    ]
    .copy()
)

# 8. Kiểm tra tổng số quan hệ ảnh

expected_image_links = (
    len(image_source_df) * 2
)

actual_image_links = (
    len(lesion_image_mapping_df)
)

print("=" * 80)
print("KIỂM TRA BẢNG ÁNH XẠ ẢNH")
print("=" * 80)

print(
    f"Số annotation nguồn: "
    f"{len(image_source_df):,}"
)

print(
    f"Số quan hệ ảnh kỳ vọng: "
    f"{expected_image_links:,}"
)

print(
    f"Số quan hệ ảnh thực tế: "
    f"{actual_image_links:,}"
)

print(
    f"Số annotation chưa đủ hai loại ảnh: "
    f"{len(invalid_image_count_df):,}"
)

if (
    actual_image_links != expected_image_links
    or not invalid_image_count_df.empty
):
    raise ValueError(
        "Bảng ánh xạ ảnh chưa bảo đảm mỗi annotation "
        "có đúng một ảnh đầy đủ và một ảnh cắt."
    )

print("[ĐẠT] Mỗi annotation có đúng hai quan hệ ảnh.")

# 9. Thống kê số ảnh duy nhất theo loại

image_catalog_summary_df = (
    image_path_catalog_df
    .groupby("image_role")
    .agg(
        so_anh_duy_nhat=("image_path_id", "nunique"),
        so_duong_dan=("image_csv_path", "count"),
    )
    .reset_index()
)

print("\n" + "=" * 80)
print("THỐNG KÊ DANH MỤC ẢNH DUY NHẤT")
print("=" * 80)

display(image_catalog_summary_df)

# 10. Kiểm tra mức độ dùng chung ảnh giữa nhiều annotation

shared_image_usage_df = (
    lesion_image_mapping_df
    .groupby(
        [
            "image_role",
            "image_path_id",
        ],
    )
    .agg(
        so_annotation_su_dung=("annotation_id", "nunique"),
        so_ung_vien_su_dung=("lesion_candidate_id", "nunique"),
        so_nhom_ra_soat_su_dung=("review_group_id", "nunique"),
    )
    .reset_index()
)

shared_image_usage_df = shared_image_usage_df.loc[
    shared_image_usage_df["so_annotation_su_dung"] > 1
].copy()

print("\n" + "=" * 80)
print("ẢNH ĐƯỢC SỬ DỤNG BỞI NHIỀU ANNOTATION")
print("=" * 80)

print(
    f"Số ảnh có nhiều annotation tham chiếu: "
    f"{len(shared_image_usage_df):,}"
)

display(
    shared_image_usage_df.head(20)
)

# 11. Xem trước bảng danh mục ảnh

print("\n" + "=" * 80)
print("XEM TRƯỚC DANH MỤC ẢNH")
print("=" * 80)

display(
    image_path_catalog_df.head(15)
)

# 12. Xem trước bảng ánh xạ ảnh

image_mapping_preview_columns = [
    "annotation_id",
    "lesion_candidate_id",
    "review_group_id",
    "image_path_id",
    "image_role",
    "image_view",
    "image_csv_path",
    "source_table",
    "source_row_number",
]

print("\n" + "=" * 80)
print("XEM TRƯỚC BẢNG ÁNH XẠ ẢNH")
print("=" * 80)

with pd.option_context("display.max_colwidth", 100):

    display(
        lesion_image_mapping_df[
            image_mapping_preview_columns
        ].head(20)
    )

KIỂM TRA DỮ LIỆU ĐẦU VÀO PHẦN 03.8.2
Tổng số annotation: 3,568
[ĐẠT] Dữ liệu đầu vào đầy đủ.
KIỂM TRA BẢNG ÁNH XẠ ẢNH
Số annotation nguồn: 3,568
Số quan hệ ảnh kỳ vọng: 7,136
Số quan hệ ảnh thực tế: 7,136
Số annotation chưa đủ hai loại ảnh: 0
[ĐẠT] Mỗi annotation có đúng hai quan hệ ảnh.

THỐNG KÊ DANH MỤC ẢNH DUY NHẤT


,image_role,so_anh_duy_nhat,so_duong_dan
0,cropped_image,3568,3568
1,full_image,3103,3103



ẢNH ĐƯỢC SỬ DỤNG BỞI NHIỀU ANNOTATION
Số ảnh có nhiều annotation tham chiếu: 293


,image_role,image_path_id,so_annotation_su_dung,so_ung_vien_su_dung,so_nhom_ra_soat_su_dung
3570,full_image,IMG_FULL_000003,2,0,1
3571,full_image,IMG_FULL_000004,2,0,1
3576,full_image,IMG_FULL_000009,2,2,0
3577,full_image,IMG_FULL_000010,2,2,0
3583,full_image,IMG_FULL_000016,2,0,1
3584,full_image,IMG_FULL_000017,2,0,1
3585,full_image,IMG_FULL_000018,2,0,1
3586,full_image,IMG_FULL_000019,2,0,1
3599,full_image,IMG_FULL_000032,2,0,1
3600,full_image,IMG_FULL_000033,2,0,1



XEM TRƯỚC DANH MỤC ẢNH


,image_role,image_path_key,image_csv_path,image_path_id
0,cropped_image,Calc-Test_P_00038_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.161465562211359959230647609981488894942/1.3...,Calc-Test_P_00038_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.161465562211359959230647609981488894942/1.3...,IMG_CROP_000001
1,cropped_image,Calc-Test_P_00038_LEFT_MLO_1/1.3.6.1.4.1.9590.100.1.2.29112199613143138535387754440942211739/1.3...,Calc-Test_P_00038_LEFT_MLO_1/1.3.6.1.4.1.9590.100.1.2.29112199613143138535387754440942211739/1.3...,IMG_CROP_000002
2,cropped_image,Calc-Test_P_00038_RIGHT_CC_1/1.3.6.1.4.1.9590.100.1.2.200764632211227648028305709420627883105/1....,Calc-Test_P_00038_RIGHT_CC_1/1.3.6.1.4.1.9590.100.1.2.200764632211227648028305709420627883105/1....,IMG_CROP_000003
3,cropped_image,Calc-Test_P_00038_RIGHT_CC_2/1.3.6.1.4.1.9590.100.1.2.248538452013626298441249276382187367143/1....,Calc-Test_P_00038_RIGHT_CC_2/1.3.6.1.4.1.9590.100.1.2.248538452013626298441249276382187367143/1....,IMG_CROP_000004
4,cropped_image,Calc-Test_P_00038_RIGHT_MLO_1/1.3.6.1.4.1.9590.100.1.2.348569460311013218440657632223354965172/1...,Calc-Test_P_00038_RIGHT_MLO_1/1.3.6.1.4.1.9590.100.1.2.348569460311013218440657632223354965172/1...,IMG_CROP_000005
5,cropped_image,Calc-Test_P_00038_RIGHT_MLO_2/1.3.6.1.4.1.9590.100.1.2.57362695012780927721008884783926968463/1....,Calc-Test_P_00038_RIGHT_MLO_2/1.3.6.1.4.1.9590.100.1.2.57362695012780927721008884783926968463/1....,IMG_CROP_000006
6,cropped_image,Calc-Test_P_00041_LEFT_CC_2/1.3.6.1.4.1.9590.100.1.2.261142802012312819739743387111814412775/1.3...,Calc-Test_P_00041_LEFT_CC_2/1.3.6.1.4.1.9590.100.1.2.261142802012312819739743387111814412775/1.3...,IMG_CROP_000007
7,cropped_image,Calc-Test_P_00041_LEFT_MLO_2/1.3.6.1.4.1.9590.100.1.2.372962290011068589008332414733650487987/1....,Calc-Test_P_00041_LEFT_MLO_2/1.3.6.1.4.1.9590.100.1.2.372962290011068589008332414733650487987/1....,IMG_CROP_000008
8,cropped_image,Calc-Test_P_00077_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.169810086012222817116620676350914651179/1.3...,Calc-Test_P_00077_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.169810086012222817116620676350914651179/1.3...,IMG_CROP_000009
9,cropped_image,Calc-Test_P_00077_LEFT_MLO_1/1.3.6.1.4.1.9590.100.1.2.73175700011214399933783883971096330370/1.3...,Calc-Test_P_00077_LEFT_MLO_1/1.3.6.1.4.1.9590.100.1.2.73175700011214399933783883971096330370/1.3...,IMG_CROP_000010



XEM TRƯỚC BẢNG ÁNH XẠ ẢNH


,annotation_id,lesion_candidate_id,review_group_id,image_path_id,image_role,image_view,image_csv_path,source_table,source_row_number
0,calc_train__00001,LCAND_000004,<NA>,IMG_FULL_000285,full_image,CC,Calc-Training_P_00005_RIGHT_CC/1.3.6.1.4.1.9590.100.1.2.408909860712120272633130274602115723157/...,calc_train,1
1,calc_train__00002,LCAND_000004,<NA>,IMG_FULL_000286,full_image,MLO,Calc-Training_P_00005_RIGHT_MLO/1.3.6.1.4.1.9590.100.1.2.427517897311902339923511678883689433338...,calc_train,2
2,calc_train__00003,LCAND_000005,<NA>,IMG_FULL_000287,full_image,CC,Calc-Training_P_00007_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.201322325113694962619881476352450072222/1...,calc_train,3
3,calc_train__00004,LCAND_000005,<NA>,IMG_FULL_000288,full_image,MLO,Calc-Training_P_00007_LEFT_MLO/1.3.6.1.4.1.9590.100.1.2.370479499712916693322010643793108454887/...,calc_train,4
4,calc_train__00005,<NA>,LREV_000001,IMG_FULL_000289,full_image,CC,Calc-Training_P_00008_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.162256682111885666305889708873412464189/1...,calc_train,5
5,calc_train__00006,<NA>,LREV_000001,IMG_FULL_000289,full_image,CC,Calc-Training_P_00008_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.162256682111885666305889708873412464189/1...,calc_train,6
6,calc_train__00007,<NA>,LREV_000001,IMG_FULL_000289,full_image,CC,Calc-Training_P_00008_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.162256682111885666305889708873412464189/1...,calc_train,7
7,calc_train__00008,<NA>,LREV_000001,IMG_FULL_000290,full_image,MLO,Calc-Training_P_00008_LEFT_MLO/1.3.6.1.4.1.9590.100.1.2.84155842712765660429462608981255328663/1...,calc_train,8
8,calc_train__00009,<NA>,LREV_000001,IMG_FULL_000290,full_image,MLO,Calc-Training_P_00008_LEFT_MLO/1.3.6.1.4.1.9590.100.1.2.84155842712765660429462608981255328663/1...,calc_train,9
9,calc_train__00010,<NA>,LREV_000001,IMG_FULL_000290,full_image,MLO,Calc-Training_P_00008_LEFT_MLO/1.3.6.1.4.1.9590.100.1.2.84155842712765660429462608981255328663/1...,calc_train,10


### 3.9. Xây dựng bảng ánh xạ ROI mask

### 3.9.1. Kiểm tra kết quả kiểm định ROI từ Notebook 02

Notebook 02 đã thực hiện kiểm định cấu trúc ROI mask và đối chiếu kích thước mask với ảnh đầy đủ.

Trong bước này, đọc lại các bảng kết quả kiểm định đã lưu để xác định:

- Các trường định danh annotation và ROI mask.
- Đường dẫn hoặc mã DICOM được sử dụng để liên kết.
- Kết quả đọc mask.
- Kích thước ROI mask và ảnh đầy đủ.
- Trạng thái tương thích kích thước.
- Các trường hợp ngoại lệ cần giữ lại.

Việc kiểm tra nhằm xác định khóa liên kết chính xác giữa bảng annotation của Notebook 03 và kết quả kiểm định Notebook 02.

In [91]:
# 3.9.1 — KIỂM TRA KẾT QUẢ KIỂM ĐỊNH ROI MASK TỪ NOTEBOOK 02

from pathlib import Path
import pandas as pd

# 1. Khai báo đường dẫn đến các kết quả Notebook 02

NB02_PROCESSED_DIR = (
    Path(r"C:\Breast-Cancer-CBIS-DDSM")
    / "Data"
    / "processed"
)

ROI_VALIDATION_FILE = (
    NB02_PROCESSED_DIR
    / "nb02_roi_mask_validation.csv"
)

ROI_VALIDATION_SUMMARY_FILE = (
    NB02_PROCESSED_DIR
    / "nb02_roi_mask_validation_summary.csv"
)

ROI_EXCEPTIONS_FILE = (
    NB02_PROCESSED_DIR
    / "nb02_roi_mask_exceptions.csv"
)

# 2. Kiểm tra sự tồn tại của các file

roi_audit_files = {
    "Bảng kiểm định ROI": ROI_VALIDATION_FILE,
    "Bảng tổng hợp kiểm định ROI": ROI_VALIDATION_SUMMARY_FILE,
    "Bảng ngoại lệ ROI": ROI_EXCEPTIONS_FILE,
}

missing_roi_files = []

print("=" * 80)
print("KIỂM TRA CÁC FILE KIỂM ĐỊNH ROI")
print("=" * 80)

for file_description, file_path in roi_audit_files.items():

    exists = file_path.exists()

    print(
        f"[{'ĐẠT' if exists else 'THIẾU'}] "
        f"{file_description}: {file_path}"
    )

    if not exists:
        missing_roi_files.append(file_description)

if missing_roi_files:
    raise FileNotFoundError(
        f"Không tìm thấy các file: {missing_roi_files}"
    )

# 3. Đọc các bảng kiểm định

nb02_roi_validation_df = pd.read_csv(
    ROI_VALIDATION_FILE,
    low_memory=False,
)

nb02_roi_validation_summary_df = pd.read_csv(
    ROI_VALIDATION_SUMMARY_FILE,
    low_memory=False,
)

nb02_roi_exceptions_df = pd.read_csv(
    ROI_EXCEPTIONS_FILE,
    low_memory=False,
)

# 4. Kiểm tra kích thước các bảng

print("\n" + "=" * 80)
print("KÍCH THƯỚC CÁC BẢNG KIỂM ĐỊNH")
print("=" * 80)

roi_table_shape_df = pd.DataFrame([
    {
        "ten_bang": "nb02_roi_validation_df",
        "so_dong": len(nb02_roi_validation_df),
        "so_cot": len(nb02_roi_validation_df.columns),
    },
    {
        "ten_bang": "nb02_roi_validation_summary_df",
        "so_dong": len(nb02_roi_validation_summary_df),
        "so_cot": len(nb02_roi_validation_summary_df.columns),
    },
    {
        "ten_bang": "nb02_roi_exceptions_df",
        "so_dong": len(nb02_roi_exceptions_df),
        "so_cot": len(nb02_roi_exceptions_df.columns),
    },
])

display(roi_table_shape_df)

# 5. Kiểm tra tên các cột

for table_name, dataframe in [
    ("Bảng kiểm định ROI", nb02_roi_validation_df),
    ("Bảng tổng hợp ROI", nb02_roi_validation_summary_df),
    ("Bảng ngoại lệ ROI", nb02_roi_exceptions_df),
]:

    print("\n" + "=" * 80)
    print(f"CÁC CỘT TRONG {table_name.upper()}")
    print("=" * 80)

    print(dataframe.columns.tolist())

# 6. Kiểm tra kiểu dữ liệu

print("\n" + "=" * 80)
print("KIỂU DỮ LIỆU CÁC CỘT TRONG BẢNG KIỂM ĐỊNH ROI")
print("=" * 80)

display(
    nb02_roi_validation_df.dtypes
    .rename("kieu_du_lieu")
    .reset_index()
    .rename(columns={"index": "ten_cot"})
)

# 7. Hiển thị mẫu bảng kiểm định ROI

print("\n" + "=" * 80)
print("MẪU BẢNG KIỂM ĐỊNH ROI")
print("=" * 80)

with pd.option_context("display.max_columns", None):
    with pd.option_context("display.max_colwidth", 100):

        display(
            nb02_roi_validation_df.head(10)
        )

# 8. Hiển thị bảng tổng hợp kiểm định ROI

print("\n" + "=" * 80)
print("BẢNG TỔNG HỢP KIỂM ĐỊNH ROI")
print("=" * 80)

display(
    nb02_roi_validation_summary_df
)

# 9. Hiển thị các trường hợp ngoại lệ

print("\n" + "=" * 80)
print("MẪU CÁC TRƯỜNG HỢP NGOẠI LỆ ROI")
print("=" * 80)

with pd.option_context("display.max_columns", None):
    with pd.option_context("display.max_colwidth", 100):

        display(
            nb02_roi_exceptions_df.head(20)
        )

# 10. Tìm các cột có khả năng dùng làm khóa liên kết

possible_key_terms = [
    "patient",
    "source",
    "split",
    "csv",
    "row",
    "image",
    "view",
    "abnormality",
    "path",
    "mask",
    "uid",
    "series",
    "study",
    "sop",
]

possible_key_columns = [
    column
    for column in nb02_roi_validation_df.columns
    if any(
        term in str(column).lower()
        for term in possible_key_terms
    )
]

print("\n" + "=" * 80)
print("CÁC CỘT CÓ KHẢ NĂNG DÙNG ĐỂ LIÊN KẾT")
print("=" * 80)

print(possible_key_columns)

KIỂM TRA CÁC FILE KIỂM ĐỊNH ROI
[ĐẠT] Bảng kiểm định ROI: C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb02_roi_mask_validation.csv
[ĐẠT] Bảng tổng hợp kiểm định ROI: C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb02_roi_mask_validation_summary.csv
[ĐẠT] Bảng ngoại lệ ROI: C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb02_roi_mask_exceptions.csv

KÍCH THƯỚC CÁC BẢNG KIỂM ĐỊNH


,ten_bang,so_dong,so_cot
0,nb02_roi_validation_df,3568,30
1,nb02_roi_validation_summary_df,36,5
2,nb02_roi_exceptions_df,80,30



CÁC CỘT TRONG BẢNG KIỂM ĐỊNH ROI
['annotation_id', 'patient_id', 'split', 'abnormality_type', 'roi_mask_file_path', 'full_image_file_path', 'resolution_method', 'mask_read_status', 'full_header_status', 'mask_shape', 'full_image_shape', 'shape_match', 'mask_dtype', 'unique_value_count', 'unique_values_sample', 'is_empty', 'nonzero_pixel_count', 'mask_area_ratio', 'connected_components', 'bbox_x_min', 'bbox_y_min', 'bbox_x_max', 'bbox_y_max', 'mask_quality_status', 'error_message', 'mask_foreground_fraction', 'mask_values_binary', 'bbox_coordinate_space', 'bbox_full_image_coordinate_eligible', 'coordinate_note']

CÁC CỘT TRONG BẢNG TỔNG HỢP ROI
['section', 'metric', 'category', 'count', 'value']

CÁC CỘT TRONG BẢNG NGOẠI LỆ ROI
['annotation_id', 'patient_id', 'split', 'abnormality_type', 'roi_mask_file_path', 'full_image_file_path', 'resolution_method', 'mask_read_status', 'full_header_status', 'mask_shape', 'full_image_shape', 'shape_match', 'mask_dtype', 'unique_value_count', 'unique

,ten_cot,kieu_du_lieu
0,annotation_id,str
1,patient_id,str
2,split,str
3,abnormality_type,str
4,roi_mask_file_path,str
5,full_image_file_path,str
6,resolution_method,str
7,mask_read_status,str
8,full_header_status,str
9,mask_shape,str



MẪU BẢNG KIỂM ĐỊNH ROI


,annotation_id,patient_id,split,abnormality_type,roi_mask_file_path,full_image_file_path,resolution_method,mask_read_status,full_header_status,mask_shape,full_image_shape,shape_match,mask_dtype,unique_value_count,unique_values_sample,is_empty,nonzero_pixel_count,mask_area_ratio,connected_components,bbox_x_min,bbox_y_min,bbox_x_max,bbox_y_max,mask_quality_status,error_message,mask_foreground_fraction,mask_values_binary,bbox_coordinate_space,bbox_full_image_coordinate_eligible,coordinate_note
0,calc_test_0,P_00038,calc_test,calcification,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\P_00038_LEFT_CC_1.dcm\2017-08-29-94942\1-cropp...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\P_00038_LEFT_CC.dcm\1-63992\9a3af421-bb04-461c...,uid_pair,success,success,"(4616, 3016)","(4616, 3016)",True,uint8,2,"[0, 255]",False,38177,0.002742,1,288,1976,513,2185,structurally_valid,NaN,0.002742,True,roi_mask_pixel_coordinates,True,Same matrix shape; spatial alignment still requires dataset-level validation
1,calc_test_1,P_00038,calc_test,calcification,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Test_P_00038_LEFT_MLO_1\2017-08-29-11739\...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\P_00038_LEFT_MLO.dcm\1-97934\e1a1589e-55e8-49e...,uid_pair,success,success,"(4728, 3064)","(4728, 3064)",True,uint8,2,"[0, 255]",False,37121,0.002562,1,560,3816,769,4041,structurally_valid,NaN,0.002562,True,roi_mask_pixel_coordinates,True,Same matrix shape; spatial alignment still requires dataset-level validation
2,calc_test_2,P_00038,calc_test,calcification,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Test_P_00038_RIGHT_CC_1\2017-08-29-83105\...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\P_00038_RIGHT_CC.dcm\1-63304\7d487097-0a08-4d9...,uid_pair,success,success,"(4688, 2744)","(4688, 2744)",True,uint8,2,"[0, 255]",False,81869,0.006364,1,2104,1024,2737,1393,structurally_valid,NaN,0.006364,True,roi_mask_pixel_coordinates,True,Same matrix shape; spatial alignment still requires dataset-level validation
3,calc_test_3,P_00038,calc_test,calcification,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\P_00038_RIGHT_CC_2.dcm\2017-08-29-67143\1-crop...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\P_00038_RIGHT_CC.dcm\1-63304\7d487097-0a08-4d9...,uid_pair,success,success,"(4688, 2744)","(4688, 2744)",True,uint8,2,"[0, 255]",False,110185,0.008565,1,2040,2416,2737,2633,structurally_valid,NaN,0.008565,True,roi_mask_pixel_coordinates,True,Same matrix shape; spatial alignment still requires dataset-level validation
4,calc_test_4,P_00038,calc_test,calcification,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\P_00038_RIGHT_MLO_1.dcm\2017-08-29-65172\1-cro...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\P_00038_RIGHT_MLO.dcm\1-08846\85b4b60f-c821-48...,uid_pair,success,success,"(4720, 2928)","(4720, 2928)",True,uint8,2,"[0, 255]",False,197724,0.014307,1,1976,1528,2769,2297,structurally_valid,NaN,0.014307,True,roi_mask_pixel_coordinates,True,Same matrix shape; spatial alignment still requires dataset-level validation
5,calc_test_5,P_00038,calc_test,calcification,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Test_P_00038_RIGHT_MLO_2\2017-08-29-68463...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\P_00038_RIGHT_MLO.dcm\1-08846\85b4b60f-c821-48...,uid_pair,success,success,"(4720, 2928)","(4720, 2928)",True,uint8,2,"[0, 255]",False,167448,0.012116,1,2136,3160,2897,3617,structurally_valid,NaN,0.012116,True,roi_mask_pixel_coordinates,True,Same matrix shape; spatial alignment still requires dataset-level validation
6,calc_test_6,P_00041,calc_test,calcification,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\P_00041_LEFT_CC_2.dcm\2017-08-29-12775\1-cropp...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\P_00041_LEFT_CC.dcm\1-92812\dd394126-69f0-4194...,uid_pair,success,success,"(5664, 3984)","(5664, 3984)",True,uint8,2,"[0, 255]",False,5529,0.000245,1,2024,1216,2121,1297,structurally_valid,NaN,0.000245,True,roi_mask_pixel_co


BẢNG TỔNG HỢP KIỂM ĐỊNH ROI


,section,metric,category,count,value
0,overall,total_annotations,all,3568.0,NaN
1,file_read,mask_read_status,success,3568.0,NaN
2,file_read,full_header_status,success,3568.0,NaN
3,mask_quality,mask_quality_status,structurally_valid,3488.0,NaN
4,mask_quality,mask_quality_status,shape_mismatch,80.0,NaN
5,shape,shape_match,True,3488.0,NaN
6,shape,shape_match,False,80.0,NaN
7,pixel_values,mask_values_binary,True,3568.0,NaN
8,mask_content,empty_masks,is_empty_true,0.0,NaN
9,connected_components,count,all_available_masks,NaN,3568.000000



MẪU CÁC TRƯỜNG HỢP NGOẠI LỆ ROI


,annotation_id,patient_id,split,abnormality_type,roi_mask_file_path,full_image_file_path,resolution_method,mask_read_status,full_header_status,mask_shape,full_image_shape,shape_match,mask_dtype,unique_value_count,unique_values_sample,is_empty,nonzero_pixel_count,mask_area_ratio,connected_components,bbox_x_min,bbox_y_min,bbox_x_max,bbox_y_max,mask_quality_status,error_message,mask_foreground_fraction,mask_values_binary,bbox_coordinate_space,bbox_full_image_coordinate_eligible,coordinate_note
0,calc_test_62,P_00353,calc_test,calcification,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Test_P_00353_LEFT_CC_2\2017-08-29-42567\1...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\P_00353_LEFT_CC.dcm\1-51571\b7c4b6de-e712-4b11...,uid_pair,success,success,"(4688, 3088)","(4624, 3056)",False,uint8,2,"[0, 255]",False,22989,NaN,1,1240,2528,1401,2737,shape_mismatch,NaN,0.001588,True,roi_mask_pixel_coordinates,False,Shape mismatch; do not transfer bbox to full-image coordinates
1,calc_test_63,P_00353,calc_test,calcification,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Test_P_00353_LEFT_MLO_2\2017-08-29-49673\...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\P_00353_LEFT_MLO.dcm\1-71359\25d58169-9ac3-432...,uid_pair,success,success,"(4624, 3080)","(4640, 3072)",False,uint8,2,"[0, 255]",False,25917,NaN,1,824,3088,1009,3265,shape_mismatch,NaN,0.001820,True,roi_mask_pixel_coordinates,False,Shape mismatch; do not transfer bbox to full-image coordinates
2,mass_test_24,P_00145,mass_test,mass,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Mass-Test_P_00145_LEFT_CC_1\2017-09-27-31267\1...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Mass-Test_P_00145_LEFT_CC\2016-10-04-91907\1-f...,uid_pair,success,success,"(4386, 1789)","(5041, 2056)",False,uint8,2,"[0, 255]",False,69541,NaN,8,636,2414,966,2758,shape_mismatch,NaN,0.008863,True,roi_mask_pixel_coordinates,False,Shape mismatch; do not transfer bbox to full-image coordinates
3,mass_test_25,P_00145,mass_test,mass,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Mass-Test_P_00145_LEFT_MLO_1\2016-10-04-52166\...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Mass-Test_P_00145_LEFT_MLO\2016-10-04-06894\1-...,uid_pair,success,success,"(4373, 2155)","(5026, 2476)",False,uint8,2,"[0, 255]",False,61557,NaN,8,783,2696,1130,2991,shape_mismatch,NaN,0.006532,True,roi_mask_pixel_coordinates,False,Shape mismatch; do not transfer bbox to full-image coordinates
4,mass_test_84,P_00379,mass_test,mass,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Mass-Test_P_00379_LEFT_MLO_1\2016-10-04-32756\...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Mass-Test_P_00379_LEFT_MLO\2016-10-04-83531\1-...,uid_pair,success,success,"(4686, 2820)","(5386, 3241)",False,uint8,2,"[0, 255]",False,75593,NaN,8,494,1308,844,1633,shape_mismatch,NaN,0.005720,True,roi_mask_pixel_coordinates,False,Shape mismatch; do not transfer bbox to full-image coordinates
5,mass_test_85,P_00381,mass_test,mass,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Mass-Test_P_00381_LEFT_CC_1\2016-10-04-78728\1...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Mass-Test_P_00381_LEFT_CC\2016-10-04-82168\1-f...,uid_pair,success,success,"(4621, 2781)","(5311, 3196)",False,uint8,2,"[0, 255]",False,24760,NaN,2,1584,1594,1798,1808,shape_mismatch,NaN,0.001927,True,roi_mask_pixel_coordinates,False,Shape mismatch; do not transfer bbox to full-image coordinates
6,mass_test_86,P_00381,mass_test,mass,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Mass-Test_P_00381_LEFT_MLO_1\2016-10-04-32671\...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Mass-Test_P_00381_LEFT_MLO\2016-10-04-10372\1-...,uid_pair,success,success,"(4686, 3186)","(5386, 3661)",False,uint8,2,"[0, 255]",False,23486,NaN,1,2059,2710,2234,2944,shape_mismatch,NaN,0.001573,True,roi_mask_pixel_coordinates,False,Shape mismatch; do not transfer bbox to full-image coordinates
7,mass_test_160,P_00699,mass_test,mass,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Mass-Tes


CÁC CỘT CÓ KHẢ NĂNG DÙNG ĐỂ LIÊN KẾT
['patient_id', 'split', 'abnormality_type', 'roi_mask_file_path', 'full_image_file_path', 'mask_read_status', 'mask_shape', 'full_image_shape', 'mask_dtype', 'mask_area_ratio', 'mask_quality_status', 'mask_foreground_fraction', 'mask_values_binary', 'bbox_full_image_coordinate_eligible']


### 3.9.2. Liên kết kết quả kiểm định ROI với annotation

Bảng kiểm định ROI của Notebook 02 sử dụng mã annotation theo định dạng tên bảng nguồn kết hợp với chỉ số dòng bắt đầu từ 0.

Trong khi đó, Notebook 03 đã tạo mã annotation riêng để quản lý dữ liệu.

Do đó, sử dụng tổ hợp `source_table` và `source_row_index` làm khóa trung gian để liên kết hai bảng.

Sau khi liên kết, kiểm tra các thông tin độc lập:

- Mã bệnh nhân.
- Bảng dữ liệu nguồn.
- Loại bất thường.
- Chỉ số dòng và số thứ tự dòng nguồn.

Kết quả kiểm định ROI được giữ nguyên, bao gồm trạng thái đọc file, kích thước mask, giá trị pixel, bounding box, hệ tọa độ và các trường hợp ngoại lệ.

In [92]:
# 3.9.2 — XÂY DỰNG BẢNG ÁNH XẠ ROI MASK

# 1. KIỂM TRA VÀ KHỞI TẠO DỮ LIỆU

required_objects = [
    "lesion_annotation_mapping_df",
    "nb02_roi_validation_df",
]

for object_name in required_objects:

    if object_name not in globals():
        raise NameError(
            f"Không tìm thấy {object_name}. "
            "Hãy kiểm tra các phần trước."
        )

# Tạo bản sao, không sửa dữ liệu nguồn.
annotation_roi_source_df = (
    lesion_annotation_mapping_df.copy()
)

roi_validation_source_df = (
    nb02_roi_validation_df.copy()
)

print("=" * 80)
print("KIỂM TRA DỮ LIỆU ĐẦU VÀO")
print("=" * 80)

print(
    f"Annotation Notebook 03: "
    f"{len(annotation_roi_source_df):,}"
)

print(
    f"ROI validation Notebook 02: "
    f"{len(roi_validation_source_df):,}"
)

# 2. KIỂM TRA CÁC CỘT CẦN THIẾT

annotation_required_columns = [
    "annotation_id",
    "source_table",
    "source_row_index",
    "source_row_number",
    "patient_id",
    "lesion_type",
    "roi_mask_csv_path",
    "full_image_csv_path",
]

roi_required_columns = [
    "annotation_id",
    "patient_id",
    "split",
    "abnormality_type",
    "roi_mask_file_path",
    "full_image_file_path",
    "mask_read_status",
    "full_header_status",
    "mask_shape",
    "full_image_shape",
    "shape_match",
    "mask_quality_status",
    "is_empty",
    "mask_values_binary",
    "bbox_coordinate_space",
    "bbox_full_image_coordinate_eligible",
]

missing_annotation_columns = [
    column
    for column in annotation_required_columns
    if column not in annotation_roi_source_df.columns
]

missing_roi_columns = [
    column
    for column in roi_required_columns
    if column not in roi_validation_source_df.columns
]

if missing_annotation_columns:
    raise ValueError(
        f"Thiếu cột trong Notebook 03: "
        f"{missing_annotation_columns}"
    )

if missing_roi_columns:
    raise ValueError(
        f"Thiếu cột trong Notebook 02: "
        f"{missing_roi_columns}"
    )

print("[ĐẠT] Các cột cần thiết đều tồn tại.")

# 3. KIỂM TRA CHỈ SỐ DÒNG NGUỒN

annotation_roi_source_df["_source_row_index_numeric"] = (
    pd.to_numeric(
        annotation_roi_source_df["source_row_index"],
        errors="coerce",
    )
    .astype("Int64")
)

annotation_roi_source_df["_source_row_number_numeric"] = (
    pd.to_numeric(
        annotation_roi_source_df["source_row_number"],
        errors="coerce",
    )
    .astype("Int64")
)

if (
    annotation_roi_source_df[
        [
            "_source_row_index_numeric",
            "_source_row_number_numeric",
        ]
    ]
    .isna()
    .any()
    .any()
):
    raise ValueError(
        "Phát hiện chỉ số dòng nguồn bị thiếu hoặc không hợp lệ."
    )

# Kiểm tra quan hệ giữa chỉ số dòng bắt đầu từ 0
# và số thứ tự dòng bắt đầu từ 1.

row_number_mismatch_df = (
    annotation_roi_source_df.loc[
        annotation_roi_source_df[
            "_source_row_number_numeric"
        ]
        != (
            annotation_roi_source_df[
                "_source_row_index_numeric"
            ] + 1
        )
    ]
    .copy()
)

print("\n" + "=" * 80)
print("KIỂM TRA CHỈ SỐ DÒNG NGUỒN")
print("=" * 80)

print(
    "Số bản ghi có source_row_number khác source_row_index + 1:",
    f"{len(row_number_mismatch_df):,}"
)

if not row_number_mismatch_df.empty:
    display(
        row_number_mismatch_df[
            [
                "source_table",
                "source_row_index",
                "source_row_number",
            ]
        ].head(20)
    )

    raise ValueError(
        "Không xác nhận được quy tắc chỉ số dòng nguồn."
    )

print("[ĐẠT] Chỉ số dòng nguồn nhất quán.")

# 4. TẠO KHÓA LIÊN KẾT TRONG NOTEBOOK 03

annotation_roi_source_df["_roi_link_key"] = (
    annotation_roi_source_df["source_table"]
    .astype("string")
    .str.strip()
    .str.lower()
    + "_"
    + annotation_roi_source_df[
        "_source_row_index_numeric"
    ].astype("string")
)

if annotation_roi_source_df["_roi_link_key"].duplicated().any():
    raise ValueError(
        "Khóa liên kết Notebook 03 bị trùng."
    )

# 5. PHÂN TÁCH SPLIT VÀ CHỈ SỐ DÒNG TỪ ANNOTATION_ID NOTEBOOK 02

# Ví dụ:
# calc_test_0 -> split = calc_test, row_index = 0
# mass_train_24 -> split = mass_train, row_index = 24

roi_annotation_parts = (
    roi_validation_source_df["annotation_id"]
    .astype("string")
    .str.extract(r"^(.*)_(\d+)$")
)

roi_validation_source_df["_parsed_split"] = (
    roi_annotation_parts[0]
)

roi_validation_source_df["_parsed_row_index"] = (
    pd.to_numeric(
        roi_annotation_parts[1],
        errors="coerce",
    )
    .astype("Int64")
)

if (
    roi_validation_source_df[
        [
            "_parsed_split",
            "_parsed_row_index",
        ]
    ]
    .isna()
    .any()
    .any()
):
    raise ValueError(
        "Không thể phân tách annotation_id của Notebook 02."
    )


# Kiểm tra split được phân tách có khớp cột split gốc.
split_mismatch_df = roi_validation_source_df.loc[
    roi_validation_source_df["_parsed_split"]
    != roi_validation_source_df["split"].astype("string")
].copy()

if not split_mismatch_df.empty:

    display(
        split_mismatch_df[
            [
                "annotation_id",
                "split",
                "_parsed_split",
                "_parsed_row_index",
            ]
        ].head(20)
    )

    raise ValueError(
        "Tên split trích xuất từ annotation_id không khớp."
    )

roi_validation_source_df["_roi_link_key"] = (
    roi_validation_source_df["split"]
    .astype("string")
    .str.strip()
    .str.lower()
    + "_"
    + roi_validation_source_df[
        "_parsed_row_index"
    ].astype("string")
)

if roi_validation_source_df["_roi_link_key"].duplicated().any():
    raise ValueError(
        "Khóa liên kết Notebook 02 bị trùng."
    )

print("\n[ĐẠT] Đã tạo được khóa liên kết trung gian.")

# 6. CHUẨN BỊ CÁC TRƯỜNG KIỂM ĐỊNH ROI

roi_columns_to_keep = [
    "_roi_link_key",
    "annotation_id",
    "patient_id",
    "split",
    "abnormality_type",
    "roi_mask_file_path",
    "full_image_file_path",
    "resolution_method",
    "mask_read_status",
    "full_header_status",
    "mask_shape",
    "full_image_shape",
    "shape_match",
    "mask_dtype",
    "unique_value_count",
    "unique_values_sample",
    "is_empty",
    "nonzero_pixel_count",
    "mask_area_ratio",
    "connected_components",
    "bbox_x_min",
    "bbox_y_min",
    "bbox_x_max",
    "bbox_y_max",
    "mask_quality_status",
    "error_message",
    "mask_foreground_fraction",
    "mask_values_binary",
    "bbox_coordinate_space",
    "bbox_full_image_coordinate_eligible",
    "coordinate_note",
]

roi_columns_to_keep = [
    column
    for column in roi_columns_to_keep
    if column in roi_validation_source_df.columns
]

roi_validation_link_df = (
    roi_validation_source_df[
        roi_columns_to_keep
    ]
    .rename(
        columns={
            "annotation_id": "nb02_roi_annotation_id",
            "patient_id": "roi_patient_id",
            "split": "roi_split",
            "abnormality_type": "roi_abnormality_type",
        }
    )
    .copy()
)

# 7. LIÊN KẾT NOTEBOOK 03 VỚI KẾT QUẢ ROI NOTEBOOK 02

lesion_roi_mapping_df = (
    annotation_roi_source_df
    .merge(
        roi_validation_link_df,
        on="_roi_link_key",
        how="left",
        validate="one_to_one",
        indicator="_roi_merge_status",
    )
)
# 8. KIỂM TRA TÍNH TOÀN VẸN PHÉP LIÊN KẾT

unmatched_roi_rows_df = (
    lesion_roi_mapping_df.loc[
        lesion_roi_mapping_df["_roi_merge_status"] != "both"
    ]
    .copy()
)

print("\n" + "=" * 80)
print("KIỂM TRA KẾT QUẢ LIÊN KẾT ROI")
print("=" * 80)

print(
    f"Số annotation Notebook 03: "
    f"{len(annotation_roi_source_df):,}"
)

print(
    f"Số bản ghi ROI Notebook 02: "
    f"{len(roi_validation_source_df):,}"
)

print(
    f"Số annotation ghép được ROI: "
    f"{(lesion_roi_mapping_df['_roi_merge_status'] == 'both').sum():,}"
)

print(
    f"Số annotation chưa ghép được ROI: "
    f"{len(unmatched_roi_rows_df):,}"
)

if not unmatched_roi_rows_df.empty:

    display(
        unmatched_roi_rows_df[
            [
                "annotation_id",
                "source_table",
                "source_row_index",
                "_roi_link_key",
            ]
        ].head(20)
    )

    raise ValueError(
        "Có annotation không tìm thấy kết quả kiểm định ROI."
    )

print("[ĐẠT] Tất cả annotation đã liên kết được với ROI validation.")

# 9. ĐỐI CHIẾU BỆNH NHÂN, SPLIT VÀ LOẠI TỔN THƯƠNG

patient_mismatch_mask = (
    lesion_roi_mapping_df["patient_id"]
    .astype("string")
    .str.strip()
    .str.upper()
    !=
    lesion_roi_mapping_df["roi_patient_id"]
    .astype("string")
    .str.strip()
    .str.upper()
)

split_mismatch_mask = (
    lesion_roi_mapping_df["source_table"]
    .astype("string")
    .str.strip()
    .str.lower()
    !=
    lesion_roi_mapping_df["roi_split"]
    .astype("string")
    .str.strip()
    .str.lower()
)

lesion_type_mismatch_mask = (
    lesion_roi_mapping_df["lesion_type"]
    .astype("string")
    .str.strip()
    .str.lower()
    !=
    lesion_roi_mapping_df["roi_abnormality_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)

roi_identity_mismatch_df = (
    lesion_roi_mapping_df.loc[
        patient_mismatch_mask
        | split_mismatch_mask
        | lesion_type_mismatch_mask
    ]
    .copy()
)

print("\n" + "=" * 80)
print("ĐỐI CHIẾU THÔNG TIN ĐỊNH DANH")
print("=" * 80)

print(
    f"Số bệnh nhân không khớp: "
    f"{int(patient_mismatch_mask.sum()):,}"
)

print(
    f"Số bảng nguồn không khớp: "
    f"{int(split_mismatch_mask.sum()):,}"
)

print(
    f"Số loại tổn thương không khớp: "
    f"{int(lesion_type_mismatch_mask.sum()):,}"
)

if not roi_identity_mismatch_df.empty:

    display(
        roi_identity_mismatch_df[
            [
                "annotation_id",
                "patient_id",
                "roi_patient_id",
                "source_table",
                "roi_split",
                "lesion_type",
                "roi_abnormality_type",
            ]
        ].head(20)
    )

    raise ValueError(
        "Thông tin định danh không nhất quán giữa hai notebook."
    )

print("[ĐẠT] Thông tin bệnh nhân, split và loại tổn thương khớp nhau.")

# 10. KIỂM TRA TÌNH TRẠNG ĐỌC FILE

mask_read_failures_df = lesion_roi_mapping_df.loc[
    lesion_roi_mapping_df["mask_read_status"] != "success"
].copy()

full_header_failures_df = lesion_roi_mapping_df.loc[
    lesion_roi_mapping_df["full_header_status"] != "success"
].copy()

print("\n" + "=" * 80)
print("KIỂM TRA TÌNH TRẠNG ĐỌC FILE")
print("=" * 80)

print(
    f"ROI mask đọc không thành công: "
    f"{len(mask_read_failures_df):,}"
)

print(
    f"Full image header đọc không thành công: "
    f"{len(full_header_failures_df):,}"
)

if (
    not mask_read_failures_df.empty
    or not full_header_failures_df.empty
):
    raise ValueError(
        "Phát hiện file ROI mask hoặc full image chưa đọc thành công."
    )

print("[ĐẠT] Toàn bộ file ROI mask và header ảnh đều đọc thành công.")

# 11. KIỂM TRA CHẤT LƯỢNG ROI MASK

roi_quality_summary_df = (
    lesion_roi_mapping_df["mask_quality_status"]
    .value_counts(dropna=False)
    .rename_axis("mask_quality_status")
    .reset_index(name="so_luong")
)

print("\n" + "=" * 80)
print("PHÂN BỐ TRẠNG THÁI CHẤT LƯỢNG ROI MASK")
print("=" * 80)

display(roi_quality_summary_df)

# Kiểm tra số lượng theo kết quả Notebook 02.
roi_status_counts = (
    lesion_roi_mapping_df["mask_quality_status"]
    .value_counts()
    .to_dict()
)

expected_roi_status_counts = {
    "structurally_valid": 3488,
    "shape_mismatch": 80,
}

if roi_status_counts != expected_roi_status_counts:

    raise ValueError(
        "Phân bố trạng thái ROI không khớp kết quả Notebook 02. "
        f"Thực tế: {roi_status_counts}"
    )

print("[ĐẠT] Phân bố trạng thái ROI khớp Notebook 02.")

# 12. KIỂM TRA MASK RỖNG VÀ GIÁ TRỊ NHỊ PHÂN

empty_mask_count = int(
    lesion_roi_mapping_df["is_empty"].sum()
)

non_binary_mask_count = int(
    (~lesion_roi_mapping_df["mask_values_binary"]).sum()
)

print("\n" + "=" * 80)
print("KIỂM TRA NỘI DUNG MASK")
print("=" * 80)

print(
    f"Số mask rỗng: {empty_mask_count:,}"
)

print(
    f"Số mask không có giá trị nhị phân: "
    f"{non_binary_mask_count:,}"
)

if empty_mask_count != 0:
    raise ValueError(
        "Phát hiện ROI mask rỗng."
    )

if non_binary_mask_count != 0:
    raise ValueError(
        "Phát hiện ROI mask không có giá trị nhị phân."
    )

print("[ĐẠT] Không có mask rỗng hoặc mask không nhị phân.")

# 13. KIỂM TRA ĐIỀU KIỆN KÍCH THƯỚC VÀ TỌA ĐỘ

bbox_eligible_summary_df = (
    lesion_roi_mapping_df[
        "bbox_full_image_coordinate_eligible"
    ]
    .value_counts(dropna=False)
    .rename_axis("bbox_full_image_coordinate_eligible")
    .reset_index(name="so_luong")
)

print("\n" + "=" * 80)
print("PHÂN BỐ ĐIỀU KIỆN SỬ DỤNG BBOX TRÊN FULL IMAGE")
print("=" * 80)

display(bbox_eligible_summary_df)

print(
    "\nLưu ý: Giá trị True chỉ thể hiện điều kiện kích thước "
    "theo quy tắc kiểm định Notebook 02."
)

print(
    "Chưa xác nhận được sự căn chỉnh không gian thực tế "
    "giữa ROI mask và full image."
)

# 14. TẠO TRẠNG THÁI QUẢN LÝ ROI TRONG NOTEBOOK 03

lesion_roi_mapping_df["roi_mapping_status"] = (
    lesion_roi_mapping_df["mask_quality_status"]
    .map({
        "structurally_valid": "ROI_hop_le_ve_cau_truc",
        "shape_mismatch": "ROI_sai_khac_kich_thuoc",
    })
    .fillna("ROI_can_kiem_tra_them")
)

# Không xem bbox là tọa độ đã xác minh.
lesion_roi_mapping_df["spatial_alignment_verified"] = False

# 15. KIỂM TRA BẢO TOÀN SỐ LƯỢNG ANNOTATION

print("\n" + "=" * 80)
print("KIỂM TRA BẢO TOÀN BẢNG ROI MAPPING")
print("=" * 80)

print(
    f"Số annotation nguồn: "
    f"{len(annotation_roi_source_df):,}"
)

print(
    f"Số dòng ROI mapping: "
    f"{len(lesion_roi_mapping_df):,}"
)

print(
    f"Số annotation_id duy nhất: "
    f"{lesion_roi_mapping_df['annotation_id'].nunique():,}"
)

if (
    len(lesion_roi_mapping_df)
    != len(annotation_roi_source_df)
):
    raise ValueError(
        "Số dòng ROI mapping không khớp annotation nguồn."
    )

if (
    lesion_roi_mapping_df["annotation_id"].nunique()
    != len(annotation_roi_source_df)
):
    raise ValueError(
        "annotation_id bị thiếu hoặc trùng trong ROI mapping."
    )

print("[ĐẠT] Bảo toàn toàn bộ 3.568 annotation.")

# 16. XEM TRƯỚC BẢNG ROI MAPPING

roi_mapping_preview_columns = [
    "annotation_id",
    "lesion_candidate_id",
    "review_group_id",
    "patient_id",
    "image_view",
    "abnormality_id",
    "roi_mask_csv_path",
    "roi_mask_file_path",
    "mask_read_status",
    "mask_shape",
    "full_image_shape",
    "shape_match",
    "mask_quality_status",
    "bbox_x_min",
    "bbox_y_min",
    "bbox_x_max",
    "bbox_y_max",
    "bbox_coordinate_space",
    "bbox_full_image_coordinate_eligible",
    "spatial_alignment_verified",
    "roi_mapping_status",
]

roi_mapping_preview_columns = [
    column
    for column in roi_mapping_preview_columns
    if column in lesion_roi_mapping_df.columns
]

print("\n" + "=" * 80)
print("XEM TRƯỚC BẢNG ROI MAPPING")
print("=" * 80)

with pd.option_context("display.max_colwidth", 100):
    display(
        lesion_roi_mapping_df[
            roi_mapping_preview_columns
        ].head(20)
    )

KIỂM TRA DỮ LIỆU ĐẦU VÀO
Annotation Notebook 03: 3,568
ROI validation Notebook 02: 3,568
[ĐẠT] Các cột cần thiết đều tồn tại.

KIỂM TRA CHỈ SỐ DÒNG NGUỒN
Số bản ghi có source_row_number khác source_row_index + 1: 0
[ĐẠT] Chỉ số dòng nguồn nhất quán.

[ĐẠT] Đã tạo được khóa liên kết trung gian.

KIỂM TRA KẾT QUẢ LIÊN KẾT ROI
Số annotation Notebook 03: 3,568
Số bản ghi ROI Notebook 02: 3,568
Số annotation ghép được ROI: 3,568
Số annotation chưa ghép được ROI: 0
[ĐẠT] Tất cả annotation đã liên kết được với ROI validation.

ĐỐI CHIẾU THÔNG TIN ĐỊNH DANH
Số bệnh nhân không khớp: 0
Số bảng nguồn không khớp: 0
Số loại tổn thương không khớp: 0
[ĐẠT] Thông tin bệnh nhân, split và loại tổn thương khớp nhau.

KIỂM TRA TÌNH TRẠNG ĐỌC FILE
ROI mask đọc không thành công: 0
Full image header đọc không thành công: 0
[ĐẠT] Toàn bộ file ROI mask và header ảnh đều đọc thành công.

PHÂN BỐ TRẠNG THÁI CHẤT LƯỢNG ROI MASK


,mask_quality_status,so_luong
0,structurally_valid,3488
1,shape_mismatch,80


[ĐẠT] Phân bố trạng thái ROI khớp Notebook 02.

KIỂM TRA NỘI DUNG MASK
Số mask rỗng: 0
Số mask không có giá trị nhị phân: 0
[ĐẠT] Không có mask rỗng hoặc mask không nhị phân.

PHÂN BỐ ĐIỀU KIỆN SỬ DỤNG BBOX TRÊN FULL IMAGE


,bbox_full_image_coordinate_eligible,so_luong
0,True,3488
1,False,80



Lưu ý: Giá trị True chỉ thể hiện điều kiện kích thước theo quy tắc kiểm định Notebook 02.
Chưa xác nhận được sự căn chỉnh không gian thực tế giữa ROI mask và full image.

KIỂM TRA BẢO TOÀN BẢNG ROI MAPPING
Số annotation nguồn: 3,568
Số dòng ROI mapping: 3,568
Số annotation_id duy nhất: 3,568
[ĐẠT] Bảo toàn toàn bộ 3.568 annotation.

XEM TRƯỚC BẢNG ROI MAPPING


,annotation_id,lesion_candidate_id,review_group_id,patient_id,image_view,abnormality_id,roi_mask_csv_path,roi_mask_file_path,mask_read_status,mask_shape,full_image_shape,shape_match,mask_quality_status,bbox_x_min,bbox_y_min,bbox_x_max,bbox_y_max,bbox_coordinate_space,bbox_full_image_coordinate_eligible,spatial_alignment_verified,roi_mapping_status
0,calc_train__00001,LCAND_000004,<NA>,P_00005,CC,1,Calc-Training_P_00005_RIGHT_CC_1/1.3.6.1.4.1.9590.100.1.2.32877891901241276921808012421408870908...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Training_P_00005_RIGHT_CC_1\2017-09-06-09...,success,"(5056, 2761)","(5056, 2761)",True,structurally_valid,1585,2669,2222,3218,roi_mask_pixel_coordinates,True,False,ROI_hop_le_ve_cau_truc
1,calc_train__00002,LCAND_000004,<NA>,P_00005,MLO,1,Calc-Training_P_00005_RIGHT_MLO_1/1.3.6.1.4.1.9590.100.1.2.6751236221031963610814850438268078193...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Training_P_00005_RIGHT_MLO_1\2017-08-30-8...,success,"(5386, 2836)","(5386, 2836)",True,structurally_valid,1540,3294,2181,3803,roi_mask_pixel_coordinates,True,False,ROI_hop_le_ve_cau_truc
2,calc_train__00003,LCAND_000005,<NA>,P_00007,CC,1,Calc-Training_P_00007_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.241202057913673145232234613012384759880...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Training_P_00007_LEFT_CC_1\2017-08-30-598...,success,"(5356, 2986)","(5356, 2986)",True,structurally_valid,1416,1260,1953,1901,roi_mask_pixel_coordinates,True,False,ROI_hop_le_ve_cau_truc
3,calc_train__00004,LCAND_000005,<NA>,P_00007,MLO,1,Calc-Training_P_00007_LEFT_MLO_1/1.3.6.1.4.1.9590.100.1.2.31425027291117028920388234902422986882...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Training_P_00007_LEFT_MLO_1\2017-09-06-68...,success,"(5491, 3166)","(5491, 3166)",True,structurally_valid,1164,2572,1885,3221,roi_mask_pixel_coordinates,True,False,ROI_hop_le_ve_cau_truc
4,calc_train__00005,<NA>,LREV_000001,P_00008,CC,1,Calc-Training_P_00008_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.336811694512764490002272925921108351157...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Training_P_00008_LEFT_CC_1\2017-09-06-511...,success,"(4568, 3064)","(4568, 3064)",True,structurally_valid,1568,1904,1705,2025,roi_mask_pixel_coordinates,True,False,ROI_hop_le_ve_cau_truc
5,calc_train__00006,<NA>,LREV_000001,P_00008,CC,2,Calc-Training_P_00008_LEFT_CC_2/1.3.6.1.4.1.9590.100.1.2.207558519313356140106496367610041842177...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Training_P_00008_LEFT_CC_2\2017-09-06-421...,success,"(4568, 3064)","(4568, 3064)",True,structurally_valid,1856,2688,1993,2817,roi_mask_pixel_coordinates,True,False,ROI_hop_le_ve_cau_truc
6,calc_train__00007,<NA>,LREV_000001,P_00008,CC,3,Calc-Training_P_00008_LEFT_CC_3/1.3.6.1.4.1.9590.100.1.2.74480335312443720440103727953726257477/...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Training_P_00008_LEFT_CC_3\2017-09-06-574...,success,"(4568, 3064)","(4568, 3064)",True,structurally_valid,1888,2208,1993,2281,roi_mask_pixel_coordinates,True,False,ROI_hop_le_ve_cau_truc
7,calc_train__00008,<NA>,LREV_000001,P_00008,MLO,1,Calc-Training_P_00008_LEFT_MLO_1/1.3.6.1.4.1.9590.100.1.2.19421664301294301022274117045309492678...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Training_P_00008_LEFT_MLO_1\2017-08-30-26...,success,"(4528, 3056)","(4528, 3056)",True,structurally_valid,2160,2176,2257,2265,roi_mask_pixel_coordinates,True,False,ROI_hop_le_ve_cau_truc
8,calc_train__00009,<NA>,LREV_000001,P_00008,MLO,2,Calc-Training_P_00008_LEFT_MLO_2/1.3.6.1.4.1.9590.100.1.2.36373558861103632922611167132350714147...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Training_P_00008_LEFT_MLO_2\2017-09-06-41...,success,"(4528, 3056)","(4528, 3056)",True,structurally_valid,2136,2720,2225,2825,roi_mask_pixel_coordinates,True,False,ROI_hop_le_ve_cau_truc
9,calc_train__00010,<NA>,LREV_000001,P_00008,MLO,3,Calc-Training_P_00008_LEFT_MLO_3/1.3.6.1.4.1.9590.100.

In [95]:
# PHẦN 03.9.3 — KIỂM TRA TÍNH NHẤT QUÁN CUỐI CÙNG CỦA ROI MAPPING

# 1. Kiểm tra dữ liệu đầu vào

if "lesion_roi_mapping_df" not in globals():
    raise NameError(
        "Không tìm thấy lesion_roi_mapping_df. "
        "Hãy chạy hoàn tất Phần 03.9.2 trước."
    )

roi_final_audit_df = lesion_roi_mapping_df.copy()


# 2. Hàm chuẩn hóa đường dẫn để đối chiếu

def normalize_path_for_comparison(value):

    if pd.isna(value):
        return ""

    return (
        str(value)
        .strip()
        .replace("\\", "/")
        .strip("/")
        .casefold()
    )


def check_path_suffix(relative_path, absolute_path):

    relative_value = normalize_path_for_comparison(
        relative_path
    )

    absolute_value = normalize_path_for_comparison(
        absolute_path
    )

    if not relative_value or not absolute_value:
        return False

    return (
        absolute_value == relative_value
        or absolute_value.endswith("/" + relative_value)
    )

# 3. Đối chiếu đường dẫn ROI mask và ảnh đầy đủ

roi_final_audit_df["_roi_path_match"] = (
    roi_final_audit_df.apply(
        lambda row: check_path_suffix(
            row["roi_mask_csv_path"],
            row["roi_mask_file_path"],
        ),
        axis=1,
    )
)

roi_final_audit_df["_full_image_path_match"] = (
    roi_final_audit_df.apply(
        lambda row: check_path_suffix(
            row["full_image_csv_path"],
            row["full_image_file_path"],
        ),
        axis=1,
    )
)

# 4. Đối chiếu thông tin định danh

roi_final_audit_df["_patient_match"] = (
    roi_final_audit_df["patient_id"]
    .astype("string")
    .str.strip()
    .str.upper()
    ==
    roi_final_audit_df["roi_patient_id"]
    .astype("string")
    .str.strip()
    .str.upper()
)

roi_final_audit_df["_split_match"] = (
    roi_final_audit_df["source_table"]
    .astype("string")
    .str.strip()
    .str.lower()
    ==
    roi_final_audit_df["roi_split"]
    .astype("string")
    .str.strip()
    .str.lower()
)

roi_final_audit_df["_lesion_type_match"] = (
    roi_final_audit_df["lesion_type"]
    .astype("string")
    .str.strip()
    .str.lower()
    ==
    roi_final_audit_df["roi_abnormality_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# 5. Tổng hợp các tiêu chí kiểm tra

audit_checks = [
    ("Liên kết annotation - ROI", "_roi_merge_status", "both"),
    ("Mã bệnh nhân khớp", "_patient_match", True),
    ("Bảng nguồn khớp", "_split_match", True),
    ("Loại tổn thương khớp", "_lesion_type_match", True),
    ("Đường dẫn ROI khớp", "_roi_path_match", True),
    ("Đường dẫn ảnh đầy đủ khớp", "_full_image_path_match", True),
    ("ROI mask đọc thành công", "mask_read_status", "success"),
    ("Header ảnh đọc thành công", "full_header_status", "success"),
    ("Mask không rỗng", "is_empty", False),
    ("Mask có giá trị nhị phân", "mask_values_binary", True),
]

final_check_records = []

for check_name, column, expected_value in audit_checks:

    if column == "_roi_merge_status":
        condition = roi_final_audit_df[column].eq(
            expected_value
        )
    else:
        condition = roi_final_audit_df[column].eq(
            expected_value
        )

    final_check_records.append({
        "noi_dung_kiem_tra": check_name,
        "so_luong_dat": int(condition.sum()),
        "so_luong_chua_dat": int((~condition).sum()),
        "trang_thai": (
            "Đạt"
            if condition.all()
            else "Cần kiểm tra"
        ),
    })

roi_final_integrity_summary_df = pd.DataFrame(
    final_check_records
)

print("=" * 80)
print("BÁO CÁO KIỂM TRA TÍNH TOÀN VẸN ROI MAPPING")
print("=" * 80)

display(roi_final_integrity_summary_df)

# 6. Kiểm tra riêng các trường hợp không khớp

check_columns = [
    "_roi_path_match",
    "_full_image_path_match",
    "_patient_match",
    "_split_match",
    "_lesion_type_match",
]

failed_roi_audit_mask = (
    ~roi_final_audit_df[check_columns].all(axis=1)
)

failed_roi_audit_df = (
    roi_final_audit_df.loc[
        failed_roi_audit_mask
    ].copy()
)

print("\n" + "=" * 80)
print("CÁC BẢN GHI CẦN KIỂM TRA THÊM")
print("=" * 80)

print(
    f"Số bản ghi có ít nhất một thông tin không khớp: "
    f"{len(failed_roi_audit_df):,}"
)

if not failed_roi_audit_df.empty:

    display(
        failed_roi_audit_df[
            [
                "annotation_id",
                "patient_id",
                "source_table",
                "roi_split",
                "roi_mask_csv_path",
                "roi_mask_file_path",
                "full_image_csv_path",
                "full_image_file_path",
            ] + check_columns
        ].head(20)
    )


# 7. Đối chiếu trạng thái kích thước và tọa độ

print("\n" + "=" * 80)
print("ĐỐI CHIẾU KẾT QUẢ KÍCH THƯỚC VÀ TỌA ĐỘ")
print("=" * 80)

display(
    pd.crosstab(
        roi_final_audit_df["mask_quality_status"],
        roi_final_audit_df[
            "bbox_full_image_coordinate_eligible"
        ],
        margins=True,
    )
)

# 8. Tổng kết

print("\n" + "=" * 80)
print("TỔNG KẾT KIỂM TRA ROI MAPPING")
print("=" * 80)

print(
    f"Tổng annotation: "
    f"{len(roi_final_audit_df):,}"
)

print(
    f"Tổng ROI mask hợp lệ về cấu trúc: "
    f"{int((roi_final_audit_df['mask_quality_status'] == 'structurally_valid').sum()):,}"
)

print(
    f"Tổng ROI mask sai khác kích thước: "
    f"{int((roi_final_audit_df['mask_quality_status'] == 'shape_mismatch').sum()):,}"
)

print(
    f"Số bản ghi cần kiểm tra thêm: "
    f"{len(failed_roi_audit_df):,}"
)

if len(failed_roi_audit_df) == 0:
    print("[ĐẠT] Tất cả tiêu chí đối chiếu đều nhất quán.")
else:
    print("[CẢNH BÁO] Cần kiểm tra các bản ghi chưa khớp.")

BÁO CÁO KIỂM TRA TÍNH TOÀN VẸN ROI MAPPING


,noi_dung_kiem_tra,so_luong_dat,so_luong_chua_dat,trang_thai
0,Liên kết annotation - ROI,3568,0,Đạt
1,Mã bệnh nhân khớp,3568,0,Đạt
2,Bảng nguồn khớp,3568,0,Đạt
3,Loại tổn thương khớp,3568,0,Đạt
4,Đường dẫn ROI khớp,0,3568,Cần kiểm tra
5,Đường dẫn ảnh đầy đủ khớp,0,3568,Cần kiểm tra
6,ROI mask đọc thành công,3568,0,Đạt
7,Header ảnh đọc thành công,3568,0,Đạt
8,Mask không rỗng,3568,0,Đạt
9,Mask có giá trị nhị phân,3568,0,Đạt



CÁC BẢN GHI CẦN KIỂM TRA THÊM
Số bản ghi có ít nhất một thông tin không khớp: 3,568


,annotation_id,patient_id,source_table,roi_split,roi_mask_csv_path,roi_mask_file_path,full_image_csv_path,full_image_file_path,_roi_path_match,_full_image_path_match,_patient_match,_split_match,_lesion_type_match
0,calc_train__00001,P_00005,calc_train,calc_train,Calc-Training_P_00005_RIGHT_CC_1/1.3.6.1.4.1.9590.100.1.2.32877891901241276921808012421408870908...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Training_P_00005_RIGHT_CC_1\2017-09-06-09...,Calc-Training_P_00005_RIGHT_CC/1.3.6.1.4.1.9590.100.1.2.408909860712120272633130274602115723157/...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Training_P_00005_RIGHT_CC\2016-08-07-2315...,False,False,True,True,True
1,calc_train__00002,P_00005,calc_train,calc_train,Calc-Training_P_00005_RIGHT_MLO_1/1.3.6.1.4.1.9590.100.1.2.6751236221031963610814850438268078193...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Training_P_00005_RIGHT_MLO_1\2017-08-30-8...,Calc-Training_P_00005_RIGHT_MLO/1.3.6.1.4.1.9590.100.1.2.427517897311902339923511678883689433338...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Training_P_00005_RIGHT_MLO\2016-08-07-333...,False,False,True,True,True
2,calc_train__00003,P_00007,calc_train,calc_train,Calc-Training_P_00007_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.241202057913673145232234613012384759880...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Training_P_00007_LEFT_CC_1\2017-08-30-598...,Calc-Training_P_00007_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.201322325113694962619881476352450072222/1...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Training_P_00007_LEFT_CC\2016-08-07-72222...,False,False,True,True,True
3,calc_train__00004,P_00007,calc_train,calc_train,Calc-Training_P_00007_LEFT_MLO_1/1.3.6.1.4.1.9590.100.1.2.31425027291117028920388234902422986882...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Training_P_00007_LEFT_MLO_1\2017-09-06-68...,Calc-Training_P_00007_LEFT_MLO/1.3.6.1.4.1.9590.100.1.2.370479499712916693322010643793108454887/...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Training_P_00007_LEFT_MLO\2016-08-07-5488...,False,False,True,True,True
4,calc_train__00005,P_00008,calc_train,calc_train,Calc-Training_P_00008_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.336811694512764490002272925921108351157...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Training_P_00008_LEFT_CC_1\2017-09-06-511...,Calc-Training_P_00008_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.162256682111885666305889708873412464189/1...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Training_P_00008_LEFT_CC\2016-08-07-64189...,False,False,True,True,True
5,calc_train__00006,P_00008,calc_train,calc_train,Calc-Training_P_00008_LEFT_CC_2/1.3.6.1.4.1.9590.100.1.2.207558519313356140106496367610041842177...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Training_P_00008_LEFT_CC_2\2017-09-06-421...,Calc-Training_P_00008_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.162256682111885666305889708873412464189/1...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Training_P_00008_LEFT_CC\2016-08-07-64189...,False,False,True,True,True
6,calc_train__00007,P_00008,calc_train,calc_train,Calc-Training_P_00008_LEFT_CC_3/1.3.6.1.4.1.9590.100.1.2.74480335312443720440103727953726257477/...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Training_P_00008_LEFT_CC_3\2017-09-06-574...,Calc-Training_P_00008_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.162256682111885666305889708873412464189/1...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Training_P_00008_LEFT_CC\2016-08-07-64189...,False,False,True,True,True
7,calc_train__00008,P_00008,calc_train,calc_train,Calc-Training_P_00008_LEFT_MLO_1/1.3.6.1.4.1.9590.100.1.2.19421664301294301022274117045309492678...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Training_P_00008_LEFT_MLO_1\2017-08-30-26...,Calc-Training_P_00008_LEFT_MLO/1.3.6.1.4.1.9590.100.1.2.84155842712765660429462608981255328663/1...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Training_P_00008_LEFT_MLO\2016-08-07-2866..


ĐỐI CHIẾU KẾT QUẢ KÍCH THƯỚC VÀ TỌA ĐỘ


bbox_full_image_coordinate_eligible,False,True,All
mask_quality_status,,,
shape_mismatch,80,0,80
structurally_valid,0,3488,3488
All,80,3488,3568



TỔNG KẾT KIỂM TRA ROI MAPPING
Tổng annotation: 3,568
Tổng ROI mask hợp lệ về cấu trúc: 3,488
Tổng ROI mask sai khác kích thước: 80
Số bản ghi cần kiểm tra thêm: 3,568
[CẢNH BÁO] Cần kiểm tra các bản ghi chưa khớp.


In [96]:
# KIỂM TRA RIÊNG THÔNG TIN ĐỊNH DANH VÀ TÌNH TRẠNG ĐỌC FILE ROI


df = lesion_roi_mapping_df.copy()

# 1. Đối chiếu thông tin định danh

patient_mismatch = (
    df["patient_id"].astype("string").str.strip().str.upper()
    != df["roi_patient_id"].astype("string").str.strip().str.upper()
)

split_mismatch = (
    df["source_table"].astype("string").str.strip().str.lower()
    != df["roi_split"].astype("string").str.strip().str.lower()
)

type_mismatch = (
    df["lesion_type"].astype("string").str.strip().str.lower()
    != df["roi_abnormality_type"].astype("string").str.strip().str.lower()
)

identity_check_df = pd.DataFrame([
    {
        "noi_dung": "patient_id",
        "so_khong_khop": int(patient_mismatch.sum()),
    },
    {
        "noi_dung": "source_table và split",
        "so_khong_khop": int(split_mismatch.sum()),
    },
    {
        "noi_dung": "lesion_type và abnormality_type",
        "so_khong_khop": int(type_mismatch.sum()),
    },
])

print("=" * 80)
print("1. ĐỐI CHIẾU THÔNG TIN ĐỊNH DANH")
print("=" * 80)

display(identity_check_df)

# 2. Kiểm tra tình trạng đọc ROI mask và header ảnh đầy đủ

print("\n" + "=" * 80)
print("2. TÌNH TRẠNG ĐỌC ROI MASK")
print("=" * 80)

display(
    df["mask_read_status"]
    .value_counts(dropna=False)
    .rename_axis("trang_thai")
    .reset_index(name="so_luong")
)

print("\n" + "=" * 80)
print("3. TÌNH TRẠNG ĐỌC HEADER ẢNH ĐẦY ĐỦ")
print("=" * 80)

display(
    df["full_header_status"]
    .value_counts(dropna=False)
    .rename_axis("trang_thai")
    .reset_index(name="so_luong")
)

# 3. Tổng hợp kết quả

print("\n" + "=" * 80)
print("4. KẾT LUẬN KIỂM TRA")
print("=" * 80)

total_identity_errors = int(
    patient_mismatch.sum()
    + split_mismatch.sum()
    + type_mismatch.sum()
)

mask_read_success = int(
    df["mask_read_status"].eq("success").sum()
)

header_read_success = int(
    df["full_header_status"].eq("success").sum()
)

print(f"Số lỗi thông tin định danh: {total_identity_errors:,}")
print(f"ROI mask đọc thành công: {mask_read_success:,}/{len(df):,}")
print(f"Header ảnh đọc thành công: {header_read_success:,}/{len(df):,}")

if (
    total_identity_errors == 0
    and mask_read_success == len(df)
    and header_read_success == len(df)
):
    print("\n[ĐẠT] Tất cả tiêu chí đều nhất quán.")
else:
    print("\n[CẦN KIỂM TRA] Có tiêu chí chưa đạt.")

1. ĐỐI CHIẾU THÔNG TIN ĐỊNH DANH


,noi_dung,so_khong_khop
0,patient_id,0
1,source_table và split,0
2,lesion_type và abnormality_type,0



2. TÌNH TRẠNG ĐỌC ROI MASK


,trang_thai,so_luong
0,success,3568



3. TÌNH TRẠNG ĐỌC HEADER ẢNH ĐẦY ĐỦ


,trang_thai,so_luong
0,success,3568



4. KẾT LUẬN KIỂM TRA
Số lỗi thông tin định danh: 0
ROI mask đọc thành công: 3,568/3,568
Header ảnh đọc thành công: 3,568/3,568

[ĐẠT] Tất cả tiêu chí đều nhất quán.


### 3.10 — KIỂM TRA TÍNH TOÀN VẸN TỔNG THỂ BỘ DỮ LIỆU

Mục tiêu:
1. Kiểm tra tính toàn vẹn của candidate master và review groups. 
2. Kiểm tra annotation mapping.
3. Kiểm tra image mapping và image catalog.
4. Kiểm tra ROI mapping.
5. Đối chiếu candidate/review ID giữa các bảng.
6. Tổng hợp kết quả trước khi chuyển sang bước lưu dữ liệu.


In [99]:
# 3.10 — KIỂM TRA TÍNH TOÀN VẸN TỔNG THỂ BỘ DỮ LIỆU

# 1. KIỂM TRA CÁC DATAFRAME ĐẦU VÀO

required_objects = [
    "lesion_master_df",
    "lesion_review_groups_df",
    "lesion_annotation_mapping_df",
    "lesion_image_mapping_df",
    "image_path_catalog_df",
    "lesion_roi_mapping_df",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        f"Không tìm thấy các DataFrame: {missing_objects}. "
        "Hãy chạy đầy đủ các phần trước của Notebook 03."
    )


qa_candidate_df = lesion_master_df.copy()
qa_review_df = lesion_review_groups_df.copy()
qa_annotation_df = lesion_annotation_mapping_df.copy()
qa_image_df = lesion_image_mapping_df.copy()
qa_catalog_df = image_path_catalog_df.copy()
qa_roi_df = lesion_roi_mapping_df.copy()


required_columns = {
    "lesion_master_df": [
        "lesion_candidate_id",
        "candidate_status",
    ],
    "lesion_review_groups_df": [
        "review_group_id",
    ],
    "lesion_annotation_mapping_df": [
        "annotation_id",
        "lesion_candidate_id",
        "review_group_id",
    ],
    "lesion_image_mapping_df": [
        "annotation_id",
        "lesion_candidate_id",
        "review_group_id",
        "image_role",
        "image_path_key",
    ],
    "image_path_catalog_df": [
        "image_role",
        "image_path_key",
    ],
    "lesion_roi_mapping_df": [
        "annotation_id",
        "lesion_candidate_id",
        "review_group_id",
        "roi_mask_file_path",
        "mask_read_status",
        "full_header_status",
        "mask_quality_status",
        "is_empty",
        "mask_values_binary",
        "bbox_full_image_coordinate_eligible",
        "spatial_alignment_verified",
    ],
}


for dataframe_name, columns in required_columns.items():

    dataframe = globals()[dataframe_name]

    missing_columns = [
        column for column in columns
        if column not in dataframe.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{dataframe_name} thiếu các cột: {missing_columns}"
        )


input_summary_df = pd.DataFrame(
    [
        {
            "dataframe": "lesion_master_df",
            "so_dong": len(qa_candidate_df),
            "so_cot": len(qa_candidate_df.columns),
        },
        {
            "dataframe": "lesion_review_groups_df",
            "so_dong": len(qa_review_df),
            "so_cot": len(qa_review_df.columns),
        },
        {
            "dataframe": "lesion_annotation_mapping_df",
            "so_dong": len(qa_annotation_df),
            "so_cot": len(qa_annotation_df.columns),
        },
        {
            "dataframe": "lesion_image_mapping_df",
            "so_dong": len(qa_image_df),
            "so_cot": len(qa_image_df.columns),
        },
        {
            "dataframe": "image_path_catalog_df",
            "so_dong": len(qa_catalog_df),
            "so_cot": len(qa_catalog_df.columns),
        },
        {
            "dataframe": "lesion_roi_mapping_df",
            "so_dong": len(qa_roi_df),
            "so_cot": len(qa_roi_df.columns),
        },
    ]
)

print("1. QUY MÔ CÁC DATAFRAME ĐẦU VÀO")
display(input_summary_df)


# 2. KHỞI TẠO CÁC HÀM KIỂM TRA

qa_checks = []


def add_qa_check(
    check_name,
    actual_value,
    expected_value,
    passed,
    note="",
):
    qa_checks.append(
        {
            "kiem_tra": check_name,
            "gia_tri_thuc_te": str(actual_value),
            "gia_tri_ky_vong": str(expected_value),
            "ket_qua": "ĐẠT" if passed else "KHÔNG ĐẠT",
            "ghi_chu": note,
        }
    )


def normalize_id(series):
    normalized = (
        series
        .astype("string")
        .str.strip()
    )

    normalized = normalized.mask(
        normalized.eq(""),
        pd.NA,
    )

    return normalized.fillna("__MISSING_ID__")


def count_nonmissing_id(series):
    normalized = (
        series
        .astype("string")
        .str.strip()
    )

    return int(
        (
            normalized.notna()
            & normalized.ne("")
        ).sum()
    )


def count_missing_or_blank(series):
    normalized = (
        series
        .astype("string")
        .str.strip()
    )

    return int(
        (
            normalized.isna()
            | normalized.eq("")
        ).sum()
    )


def compare_annotation_links(base_df, other_df):

    key_columns = [
        "annotation_id",
        "lesion_candidate_id",
        "review_group_id",
    ]

    base = (
        base_df[key_columns]
        .copy()
        .drop_duplicates(subset=["annotation_id"])
    )

    other = (
        other_df[key_columns]
        .copy()
        .drop_duplicates(subset=["annotation_id"])
    )

    merged_df = base.merge(
        other,
        on="annotation_id",
        how="outer",
        suffixes=("_base", "_other"),
        indicator=True,
    )

    mismatch_mask = merged_df["_merge"].ne("both")

    for column in [
        "lesion_candidate_id",
        "review_group_id",
    ]:
        mismatch_mask |= (
            normalize_id(merged_df[f"{column}_base"])
            != normalize_id(merged_df[f"{column}_other"])
        )

    mismatch_df = merged_df.loc[mismatch_mask].copy()

    return merged_df, mismatch_df


# 3. KIỂM TRA CANDIDATE MASTER

candidate_id_unique = int(
    qa_candidate_df["lesion_candidate_id"].nunique()
)

candidate_id_nonmissing_count = count_nonmissing_id(
    qa_candidate_df["lesion_candidate_id"]
)

candidate_id_missing_count = (
    len(qa_candidate_df) - candidate_id_nonmissing_count
)

candidate_id_duplicate_count = int(
    qa_candidate_df["lesion_candidate_id"]
    .duplicated()
    .sum()
)

candidate_status_counts = (
    qa_candidate_df["candidate_status"]
    .value_counts(dropna=False)
    .to_dict()
)

expected_candidate_status_counts = {
    "Cap_CC_MLO_nhat_quan": 850,
    "Cap_CC_MLO_can_ra_soat": 106,
    "Ung_vien_don_annotation": 1033,
}


add_qa_check(
    "Số Candidate duy nhất",
    candidate_id_unique,
    1989,
    candidate_id_unique == 1989,
)

add_qa_check(
    "Candidate ID bị thiếu",
    candidate_id_missing_count,
    0,
    candidate_id_missing_count == 0,
)

add_qa_check(
    "Candidate ID bị trùng",
    candidate_id_duplicate_count,
    0,
    candidate_id_duplicate_count == 0,
)

add_qa_check(
    "Phân bố candidate_status",
    candidate_status_counts,
    expected_candidate_status_counts,
    candidate_status_counts == expected_candidate_status_counts,
)


candidate_status_summary_df = (
    qa_candidate_df["candidate_status"]
    .value_counts(dropna=False)
    .rename_axis("candidate_status")
    .reset_index(name="so_candidate")
)

print("\n3. PHÂN BỐ CANDIDATE STATUS")
display(candidate_status_summary_df)


# 4. KIỂM TRA REVIEW GROUP

review_id_unique = int(
    qa_review_df["review_group_id"].nunique()
)

review_id_nonmissing_count = count_nonmissing_id(
    qa_review_df["review_group_id"]
)

review_id_missing_count = (
    len(qa_review_df) - review_id_nonmissing_count
)

review_id_duplicate_count = int(
    qa_review_df["review_group_id"]
    .duplicated()
    .sum()
)


add_qa_check(
    "Số Review Group duy nhất",
    review_id_unique,
    140,
    review_id_unique == 140,
)

add_qa_check(
    "Review Group ID bị thiếu",
    review_id_missing_count,
    0,
    review_id_missing_count == 0,
)

add_qa_check(
    "Review Group ID bị trùng",
    review_id_duplicate_count,
    0,
    review_id_duplicate_count == 0,
)


# 5. KIỂM TRA ANNOTATION MAPPING

annotation_count = len(qa_annotation_df)

annotation_unique_count = int(
    qa_annotation_df["annotation_id"].nunique()
)

annotation_duplicate_count = int(
    qa_annotation_df["annotation_id"]
    .duplicated()
    .sum()
)

annotation_missing_id_count = count_missing_or_blank(
    qa_annotation_df["annotation_id"]
)


annotation_candidate_mask = (
    qa_annotation_df["lesion_candidate_id"].notna()
    & normalize_id(
        qa_annotation_df["lesion_candidate_id"]
    ).ne("__MISSING_ID__")
)

annotation_review_mask = (
    qa_annotation_df["review_group_id"].notna()
    & normalize_id(
        qa_annotation_df["review_group_id"]
    ).ne("__MISSING_ID__")
)

annotation_exclusive_mask = (
    annotation_candidate_mask
    ^ annotation_review_mask
)

candidate_annotation_count = int(
    annotation_candidate_mask.sum()
)

review_annotation_count = int(
    annotation_review_mask.sum()
)

invalid_annotation_assignment_count = int(
    (~annotation_exclusive_mask).sum()
)


add_qa_check(
    "Tổng số annotation",
    annotation_count,
    3568,
    annotation_count == 3568,
)

add_qa_check(
    "Annotation ID duy nhất",
    annotation_unique_count,
    3568,
    annotation_unique_count == 3568,
)

add_qa_check(
    "Annotation ID bị thiếu",
    annotation_missing_id_count,
    0,
    annotation_missing_id_count == 0,
)

add_qa_check(
    "Annotation ID bị trùng",
    annotation_duplicate_count,
    0,
    annotation_duplicate_count == 0,
)

add_qa_check(
    "Annotation thuộc Candidate",
    candidate_annotation_count,
    2945,
    candidate_annotation_count == 2945,
)

add_qa_check(
    "Annotation thuộc Review Group",
    review_annotation_count,
    623,
    review_annotation_count == 623,
)

add_qa_check(
    "Annotation được gán đúng một nhóm",
    invalid_annotation_assignment_count,
    0,
    invalid_annotation_assignment_count == 0,
)


master_candidate_ids = set(
    normalize_id(
        qa_candidate_df["lesion_candidate_id"]
    )
)

mapped_candidate_ids = set(
    normalize_id(
        qa_annotation_df.loc[
            annotation_candidate_mask,
            "lesion_candidate_id",
        ]
    )
)

candidate_ids_not_in_master = (
    mapped_candidate_ids - master_candidate_ids
)

candidate_ids_without_annotation = (
    master_candidate_ids - mapped_candidate_ids
)


add_qa_check(
    "Candidate ID tham chiếu không hợp lệ",
    len(candidate_ids_not_in_master),
    0,
    len(candidate_ids_not_in_master) == 0,
)

add_qa_check(
    "Candidate không có annotation",
    len(candidate_ids_without_annotation),
    0,
    len(candidate_ids_without_annotation) == 0,
)


review_group_ids = set(
    normalize_id(
        qa_review_df["review_group_id"]
    )
)

mapped_review_ids = set(
    normalize_id(
        qa_annotation_df.loc[
            annotation_review_mask,
            "review_group_id",
        ]
    )
)

review_ids_not_in_groups = (
    mapped_review_ids - review_group_ids
)

review_ids_without_annotation = (
    review_group_ids - mapped_review_ids
)


add_qa_check(
    "Review Group ID tham chiếu không hợp lệ",
    len(review_ids_not_in_groups),
    0,
    len(review_ids_not_in_groups) == 0,
)

add_qa_check(
    "Review Group không có annotation",
    len(review_ids_without_annotation),
    0,
    len(review_ids_without_annotation) == 0,
)


annotation_assignment_summary_df = pd.DataFrame(
    [
        {
            "loai_lien_ket": "Candidate",
            "so_annotation": candidate_annotation_count,
            "so_ID": len(mapped_candidate_ids),
        },
        {
            "loai_lien_ket": "Review Group",
            "so_annotation": review_annotation_count,
            "so_ID": len(mapped_review_ids),
        },
    ]
)

print("\n5. PHÂN BỐ ANNOTATION THEO NHÓM")
display(annotation_assignment_summary_df)


# 6. KIỂM TRA IMAGE MAPPING

image_mapping_count = len(qa_image_df)

image_annotation_count = int(
    qa_image_df["annotation_id"].nunique()
)

image_missing_annotation_id_count = count_missing_or_blank(
    qa_image_df["annotation_id"]
)

image_duplicate_relationship_count = int(
    qa_image_df.duplicated(
        subset=[
            "annotation_id",
            "image_role",
        ]
    ).sum()
)

expected_image_role_counts = {
    "full_image": 3568,
    "cropped_image": 3568,
}

image_role_counts = (
    qa_image_df["image_role"]
    .value_counts(dropna=False)
    .to_dict()
)


add_qa_check(
    "Tổng số quan hệ annotation-image",
    image_mapping_count,
    7136,
    image_mapping_count == 7136,
)

add_qa_check(
    "Số annotation có liên kết ảnh",
    image_annotation_count,
    3568,
    image_annotation_count == 3568,
)

add_qa_check(
    "Annotation ID trong image mapping bị thiếu",
    image_missing_annotation_id_count,
    0,
    image_missing_annotation_id_count == 0,
)

add_qa_check(
    "Quan hệ annotation-role bị trùng",
    image_duplicate_relationship_count,
    0,
    image_duplicate_relationship_count == 0,
)

add_qa_check(
    "Phân bố vai trò ảnh",
    image_role_counts,
    expected_image_role_counts,
    image_role_counts == expected_image_role_counts,
)


image_role_matrix = (
    qa_image_df
    .groupby(
        ["annotation_id", "image_role"],
        dropna=False,
    )
    .size()
    .unstack(fill_value=0)
)

for expected_role in [
    "full_image",
    "cropped_image",
]:
    if expected_role not in image_role_matrix.columns:
        image_role_matrix[expected_role] = 0


image_role_matrix = image_role_matrix[
    ["full_image", "cropped_image"]
]

image_role_count_mismatch_df = (
    image_role_matrix.loc[
        (image_role_matrix["full_image"] != 1)
        | (image_role_matrix["cropped_image"] != 1)
    ]
    .copy()
)


add_qa_check(
    "Annotation có đúng 1 full và 1 crop",
    len(image_role_count_mismatch_df),
    0,
    image_role_count_mismatch_df.empty,
)


image_link_id_consistency = (
    qa_image_df
    .groupby("annotation_id")
    .agg(
        candidate_id_variants=(
            "lesion_candidate_id",
            lambda x: normalize_id(x).nunique(),
        ),
        review_id_variants=(
            "review_group_id",
            lambda x: normalize_id(x).nunique(),
        ),
    )
)

image_link_id_inconsistency_df = (
    image_link_id_consistency.loc[
        (
            image_link_id_consistency["candidate_id_variants"] > 1
        )
        |
        (
            image_link_id_consistency["review_id_variants"] > 1
        )
    ]
    .copy()
)


add_qa_check(
    "Candidate/Review ID nhất quán trong image mapping",
    len(image_link_id_inconsistency_df),
    0,
    image_link_id_inconsistency_df.empty,
)


image_mapping_missing_path_count = count_missing_or_blank(
    qa_image_df["image_path_key"]
)


add_qa_check(
    "Đường dẫn ảnh trong image mapping bị thiếu",
    image_mapping_missing_path_count,
    0,
    image_mapping_missing_path_count == 0,
)


# 7. KIỂM TRA IMAGE PATH CATALOG

catalog_count = len(qa_catalog_df)

catalog_role_counts = (
    qa_catalog_df["image_role"]
    .value_counts(dropna=False)
    .to_dict()
)

catalog_duplicate_key_count = int(
    qa_catalog_df.duplicated(
        subset=[
            "image_role",
            "image_path_key",
        ]
    ).sum()
)

catalog_missing_path_count = count_missing_or_blank(
    qa_catalog_df["image_path_key"]
)


add_qa_check(
    "Tổng số đường dẫn trong image catalog",
    catalog_count,
    6671,
    catalog_count == 6671,
)

add_qa_check(
    "Số đường dẫn full image duy nhất",
    catalog_role_counts.get("full_image", 0),
    3103,
    catalog_role_counts.get("full_image", 0) == 3103,
)

add_qa_check(
    "Số đường dẫn cropped image duy nhất",
    catalog_role_counts.get("cropped_image", 0),
    3568,
    catalog_role_counts.get("cropped_image", 0) == 3568,
)

add_qa_check(
    "Khóa role + path trong catalog bị trùng",
    catalog_duplicate_key_count,
    0,
    catalog_duplicate_key_count == 0,
)

add_qa_check(
    "Đường dẫn trong catalog bị thiếu",
    catalog_missing_path_count,
    0,
    catalog_missing_path_count == 0,
)


catalog_role_summary_df = (
    qa_catalog_df
    .groupby("image_role")
    .agg(
        so_duong_dan=(
            "image_path_key",
            "nunique",
        )
    )
    .reset_index()
)

print("\n7. PHÂN BỐ IMAGE PATH CATALOG")
display(catalog_role_summary_df)


# 8. ĐỐI CHIẾU IMAGE MAPPING VỚI IMAGE CATALOG

catalog_lookup_df = (
    qa_catalog_df[
        [
            "image_role",
            "image_path_key",
        ]
    ]
    .drop_duplicates()
    .copy()
)

image_mapping_for_catalog_df = (
    qa_image_df[
        [
            "annotation_id",
            "image_role",
            "image_path_key",
        ]
    ]
    .copy()
)

image_catalog_merge_df = (
    image_mapping_for_catalog_df
    .merge(
        catalog_lookup_df,
        on=[
            "image_role",
            "image_path_key",
        ],
        how="left",
        validate="many_to_one",
        indicator=True,
    )
)

image_without_catalog_df = (
    image_catalog_merge_df.loc[
        image_catalog_merge_df["_merge"] != "both"
    ]
    .copy()
)

image_catalog_matched_count = int(
    image_catalog_merge_df["_merge"]
    .eq("both")
    .sum()
)


add_qa_check(
    "Image mapping không tìm thấy catalog",
    len(image_without_catalog_df),
    0,
    image_without_catalog_df.empty,
)

add_qa_check(
    "Số quan hệ ảnh tìm thấy trong catalog",
    image_catalog_matched_count,
    len(image_mapping_for_catalog_df),
    image_catalog_matched_count == len(image_mapping_for_catalog_df),
)


print("\n8. KẾT QUẢ ĐỐI CHIẾU IMAGE CATALOG")

print(
    "Số quan hệ ảnh được đối chiếu:",
    f"{image_catalog_matched_count:,}/"
    f"{len(image_mapping_for_catalog_df):,}",
)

print(
    "Số quan hệ không tìm thấy trong catalog:",
    f"{len(image_without_catalog_df):,}",
)


# 9. ĐỐI CHIẾU CANDIDATE/REVIEW ID GIỮA CÁC BẢNG

qa_image_annotation_links_df = (
    qa_image_df[
        [
            "annotation_id",
            "lesion_candidate_id",
            "review_group_id",
        ]
    ]
    .drop_duplicates(
        subset=["annotation_id"],
        keep="first",
    )
    .copy()
)


(
    annotation_image_comparison_df,
    annotation_image_mismatch_df,
) = compare_annotation_links(
    qa_annotation_df,
    qa_image_annotation_links_df,
)


add_qa_check(
    "Candidate/Review ID giữa annotation và image mapping",
    len(annotation_image_mismatch_df),
    0,
    annotation_image_mismatch_df.empty,
)


(
    annotation_roi_comparison_df,
    annotation_roi_mismatch_df,
) = compare_annotation_links(
    qa_annotation_df,
    qa_roi_df,
)


add_qa_check(
    "Candidate/Review ID giữa annotation và ROI mapping",
    len(annotation_roi_mismatch_df),
    0,
    annotation_roi_mismatch_df.empty,
)


# 10. KIỂM TRA ROI MAPPING

roi_count = len(qa_roi_df)

roi_annotation_unique_count = int(
    qa_roi_df["annotation_id"].nunique()
)

roi_duplicate_annotation_count = int(
    qa_roi_df["annotation_id"]
    .duplicated()
    .sum()
)

roi_missing_annotation_count = count_missing_or_blank(
    qa_roi_df["annotation_id"]
)

roi_missing_mask_path_count = count_missing_or_blank(
    qa_roi_df["roi_mask_file_path"]
)


add_qa_check(
    "Tổng số dòng ROI mapping",
    roi_count,
    3568,
    roi_count == 3568,
)

add_qa_check(
    "Annotation ID duy nhất trong ROI mapping",
    roi_annotation_unique_count,
    3568,
    roi_annotation_unique_count == 3568,
)

add_qa_check(
    "Annotation ID trong ROI mapping bị thiếu",
    roi_missing_annotation_count,
    0,
    roi_missing_annotation_count == 0,
)

add_qa_check(
    "Annotation ID trùng trong ROI mapping",
    roi_duplicate_annotation_count,
    0,
    roi_duplicate_annotation_count == 0,
)

add_qa_check(
    "Đường dẫn ROI mask bị thiếu",
    roi_missing_mask_path_count,
    0,
    roi_missing_mask_path_count == 0,
)


roi_quality_counts = (
    qa_roi_df["mask_quality_status"]
    .value_counts(dropna=False)
    .to_dict()
)

expected_roi_quality_counts = {
    "structurally_valid": 3488,
    "shape_mismatch": 80,
}


add_qa_check(
    "Phân bố chất lượng ROI mask",
    roi_quality_counts,
    expected_roi_quality_counts,
    roi_quality_counts == expected_roi_quality_counts,
)


roi_mask_status_counts = (
    qa_roi_df["mask_read_status"]
    .value_counts(dropna=False)
    .to_dict()
)

full_header_status_counts = (
    qa_roi_df["full_header_status"]
    .value_counts(dropna=False)
    .to_dict()
)


add_qa_check(
    "ROI mask đọc thành công",
    roi_mask_status_counts,
    {"success": 3568},
    roi_mask_status_counts == {"success": 3568},
)

add_qa_check(
    "Full image header đọc thành công",
    full_header_status_counts,
    {"success": 3568},
    full_header_status_counts == {"success": 3568},
)


roi_is_empty_text = (
    qa_roi_df["is_empty"]
    .astype("string")
    .str.strip()
    .str.lower()
)

roi_empty_count = int(
    roi_is_empty_text.eq("true").sum()
)

roi_empty_invalid_count = int(
    (~roi_is_empty_text.isin(["true", "false"])).sum()
)


add_qa_check(
    "Số ROI mask rỗng",
    roi_empty_count,
    0,
    roi_empty_count == 0,
)

add_qa_check(
    "Giá trị is_empty không hợp lệ hoặc bị thiếu",
    roi_empty_invalid_count,
    0,
    roi_empty_invalid_count == 0,
)


roi_binary_text = (
    qa_roi_df["mask_values_binary"]
    .astype("string")
    .str.strip()
    .str.lower()
)

roi_non_binary_count = int(
    roi_binary_text.eq("false").sum()
)

roi_binary_invalid_count = int(
    (~roi_binary_text.isin(["true", "false"])).sum()
)


add_qa_check(
    "Số ROI mask không có giá trị nhị phân",
    roi_non_binary_count,
    0,
    roi_non_binary_count == 0,
)

add_qa_check(
    "Giá trị mask_values_binary không hợp lệ hoặc bị thiếu",
    roi_binary_invalid_count,
    0,
    roi_binary_invalid_count == 0,
)


bbox_eligible_text = (
    qa_roi_df["bbox_full_image_coordinate_eligible"]
    .astype("string")
    .str.strip()
    .str.lower()
)

bbox_eligible_counts = (
    bbox_eligible_text
    .value_counts(dropna=False)
    .to_dict()
)

expected_bbox_eligible_counts = {
    "true": 3488,
    "false": 80,
}


add_qa_check(
    "Phân bố điều kiện kích thước bbox",
    bbox_eligible_counts,
    expected_bbox_eligible_counts,
    bbox_eligible_counts == expected_bbox_eligible_counts,
)


spatial_alignment_text = (
    qa_roi_df["spatial_alignment_verified"]
    .astype("string")
    .str.strip()
    .str.lower()
)

spatial_alignment_true_count = int(
    spatial_alignment_text.eq("true").sum()
)

spatial_alignment_invalid_count = int(
    (~spatial_alignment_text.isin(["true", "false"])).sum()
)


add_qa_check(
    "Số ROI được đánh dấu đã xác minh căn chỉnh không gian",
    spatial_alignment_true_count,
    0,
    spatial_alignment_true_count == 0,
)

add_qa_check(
    "Giá trị spatial_alignment_verified không hợp lệ hoặc bị thiếu",
    spatial_alignment_invalid_count,
    0,
    spatial_alignment_invalid_count == 0,
)


roi_quality_summary_df = (
    qa_roi_df
    .groupby(
        "mask_quality_status",
        dropna=False,
    )
    .agg(
        so_annotation=(
            "annotation_id",
            "nunique",
        ),
        so_mask_rong=(
            "is_empty",
            lambda x: (
                x.astype("string")
                .str.strip()
                .str.lower()
                .eq("true")
                .sum()
            ),
        ),
        so_bbox_du_dieu_kien=(
            "bbox_full_image_coordinate_eligible",
            lambda x: (
                x.astype("string")
                .str.strip()
                .str.lower()
                .eq("true")
                .sum()
            ),
        ),
    )
    .reset_index()
)

print("\n10. TỔNG HỢP CHẤT LƯỢNG ROI MASK")
display(roi_quality_summary_df)


# 11. ĐỐI CHIẾU TẬP ANNOTATION GIỮA CÁC BẢNG MAPPING

annotation_ids = set(
    qa_annotation_df["annotation_id"].dropna()
)

image_annotation_ids = set(
    qa_image_df["annotation_id"].dropna()
)

roi_annotation_ids = set(
    qa_roi_df["annotation_id"].dropna()
)


annotation_missing_from_image = (
    annotation_ids - image_annotation_ids
)

image_annotation_not_in_source = (
    image_annotation_ids - annotation_ids
)

annotation_missing_from_roi = (
    annotation_ids - roi_annotation_ids
)

roi_annotation_not_in_source = (
    roi_annotation_ids - annotation_ids
)


add_qa_check(
    "Annotation thiếu trong image mapping",
    len(annotation_missing_from_image),
    0,
    len(annotation_missing_from_image) == 0,
)

add_qa_check(
    "Annotation lạ trong image mapping",
    len(image_annotation_not_in_source),
    0,
    len(image_annotation_not_in_source) == 0,
)

add_qa_check(
    "Annotation thiếu trong ROI mapping",
    len(annotation_missing_from_roi),
    0,
    len(annotation_missing_from_roi) == 0,
)

add_qa_check(
    "Annotation lạ trong ROI mapping",
    len(roi_annotation_not_in_source),
    0,
    len(roi_annotation_not_in_source) == 0,
)


# 12. TỔNG HỢP KẾT QUẢ KIỂM TRA

nb03_integrity_check_df = pd.DataFrame(
    qa_checks
)

passed_check_count = int(
    nb03_integrity_check_df["ket_qua"]
    .eq("ĐẠT")
    .sum()
)

failed_check_count = int(
    nb03_integrity_check_df["ket_qua"]
    .eq("KHÔNG ĐẠT")
    .sum()
)


print("\n12. TỔNG HỢP KẾT QUẢ KIỂM TRA")

print(
    f"Tổng số phép kiểm tra: "
    f"{len(nb03_integrity_check_df):,}"
)

print(
    f"Số phép kiểm tra đạt: "
    f"{passed_check_count:,}"
)

print(
    f"Số phép kiểm tra không đạt: "
    f"{failed_check_count:,}"
)

display(nb03_integrity_check_df)


# 13. HIỂN THỊ CÁC TRƯỜNG HỢP CẦN XEM XÉT

if failed_check_count > 0:

    print("\n13. CÁC PHÉP KIỂM TRA CHƯA ĐẠT")

    display(
        nb03_integrity_check_df.loc[
            nb03_integrity_check_df["ket_qua"]
            == "KHÔNG ĐẠT"
        ]
    )


    diagnostic_sets = {
        "Candidate ID không có annotation":
            candidate_ids_without_annotation,

        "Review Group không có annotation":
            review_ids_without_annotation,

        "Candidate ID không tồn tại trong master":
            candidate_ids_not_in_master,

        "Review Group ID không tồn tại":
            review_ids_not_in_groups,
    }

    for diagnostic_name, diagnostic_data in diagnostic_sets.items():

        if len(diagnostic_data) > 0:

            print(f"\n{diagnostic_name}")

            display(
                pd.DataFrame(
                    {"id": list(diagnostic_data)}
                ).head(20)
            )


    diagnostic_tables = {
        "Annotation-image ID mismatch":
            annotation_image_mismatch_df,

        "Annotation-ROI ID mismatch":
            annotation_roi_mismatch_df,

        "Image không có trong catalog":
            image_without_catalog_df,

        "Annotation có số lượng vai trò ảnh bất thường":
            image_role_count_mismatch_df,

        "Image mapping có Candidate/Review ID không nhất quán":
            image_link_id_inconsistency_df,
    }

    for diagnostic_name, diagnostic_df in diagnostic_tables.items():

        if not diagnostic_df.empty:

            print(f"\n{diagnostic_name}")

            display(
                diagnostic_df.head(20)
            )


# 14. KẾT LUẬN KIỂM TRA

if failed_check_count == 0:

    print("\n14. KẾT LUẬN KIỂM TRA")
    print("[ĐẠT] Tất cả phép kiểm tra tính toàn vẹn đều thành công.")

    print("\nQUY MÔ CÁC BẢNG:")

    print(f"- Candidate: {candidate_id_unique:,}")
    print(f"- Review Group: {review_id_unique:,}")
    print(f"- Annotation: {annotation_unique_count:,}")

    print(
        f"- Annotation thuộc Candidate: "
        f"{candidate_annotation_count:,}"
    )

    print(
        f"- Annotation thuộc Review Group: "
        f"{review_annotation_count:,}"
    )

    print(
        f"- Quan hệ annotation-image: "
        f"{image_mapping_count:,}"
    )

    print(
        f"- Đường dẫn full image trong catalog: "
        f"{catalog_role_counts.get('full_image', 0):,}"
    )

    print(
        f"- Đường dẫn cropped image trong catalog: "
        f"{catalog_role_counts.get('cropped_image', 0):,}"
    )

    print(f"- ROI mapping: {roi_count:,}")

    print(
        f"- ROI hợp lệ về cấu trúc: "
        f"{roi_quality_counts.get('structurally_valid', 0):,}"
    )

    print(
        f"- ROI sai khác kích thước: "
        f"{roi_quality_counts.get('shape_mismatch', 0):,}"
    )

    print("\nTRẠNG THÁI KIỂM SOÁT:")

    print("- Các bảng mapping đã được đối chiếu theo annotation_id.")

    print(
        "- Candidate ID và Review Group ID đã được đối chiếu "
        "giữa các bảng."
    )

    print(
        "- Candidate chưa được xác nhận là tổn thương "
        "đã kiểm chứng lâm sàng."
    )

    print(
        "- Chưa xác nhận căn chỉnh không gian giữa ROI mask "
        "và full image."
    )

    print(
        "\nChưa lưu file. Các bảng vẫn được giữ trong bộ nhớ Notebook."
    )

else:

    print("\n14. KẾT LUẬN KIỂM TRA")
    print(
        f"[CHƯA ĐẠT] Có {failed_check_count:,} phép kiểm tra "
        "cần xem xét."
    )

    raise ValueError(
        "Kiểm tra tính toàn vẹn tổng thể chưa đạt. "
        "Hãy xem bảng kiểm tra và các bảng chẩn đoán ở mục 13 "
        "trước khi chuyển sang bước lưu dữ liệu."
    )

1. QUY MÔ CÁC DATAFRAME ĐẦU VÀO


,dataframe,so_dong,so_cot
0,lesion_master_df,1989,42
1,lesion_review_groups_df,140,36
2,lesion_annotation_mapping_df,3568,51
3,lesion_image_mapping_df,7136,17
4,image_path_catalog_df,6671,4
5,lesion_roi_mapping_df,3568,87



3. PHÂN BỐ CANDIDATE STATUS


,candidate_status,so_candidate
0,Ung_vien_don_annotation,1033
1,Cap_CC_MLO_nhat_quan,850
2,Cap_CC_MLO_can_ra_soat,106



5. PHÂN BỐ ANNOTATION THEO NHÓM


,loai_lien_ket,so_annotation,so_ID
0,Candidate,2945,1989
1,Review Group,623,140


... (3 outputs left out)

,mask_quality_status,so_annotation,so_mask_rong,so_bbox_du_dieu_kien
0,shape_mismatch,80,0,0
1,structurally_valid,3488,0,3488



12. TỔNG HỢP KẾT QUẢ KIỂM TRA
Tổng số phép kiểm tra: 54
Số phép kiểm tra đạt: 54
Số phép kiểm tra không đạt: 0


,kiem_tra,gia_tri_thuc_te,gia_tri_ky_vong,ket_qua,ghi_chu
0,Số Candidate duy nhất,1989,1989,ĐẠT,
1,Candidate ID bị thiếu,0,0,ĐẠT,
2,Candidate ID bị trùng,0,0,ĐẠT,
3,Phân bố candidate_status,"{'Ung_vien_don_annotation': 1033, 'Cap_CC_MLO_nhat_quan': 850, 'Cap_CC_MLO_can_ra_soat': 106}","{'Cap_CC_MLO_nhat_quan': 850, 'Cap_CC_MLO_can_ra_soat': 106, 'Ung_vien_don_annotation': 1033}",ĐẠT,
4,Số Review Group duy nhất,140,140,ĐẠT,
5,Review Group ID bị thiếu,0,0,ĐẠT,
6,Review Group ID bị trùng,0,0,ĐẠT,
7,Tổng số annotation,3568,3568,ĐẠT,
8,Annotation ID duy nhất,3568,3568,ĐẠT,
9,Annotation ID bị thiếu,0,0,ĐẠT,



14. KẾT LUẬN KIỂM TRA
[ĐẠT] Tất cả phép kiểm tra tính toàn vẹn đều thành công.

QUY MÔ CÁC BẢNG:
- Candidate: 1,989
- Review Group: 140
- Annotation: 3,568
- Annotation thuộc Candidate: 2,945
- Annotation thuộc Review Group: 623
- Quan hệ annotation-image: 7,136
- Đường dẫn full image trong catalog: 3,103
- Đường dẫn cropped image trong catalog: 3,568
- ROI mapping: 3,568
- ROI hợp lệ về cấu trúc: 3,488
- ROI sai khác kích thước: 80

TRẠNG THÁI KIỂM SOÁT:
- Các bảng mapping đã được đối chiếu theo annotation_id.
- Candidate ID và Review Group ID đã được đối chiếu giữa các bảng.
- Candidate chưa được xác nhận là tổn thương đã kiểm chứng lâm sàng.
- Chưa xác nhận căn chỉnh không gian giữa ROI mask và full image.

Chưa lưu file. Các bảng vẫn được giữ trong bộ nhớ Notebook.


In [100]:
from datetime import datetime
import tempfile
import shutil

# 1. KIỂM TRA ĐIỀU KIỆN TRƯỚC KHI LƯU

PROJECT_ROOT = Path(r"C:\Breast-Cancer-CBIS-DDSM")
PROCESSED_DIR = PROJECT_ROOT / "Data" / "processed"

OUTPUT_DIR = PROCESSED_DIR / "nb03_lesion_dataset"

if not PROCESSED_DIR.exists():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục processed: {PROCESSED_DIR}"
    )

if not PROCESSED_DIR.is_dir():
    raise NotADirectoryError(
        f"Đường dẫn processed không phải thư mục: {PROCESSED_DIR}"
    )


# Kiểm tra kết quả kiểm định 03.10.

if "nb03_integrity_check_df" not in globals():
    raise NameError(
        "Không tìm thấy nb03_integrity_check_df. "
        "Hãy chạy thành công phần 03.10 trước."
    )

if nb03_integrity_check_df.empty:
    raise ValueError(
        "Bảng kiểm định đang rỗng. Không thể xác nhận điều kiện lưu."
    )

failed_checks = nb03_integrity_check_df.loc[
    nb03_integrity_check_df["ket_qua"].ne("ĐẠT")
].copy()

if not failed_checks.empty:
    display(failed_checks)
    raise ValueError(
        "Vẫn còn phép kiểm tra chưa đạt. Không tiến hành lưu dữ liệu."
    )


# Kiểm tra các DataFrame cần lưu.

tables_to_save = {
    "lesion_master.csv": lesion_master_df,
    "lesion_review_groups.csv": lesion_review_groups_df,
    "lesion_annotation_mapping.csv": lesion_annotation_mapping_df,
    "lesion_image_mapping.csv": lesion_image_mapping_df,
    "image_path_catalog.csv": image_path_catalog_df,
    "lesion_roi_mapping.csv": lesion_roi_mapping_df,
    "nb03_integrity_check.csv": nb03_integrity_check_df,
}

for filename, dataframe in tables_to_save.items():

    if not isinstance(dataframe, pd.DataFrame):
        raise TypeError(
            f"{filename} không phải pandas DataFrame."
        )

    if dataframe.empty:
        raise ValueError(
            f"{filename} đang rỗng. Không tiến hành lưu."
        )


# Không ghi đè thư mục kết quả đã tồn tại.

if OUTPUT_DIR.exists():
    raise FileExistsError(
        f"Thư mục kết quả đã tồn tại:\n{OUTPUT_DIR}\n\n"
        "Cell đã dừng để bảo vệ dữ liệu cũ. "
        "Không có file nào bị ghi đè."
    )


print("1. ĐIỀU KIỆN LƯU DỮ LIỆU")
print(f"- Thư mục dự án: {PROJECT_ROOT}")
print(f"- Thư mục kết quả: {OUTPUT_DIR}")
print(f"- Số phép kiểm tra: {len(nb03_integrity_check_df):,}")
print(f"- Số phép kiểm tra đạt: {len(nb03_integrity_check_df):,}")
print(f"- Số phép kiểm tra không đạt: {len(failed_checks):,}")
print(f"- Số bảng dự kiến lưu: {len(tables_to_save):,}")
print("[ĐẠT] Đủ điều kiện tiến hành lưu.")


# 2. TỔNG HỢP THÔNG TIN CHO BÁO CÁO

candidate_status_summary = (
    lesion_master_df["candidate_status"]
    .value_counts(dropna=False)
    .to_dict()
)

annotation_candidate_count = int(
    lesion_annotation_mapping_df[
        "lesion_candidate_id"
    ].notna().sum()
)

annotation_review_count = int(
    lesion_annotation_mapping_df[
        "review_group_id"
    ].notna().sum()
)

image_role_summary = (
    lesion_image_mapping_df["image_role"]
    .value_counts(dropna=False)
    .to_dict()
)

catalog_role_summary = (
    image_path_catalog_df["image_role"]
    .value_counts(dropna=False)
    .to_dict()
)

roi_quality_summary = (
    lesion_roi_mapping_df["mask_quality_status"]
    .value_counts(dropna=False)
    .to_dict()
)

roi_mask_status_summary = (
    lesion_roi_mapping_df["mask_read_status"]
    .value_counts(dropna=False)
    .to_dict()
)

full_header_status_summary = (
    lesion_roi_mapping_df["full_header_status"]
    .value_counts(dropna=False)
    .to_dict()
)

roi_empty_count = int(
    lesion_roi_mapping_df["is_empty"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("true")
    .sum()
)

roi_non_binary_count = int(
    lesion_roi_mapping_df["mask_values_binary"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("false")
    .sum()
)

spatial_alignment_verified_count = int(
    lesion_roi_mapping_df["spatial_alignment_verified"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("true")
    .sum()
)


# 3. TẠO BÁO CÁO TỔNG KẾT NOTEBOOK 03

report_lines = [
    "NOTEBOOK 03 — LESION DATASET CONSTRUCTION",
    "=" * 72,
    "",
    f"Thời điểm tạo báo cáo: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    f"Thư mục dự án: {PROJECT_ROOT}",
    "",
    "I. TỔNG QUAN CÁC BẢNG DỮ LIỆU",
    "-" * 72,
    f"lesion_master_df: {len(lesion_master_df):,} dòng, "
    f"{len(lesion_master_df.columns):,} cột",
    f"lesion_review_groups_df: {len(lesion_review_groups_df):,} dòng, "
    f"{len(lesion_review_groups_df.columns):,} cột",
    f"lesion_annotation_mapping_df: {len(lesion_annotation_mapping_df):,} dòng, "
    f"{len(lesion_annotation_mapping_df.columns):,} cột",
    f"lesion_image_mapping_df: {len(lesion_image_mapping_df):,} dòng, "
    f"{len(lesion_image_mapping_df.columns):,} cột",
    f"image_path_catalog_df: {len(image_path_catalog_df):,} dòng, "
    f"{len(image_path_catalog_df.columns):,} cột",
    f"lesion_roi_mapping_df: {len(lesion_roi_mapping_df):,} dòng, "
    f"{len(lesion_roi_mapping_df.columns):,} cột",
    "",
    "II. PHÂN BỐ CANDIDATE",
    "-" * 72,
]

for status, count in candidate_status_summary.items():
    report_lines.append(f"{status}: {count:,}")

report_lines.extend([
    "",
    f"Tổng số Candidate: {len(lesion_master_df):,}",
    f"Tổng số Review Group: {len(lesion_review_groups_df):,}",
    "",
    "III. ANNOTATION MAPPING",
    "-" * 72,
    f"Tổng số annotation: {len(lesion_annotation_mapping_df):,}",
    f"Annotation thuộc Candidate: {annotation_candidate_count:,}",
    f"Annotation thuộc Review Group: {annotation_review_count:,}",
    "",
    "IV. IMAGE MAPPING VÀ IMAGE CATALOG",
    "-" * 72,
    f"Tổng số quan hệ annotation-image: {len(lesion_image_mapping_df):,}",
])

for role, count in image_role_summary.items():
    report_lines.append(
        f"Số quan hệ theo vai trò {role}: {count:,}"
    )

report_lines.append("")

for role, count in catalog_role_summary.items():
    report_lines.append(
        f"Số đường dẫn duy nhất theo vai trò {role}: {count:,}"
    )

report_lines.extend([
    "",
    "V. ROI MAPPING VÀ CHẤT LƯỢNG MASK",
    "-" * 72,
    f"Tổng số ROI mapping: {len(lesion_roi_mapping_df):,}",
])

for status, count in roi_quality_summary.items():
    report_lines.append(
        f"{status}: {count:,}"
    )

report_lines.append("")

for status, count in roi_mask_status_summary.items():
    report_lines.append(
        f"Trạng thái đọc ROI mask - {status}: {count:,}"
    )

for status, count in full_header_status_summary.items():
    report_lines.append(
        f"Trạng thái đọc full image header - {status}: {count:,}"
    )

report_lines.extend([
    f"Số ROI mask rỗng: {roi_empty_count:,}",
    f"Số ROI mask không nhị phân: {roi_non_binary_count:,}",
    f"Số ROI được đánh dấu đã xác minh căn chỉnh không gian: "
    f"{spatial_alignment_verified_count:,}",
    "",
    "VI. KẾT QUẢ KIỂM ĐỊNH TỔNG THỂ",
    "-" * 72,
    f"Tổng số phép kiểm tra: {len(nb03_integrity_check_df):,}",
    f"Số phép kiểm tra đạt: "
    f"{nb03_integrity_check_df['ket_qua'].eq('ĐẠT').sum():,}",
    f"Số phép kiểm tra không đạt: "
    f"{nb03_integrity_check_df['ket_qua'].ne('ĐẠT').sum():,}",
    "",
    "VII. CÁC LƯU Ý VỀ DỮ LIỆU",
    "-" * 72,
    "- Candidate là các bản ghi ứng viên được xây dựng từ annotation "
    "và quy tắc ghép descriptor.",
    "- Candidate chưa được xác minh độc lập là tổn thương lâm sàng.",
    "- Các trường hợp review group được giữ riêng để rà soát.",
    "- 80 ROI mask sai khác kích thước được giữ nguyên, không tự ý resize.",
    "- Chưa xác nhận căn chỉnh không gian giữa ROI mask và full image.",
    "- Không sử dụng bbox như tọa độ đã xác minh trên full image.",
    "- Các split gốc được bảo toàn; cần xem xét chia tập theo patient "
    "nếu kết hợp các nguồn Mass và Calc cho thực nghiệm sau này.",
    "",
    "VIII. DANH SÁCH FILE ĐƯỢC LƯU",
    "-" * 72,
])

for filename, dataframe in tables_to_save.items():
    report_lines.append(
        f"{filename}: {len(dataframe):,} dòng, "
        f"{len(dataframe.columns):,} cột"
    )

report_lines.extend([
    "",
    "Báo cáo này tổng kết quá trình xây dựng và kiểm định dữ liệu "
    "trong Notebook 03.",
])

report_content = "\n".join(report_lines)


# 4. GHI DỮ LIỆU VÀO THƯ MỤC TẠM

staging_dir = None

try:

    staging_dir = Path(
        tempfile.mkdtemp(
            prefix=".nb03_lesion_dataset_staging_",
            dir=str(PROCESSED_DIR),
        )
    )

    print("\n4. ĐANG GHI DỮ LIỆU VÀO THƯ MỤC TẠM")

    saved_file_records = []

    for filename, dataframe in tables_to_save.items():

        output_path = staging_dir / filename

        dataframe.to_csv(
            output_path,
            index=False,
            encoding="utf-8-sig",
        )

        if not output_path.exists() or output_path.stat().st_size == 0:
            raise IOError(
                f"File không được ghi thành công: {filename}"
            )

        saved_file_records.append(
            {
                "ten_file": filename,
                "so_dong": len(dataframe),
                "so_cot": len(dataframe.columns),
                "dung_luong_bytes": output_path.stat().st_size,
            }
        )

        print(f"[ĐÃ GHI] {filename}")


    # Ghi báo cáo tổng kết.

    report_path = staging_dir / "nb03_summary_report.txt"

    report_path.write_text(
        report_content,
        encoding="utf-8-sig",
    )

    if not report_path.exists() or report_path.stat().st_size == 0:
        raise IOError(
            "Không ghi được nb03_summary_report.txt"
        )

    saved_file_records.append(
        {
            "ten_file": "nb03_summary_report.txt",
            "so_dong": len(report_lines),
            "so_cot": 1,
            "dung_luong_bytes": report_path.stat().st_size,
        }
    )

    print("[ĐÃ GHI] nb03_summary_report.txt")


    # 5. KIỂM TRA FILE TRƯỚC KHI CHUYỂN SANG THƯ MỤC CHÍNH THỨC

    print("\n5. KIỂM TRA FILE ĐÃ GHI")

    saved_file_summary_df = pd.DataFrame(
        saved_file_records
    )

    expected_file_count = len(tables_to_save) + 1

    actual_file_count = len(
        list(staging_dir.iterdir())
    )

    if actual_file_count != expected_file_count:
        raise RuntimeError(
            f"Số file trong thư mục tạm không đúng. "
            f"Kỳ vọng {expected_file_count}, thực tế {actual_file_count}."
        )

    if saved_file_summary_df["dung_luong_bytes"].le(0).any():
        raise IOError(
            "Phát hiện file có dung lượng bằng 0."
        )

    display(saved_file_summary_df)


    # 6. CHUYỂN THƯ MỤC TẠM THÀNH THƯ MỤC KẾT QUẢ CHÍNH THỨC

    if OUTPUT_DIR.exists():
        raise FileExistsError(
            f"Thư mục đích vừa xuất hiện: {OUTPUT_DIR}. "
            "Không tiến hành ghi đè."
        )

    staging_dir.rename(OUTPUT_DIR)

    staging_dir = None

    print("\n6. KẾT QUẢ LƯU DỮ LIỆU")
    print("[THÀNH CÔNG] Đã lưu bộ dữ liệu Notebook 03.")

    print(f"\nThư mục kết quả:\n{OUTPUT_DIR}")

    print("\nDanh sách file:")

    for filename in [
        *tables_to_save.keys(),
        "nb03_summary_report.txt",
    ]:
        print(f"- {OUTPUT_DIR / filename}")

    print("\nTổng số file đã lưu:", expected_file_count)


except Exception:

    # Chỉ xóa thư mục tạm do chính cell này tạo ra.
    # Không tác động đến dữ liệu hoặc thư mục có sẵn của người dùng.

    if staging_dir is not None and staging_dir.exists():
        shutil.rmtree(staging_dir)

    raise

1. ĐIỀU KIỆN LƯU DỮ LIỆU
- Thư mục dự án: C:\Breast-Cancer-CBIS-DDSM
- Thư mục kết quả: C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb03_lesion_dataset
- Số phép kiểm tra: 54
- Số phép kiểm tra đạt: 54
- Số phép kiểm tra không đạt: 0
- Số bảng dự kiến lưu: 7
[ĐẠT] Đủ điều kiện tiến hành lưu.

4. ĐANG GHI DỮ LIỆU VÀO THƯ MỤC TẠM
[ĐÃ GHI] lesion_master.csv
[ĐÃ GHI] lesion_review_groups.csv
[ĐÃ GHI] lesion_annotation_mapping.csv
[ĐÃ GHI] lesion_image_mapping.csv
[ĐÃ GHI] image_path_catalog.csv
[ĐÃ GHI] lesion_roi_mapping.csv
[ĐÃ GHI] nb03_integrity_check.csv
[ĐÃ GHI] nb03_summary_report.txt

5. KIỂM TRA FILE ĐÃ GHI


,ten_file,so_dong,so_cot,dung_luong_bytes
0,lesion_master.csv,1989,42,1272666
1,lesion_review_groups.csv,140,36,89723
2,lesion_annotation_mapping.csv,3568,51,5190900
3,lesion_image_mapping.csv,7136,17,3598548
4,image_path_catalog.csv,6671,4,2495013
5,lesion_roi_mapping.csv,3568,87,7751141
6,nb03_integrity_check.csv,54,5,3486
7,nb03_summary_report.txt,78,1,3471



6. KẾT QUẢ LƯU DỮ LIỆU
[THÀNH CÔNG] Đã lưu bộ dữ liệu Notebook 03.

Thư mục kết quả:
C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb03_lesion_dataset

Danh sách file:
- C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb03_lesion_dataset\lesion_master.csv
- C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb03_lesion_dataset\lesion_review_groups.csv
- C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb03_lesion_dataset\lesion_annotation_mapping.csv
- C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb03_lesion_dataset\lesion_image_mapping.csv
- C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb03_lesion_dataset\image_path_catalog.csv
- C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb03_lesion_dataset\lesion_roi_mapping.csv
- C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb03_lesion_dataset\nb03_integrity_check.csv
- C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb03_lesion_dataset\nb03_summary_report.txt

Tổng số file đã lưu: 8


### Kết quả đạt được

Nguồn annotation:
- Tổng cộng 3.568 annotation.
- Mass: 1.696 annotation.
- Calcification: 1.872 annotation.

Candidate và Review Group:
- 1.989 candidate records:
- 850 cặp CC–MLO nhất quán theo descriptor.
- 106 cặp CC–MLO cần rà soát.
- 1.033 ứng viên có một annotation trong quy tắc ghép hiện tại.
- 140 review groups, gồm 623 annotation.

Image Mapping:
- 7.136 quan hệ annotation–image.
- 3.103 đường dẫn Full Image duy nhất.
- 3.568 đường dẫn Cropped Image duy nhất.
- 100% quan hệ ảnh tìm thấy trong Image Catalog.

ROI Mapping:
- 3.568 ROI được liên kết.
- 3.488 mask hợp lệ về mặt cấu trúc.
- 80 mask sai khác kích thước.
- Không có mask rỗng hoặc mask không nhị phân.
- Chưa xác minh căn chỉnh không gian với Full Image.

Kiểm định tổng thể:
- 54/54 phép kiểm tra đạt.
- Không phát hiện sai lệch khóa định danh hoặc liên kết giữa các bảng.

Các file đầu ra lưu tại:

*C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb03_lesion_dataset*